In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Data Cleaning

In [ ]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')


# ============================================================
# Step 1: Load Cleaned Dataset
# ============================================================
print("=" * 80)
print("LOAD CLEANED DATASET")
print("=" * 80)

#'/kaggle/working/LendingClub_Cleaned_2016_2020.csv'
df = pd.read_csv('/kaggle/input/datasets/thandarphyo/d-dataset/LendingClub_Cleaned_2016_2020.csv')
print(f"✓ Loaded: {df.shape[0]:,} records, {df.shape[1]} columns\n")


print("\n" + "=" * 80)
print("3 rows record (Original Raw data)")
print("=" * 80)
print(df.head(3))

# ============================================================
# Step 1a: Convert Data Types
# ============================================================
print("=" * 80)
print("DATA TYPE CONVERSION")
print("=" * 80)

# Convert issue_d to datetime
if 'issue_d' in df.columns:
    df['issue_d'] = pd.to_datetime(df['issue_d'], errors='coerce')
    print(f"✓ issue_d converted to datetime")

# Convert int_rate: remove '%' and convert to float
if 'int_rate' in df.columns:
    if df['int_rate'].dtype == 'object':
        df['int_rate'] = df['int_rate'].str.replace('%', '').astype(float)
        print(f"✓ int_rate converted to float (removed '%')")

print()


# ============================================================
# Step 2: Handle Duplicate IDs
# ============================================================
print("=" * 80)
print("DUPLICATE ID HANDLING")
print("=" * 80)

initial_count = len(df)
duplicates = df['id'].duplicated().sum()
print(f"Duplicate IDs found: {duplicates:,}")

if duplicates > 0:
    dup_ids = df[df['id'].duplicated(keep=False)]['id'].unique()
    print(f"Unique duplicate IDs: {len(dup_ids):,}\nExamples:")
    for dup_id in dup_ids[:3]:
        print(f"  ID {dup_id}: appears {(df['id'] == dup_id).sum()} times")
    
    df = df.drop_duplicates(subset=['id'], keep='first').reset_index(drop=True)
    print(f"\n✓ Records: {initial_count:,} → {len(df):,} (removed {initial_count - len(df):,})\n")
else:
    print("✓ No duplicates found\n")


# ============================================================
# Step 3: Handle Missing Values
# ============================================================
print("=" * 80)
print("MISSING VALUES HANDLING")
print("=" * 80)

missing_before = df.isnull().sum().sum()
print(f"Missing values BEFORE: {missing_before:,}")

if missing_before > 0:
    print("\nColumns with missing:")
    for col in df.columns[df.isnull().any()]:
        print(f"  {col}: {df[col].isnull().sum():,}")

# Fill missing values
numeric_cols = df.select_dtypes(include=[np.number]).columns
categorical_cols = df.select_dtypes(include=['object']).columns
datetime_cols = df.select_dtypes(include=['datetime64']).columns

for col in numeric_cols:
    if df[col].isnull().sum() > 0:
        df[col].fillna(df[col].mean(), inplace=True)

for col in categorical_cols:
    if df[col].isnull().sum() > 0:
        mode_val = df[col].mode()[0] if len(df[col].mode()) > 0 else 'Unknown'
        df[col].fillna(mode_val, inplace=True)

for col in datetime_cols:
    if df[col].isnull().sum() > 0:
        df[col].fillna(method='ffill', inplace=True)
        df[col].fillna(method='bfill', inplace=True)

missing_after = df.isnull().sum().sum()
print(f"Missing values AFTER: {missing_after:,}")
print(f"✓ All handled!\n")


# ============================================================
# Step 4: Outlier Detection & Winsorization
# ============================================================
print("=" * 80)
print("OUTLIER DETECTION & WINSORIZATION")
print("=" * 80)

numeric_cols_outlier = df.select_dtypes(include=[np.number]).columns.tolist()
outlier_summary = {}

# Store original data for before/after comparison
df_original = df.copy()

print(f"Analyzing {len(numeric_cols_outlier)} numeric columns:\n")

for col in numeric_cols_outlier:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    
    outliers = df[(df[col] < lower_bound) | (df[col] > upper_bound)]
    count = len(outliers)
    pct = (count / len(df)) * 100
    
    outlier_summary[col] = {'count': count, 'percentage': pct, 'lower': lower_bound, 'upper': upper_bound}
    
    if count > 0:
        print(f"{col}: {count:,} outliers ({pct:.2f}%) | Range: [{lower_bound:.2f}, {upper_bound:.2f}]")
    else:
        print(f"{col}: ✓ No outliers")

print("\n" + "=" * 80)
print("APPLYING WINSORIZATION (CAPPING OUTLIERS)")
print("=" * 80)

print("\nCapping outliers (skip Default & zero-inflated columns):\n")

# Columns to skip Winsorization (binary or zero-inflated)
skip_columns = ['Default', 'out_prncp']

for col in numeric_cols_outlier:
    if col in skip_columns:
        reason = "binary target variable" if col == 'Default' else "zero-inflated (mostly zeros)"
        print(f"{col}: ✓ Skipped ({reason})")
        continue
    
    if outlier_summary[col]['count'] > 0:
        lower_bound = outlier_summary[col]['lower']
        upper_bound = outlier_summary[col]['upper']
        
        # Store original values for comparison
        before_min = df[col].min()
        before_max = df[col].max()
        
        # Clip values to bounds
        df[col] = df[col].clip(lower=lower_bound, upper=upper_bound)
        
        after_min = df[col].min()
        after_max = df[col].max()
        
        print(f"{col}:")
        print(f"  Capped {outlier_summary[col]['count']:,} outliers")
        print(f"  Range: [{before_min:.2f}, {before_max:.2f}] → [{after_min:.2f}, {after_max:.2f}]")

print("\n✓ Winsorization complete!\n")

# ============================================================
# Step 4a: Visualize Before & After Winsorization
# ============================================================
print("=" * 80)
print("GENERATING BEFORE & AFTER VISUALIZATIONS")
print("=" * 80)

# Get columns with outliers (excluding Default and zero-inflated columns)
skip_columns = ['Default', 'out_prncp']
outlier_cols = [col for col in outlier_summary 
                if outlier_summary[col]['count'] > 0 and col not in skip_columns]

if outlier_cols:
    # Box plots comparison
    n_cols = min(3, len(outlier_cols))
    n_rows = (len(outlier_cols) + n_cols - 1) // n_cols
    
    fig, axes = plt.subplots(n_rows, n_cols * 2, figsize=(16, 4 * n_rows))
    axes = axes.flatten()
    
    plot_idx = 0
    for col in outlier_cols:
        # BEFORE
        ax_before = axes[plot_idx]
        ax_before.boxplot(df_original[col].dropna(), vert=True)
        ax_before.set_ylabel(col, fontsize=10)
        ax_before.set_title(f"BEFORE: {col}\n({outlier_summary[col]['count']:,} outliers)", 
                           fontsize=9, fontweight='bold', color='red')
        ax_before.grid(True, alpha=0.3)
        
        # AFTER
        ax_after = axes[plot_idx + 1]
        ax_after.boxplot(df[col].dropna(), vert=True)
        ax_after.set_ylabel(col, fontsize=10)
        ax_after.set_title(f"AFTER: {col}\n(capped)", 
                          fontsize=9, fontweight='bold', color='green')
        ax_after.grid(True, alpha=0.3)
        
        plot_idx += 2
    
    # Hide unused subplots
    for idx in range(plot_idx, len(axes)):
        axes[idx].axis('off')
    
    plt.tight_layout()
    boxplot_path = '/kaggle/working/outliers_before_after_boxplot.png'
    plt.savefig(boxplot_path, dpi=100, bbox_inches='tight')
    print(f"✓ Saved: outliers_before_after_boxplot.png")
    plt.show()

# Histograms comparison for columns with outliers
if outlier_cols:
    n_cols = min(3, len(outlier_cols))
    n_rows = (len(outlier_cols) + n_cols - 1) // n_cols
    
    fig, axes = plt.subplots(n_rows, n_cols * 2, figsize=(16, 4 * n_rows))
    axes = axes.flatten()
    
    plot_idx = 0
    for col in outlier_cols:
        # BEFORE histogram
        ax_before = axes[plot_idx]
        ax_before.hist(df_original[col].dropna(), bins=50, color='lightcoral', edgecolor='black', alpha=0.7)
        ax_before.set_xlabel(col, fontsize=9)
        ax_before.set_ylabel('Frequency', fontsize=9)
        ax_before.set_title(f"BEFORE: {col}", fontsize=9, fontweight='bold', color='red')
        ax_before.grid(True, alpha=0.3, axis='y')
        
        # AFTER histogram
        ax_after = axes[plot_idx + 1]
        ax_after.hist(df[col].dropna(), bins=50, color='lightgreen', edgecolor='black', alpha=0.7)
        ax_after.set_xlabel(col, fontsize=9)
        ax_after.set_ylabel('Frequency', fontsize=9)
        ax_after.set_title(f"AFTER: {col}", fontsize=9, fontweight='bold', color='green')
        ax_after.grid(True, alpha=0.3, axis='y')
        
        plot_idx += 2
    
    # Hide unused subplots
    for idx in range(plot_idx, len(axes)):
        axes[idx].axis('off')
    
    plt.tight_layout()
    histogram_path = '/kaggle/working/outliers_before_after_histogram.png'
    plt.savefig(histogram_path, dpi=100, bbox_inches='tight')
    print(f"✓ Saved: outliers_before_after_histogram.png\n")
    plt.show()
else:
    print("✓ No outliers to visualize\n")


# ============================================================
# Step 5: One-Hot Encoding for Categorical Variables
# ============================================================
print("=" * 80)
print("ONE-HOT ENCODING CATEGORICAL VARIABLES")
print("=" * 80)

# Columns to one-hot encode
encode_columns = ['state_region', 'loan_status']

print(f"Encoding {len(encode_columns)} categorical columns:\n")

# Show unique values before encoding
for col in encode_columns:
    unique_count = df[col].nunique()
    print(f"{col}: {unique_count} unique values")
    print(f"  Values: {df[col].unique()}\n")

# One-hot encode the selected columns
df_encoded = pd.get_dummies(df, columns=encode_columns, drop_first=False)

print(f"✓ One-hot encoding complete!")
print(f"  Original shape: {df.shape}")
print(f"  Encoded shape: {df_encoded.shape}")
print(f"  New columns added: {df_encoded.shape[1] - df.shape[1]}\n")

# Show new column names
print("New encoded columns:")
new_cols = [col for col in df_encoded.columns if col not in df.columns]
for col in new_cols:
    print(f"  - {col}")

print("\n" + "=" * 80)
print("UPDATED COLUMN LIST")
print("=" * 80)
print(f"Total columns after encoding: {len(df_encoded.columns)}\n")
print(f"All columns:\n{df_encoded.columns.tolist()}\n")

# Use encoded dataframe for remaining steps
df = df_encoded


# ============================================================
# Step 6: Final Data Quality Check
# ============================================================
print("=" * 80)
print("FINAL DATA QUALITY CHECK")
print("=" * 80)

print(f"Total records: {len(df):,}")
print(f"Total columns: {len(df.columns)}")
print(f"Shape: {df.shape}")
print(f"Memory: {df.memory_usage(deep=True).sum() / (1024**2):.2f} MB")
print(f"Duplicate IDs: {df['id'].duplicated().sum():,}")
print(f"Missing values: {df.isnull().sum().sum():,}")
print(f"✓ Data is clean!\n")


# ============================================================
# Step 7: Save Final Dataset
# ============================================================
print("=" * 80)
print("SAVE FINAL DATASET")
print("=" * 80)

output_path = '/kaggle/working/LendingClub_Cleaned_Final.csv'
df.to_csv(output_path, index=False)
print(f"✓ Saved: {output_path}")
print(f"✓ Records: {len(df):,} | Columns: {len(df.columns)}\n")


# ============================================================
# Step 8: Data Summary
# ============================================================
print("=" * 80)
print("DATA SUMMARY")
print("=" * 80)

print("\nFirst 5 rows:")
print(df.head())

print("\n" + "=" * 80)
print("DATA TYPES")
print("=" * 80)
print(df.dtypes)

print("\n" + "=" * 80)
print("DESCRIPTIVE STATISTICS")
print("=" * 80)
print(df.describe())


print("\n" + "=" * 80)
print("3 rows record (After cleaning Raw data)")
print("=" * 80)
print(df.head(3))

# ============================================================
# Step 8: Data plot
# ============================================================

import matplotlib.pyplot as plt

# Count and convert to percentage
counts = df['Default'].value_counts(normalize=True).sort_index() * 100

# Labels
labels = ['Non-Default (0)', 'Default (1)']
values = [counts.get(0, 0), counts.get(1, 0)]

# Plot
fig, ax = plt.subplots(figsize=(6, 5))

bars = ax.bar(labels, values)

ax.set_xlabel('Loan Status', fontsize=12)
ax.set_ylabel('Percentage (%)', fontsize=12)
ax.set_title('Distribution of Default vs Non-Default Loans (%)', fontsize=14, fontweight='bold')

ax.grid(axis='y', alpha=0.3)

# Add percentage labels on top of bars
for bar in bars:
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2, height, f'{height:.2f}%', 
            ha='center', va='bottom')

plt.show()



# train model before choosing top 3

In [ ]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")

from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import (r2_score, mean_squared_error,
                             accuracy_score, classification_report,
                             roc_auc_score, confusion_matrix)
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.gaussian_process import GaussianProcessClassifier, GaussianProcessRegressor
from sklearn.gaussian_process.kernels import ConstantKernel, Matern
from statsmodels.tsa.statespace.sarimax import SARIMAX

# ─────────────────────────────────────────────
# 1. Load & normalize columns
# ─────────────────────────────────────────────
FILE_PATH = "/kaggle/working/LendingClub_Cleaned_Final.csv"
df = pd.read_csv(FILE_PATH)
df.columns = df.columns.str.strip().str.lower().str.replace(" ", "_").str.replace(r"[()]", "", regex=True)

print("Shape:", df.shape)
print("Columns:", df.columns.tolist())

# ─────────────────────────────────────────────
# 2. Prepare target
# ─────────────────────────────────────────────
TARGET = "default"
df = df.dropna(subset=[TARGET])
print(f"\nTarget value counts:\n{df[TARGET].value_counts()}")

LEAKAGE_COLS = [
    "loan_status_charged_off", "loan_status_default",
    "loan_status_fully_paid",  "loan_status_in_grace_period",
    "loan_status_late_16-30_days", "loan_status_late_31-120_days",
    "total_pymnt", "total_rec_prncp", "total_rec_int", "id"
]
drop_cols = [c for c in LEAKAGE_COLS if c in df.columns]
X = df.drop(columns=[TARGET] + drop_cols)
y = df[TARGET].astype(int).reset_index(drop=True)

# ─────────────────────────────────────────────
# 3. Encode categoricals
# ─────────────────────────────────────────────
le = LabelEncoder()
for col in X.select_dtypes(include=["object", "category"]).columns:
    X[col] = le.fit_transform(X[col].astype(str))

X = X.fillna(X.median(numeric_only=True)).reset_index(drop=True)
print(f"\nFeatures used ({len(X.columns)}): {X.columns.tolist()}")

# ─────────────────────────────────────────────
# 4. 80/20 split (no shuffle — preserves order)
# ─────────────────────────────────────────────
split_idx = int(len(y) * 0.80)
X_train, X_test = X.iloc[:split_idx], X.iloc[split_idx:]
y_train, y_test = y.iloc[:split_idx], y.iloc[split_idx:]

print(f"\nTrain: {len(y_train):,} rows | Test: {len(y_test):,} rows")

# ─────────────────────────────────────────────
# 5. Helper — unified evaluation (classifier)
# ─────────────────────────────────────────────
def evaluate(name, y_tr, yhat_tr, y_te, yhat_te, prob_te=None):
    r2_tr   = r2_score(y_tr, yhat_tr)
    rmse_tr = np.sqrt(mean_squared_error(y_tr, yhat_tr))
    r2_te   = r2_score(y_te, yhat_te)
    rmse_te = np.sqrt(mean_squared_error(y_te, yhat_te))
    acc_te  = accuracy_score(y_te, yhat_te)
    auc_te  = roc_auc_score(y_te, prob_te) if prob_te is not None else float("nan")

    print(f"\n{'═'*50}")
    print(f"  {name}")
    print(f"{'═'*50}")
    print(f"  Train  R²      : {r2_tr:.4f}")
    print(f"  Train  RMSE    : {rmse_tr:.4f}")
    print(f"  Test   R²      : {r2_te:.4f}")
    print(f"  Test   RMSE    : {rmse_te:.4f}")
    print(f"  Test   Accuracy: {acc_te:.4f}")
    print(f"  Test   ROC-AUC : {auc_te:.4f}")
    print(f"\n{classification_report(y_te, yhat_te, target_names=['No Default','Default'])}")
    return {"name": name, "r2_train": r2_tr, "rmse_train": rmse_tr,
            "r2_test": r2_te, "rmse_test": rmse_te,
            "accuracy": acc_te, "roc_auc": auc_te}

# ─────────────────────────────────────────────
# 5b. Helper — GPR-specific evaluation
#     (float output → clip/round → binary metrics)
# ─────────────────────────────────────────────
def evaluate_gpr(name, y_tr, pred_tr_float, y_te, pred_te_float, std_te=None):
    # Raw float metrics (regression view)
    r2_tr_raw   = r2_score(y_tr, pred_tr_float)
    rmse_tr_raw = np.sqrt(mean_squared_error(y_tr, pred_tr_float))
    r2_te_raw   = r2_score(y_te, pred_te_float)
    rmse_te_raw = np.sqrt(mean_squared_error(y_te, pred_te_float))

    # Binary metrics (clip & round floats to 0/1)
    pred_tr_bin = np.clip(np.round(pred_tr_float), 0, 1).astype(int)
    pred_te_bin = np.clip(np.round(pred_te_float), 0, 1).astype(int)
    acc_te      = accuracy_score(y_te, pred_te_bin)

    # Use raw float as proxy probability for AUC (clipped to [0,1])
    prob_te     = np.clip(pred_te_float, 0, 1)
    auc_te      = roc_auc_score(y_te, prob_te)

    print(f"\n{'═'*50}")
    print(f"  {name}")
    print(f"{'═'*50}")
    print(f"  ── Regression view (raw float output) ──")
    print(f"  Train  R²      : {r2_tr_raw:.4f}")
    print(f"  Train  RMSE    : {rmse_tr_raw:.4f}")
    print(f"  Test   R²      : {r2_te_raw:.4f}")
    print(f"  Test   RMSE    : {rmse_te_raw:.4f}")
    print(f"  ── Classification view (rounded to 0/1) ──")
    print(f"  Test   Accuracy: {acc_te:.4f}")
    print(f"  Test   ROC-AUC : {auc_te:.4f}")
    if std_te is not None:
        print(f"  Test   Avg Uncertainty (std): {std_te.mean():.4f}")
    print(f"\n{classification_report(y_te, pred_te_bin, target_names=['No Default','Default'])}")

    return {"name": name,
            "r2_train": r2_tr_raw, "rmse_train": rmse_tr_raw,
            "r2_test":  r2_te_raw, "rmse_test":  rmse_te_raw,
            "accuracy": acc_te,    "roc_auc":    auc_te}

results = []

# ─────────────────────────────────────────────
# 6. SARIMA (time-series baseline)
# ─────────────────────────────────────────────
SARIMA_N  = 50_000
y_s_train = y_train.iloc[:SARIMA_N].reset_index(drop=True)
y_s_test  = y_test.iloc[:SARIMA_N].reset_index(drop=True)

print(f"\nFitting SARIMA on {SARIMA_N:,} rows …")
sarima_fit = SARIMAX(
    y_s_train,
    order=(1, 1, 1),
    seasonal_order=(0, 0, 0, 0),
    enforce_stationarity=False,
    enforce_invertibility=False,
).fit(disp=False)

train_pred = np.clip(np.round(sarima_fit.fittedvalues.dropna()), 0, 1)
aligned    = sarima_fit.fittedvalues.dropna().index
forecast   = np.clip(np.round(sarima_fit.forecast(steps=len(y_s_test))), 0, 1)

results.append(evaluate(
    "SARIMA (50k sample)",
    y_s_train.loc[aligned], train_pred,
    y_s_test, forecast
))

# ─────────────────────────────────────────────
# 7. Logistic Regression
# ─────────────────────────────────────────────
print("\nFitting Logistic Regression …")
lr = LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42)
lr.fit(X_train, y_train)

results.append(evaluate(
    "Logistic Regression",
    y_train, lr.predict(X_train),
    y_test,  lr.predict(X_test),
    prob_te=lr.predict_proba(X_test)[:, 1]
))

# ─────────────────────────────────────────────
# 8. Random Forest
# ─────────────────────────────────────────────
print("\nFitting Random Forest …")
rf = RandomForestClassifier(
    n_estimators=200, max_depth=12,
    class_weight="balanced", random_state=42, n_jobs=-1
)
rf.fit(X_train, y_train)

results.append(evaluate(
    "Random Forest",
    y_train, rf.predict(X_train),
    y_test,  rf.predict(X_test),
    prob_te=rf.predict_proba(X_test)[:, 1]
))

# ─────────────────────────────────────────────
# 9. Gradient Boosting
# ─────────────────────────────────────────────
print("\nFitting Gradient Boosting …")
gb = GradientBoostingClassifier(
    n_estimators=200, max_depth=5,
    learning_rate=0.05, subsample=0.8, random_state=42
)
gb.fit(X_train, y_train)

results.append(evaluate(
    "Gradient Boosting",
    y_train, gb.predict(X_train),
    y_test,  gb.predict(X_test),
    prob_te=gb.predict_proba(X_test)[:, 1]
))

# ─────────────────────────────────────────────
# 10. Shared GP subsample setup
#     (both GPC & GPR use same stratified sample)
# ─────────────────────────────────────────────
GP_TRAIN_N = 3_000
GP_TEST_N  = 1_000

rng = np.random.default_rng(42)

# Train stratified subsample
idx_0_tr  = np.where(y_train.values == 0)[0]
idx_1_tr  = np.where(y_train.values == 1)[0]
n1_tr     = int(GP_TRAIN_N * y_train.mean())
n0_tr     = GP_TRAIN_N - n1_tr
gp_tr_idx = np.concatenate([
    rng.choice(idx_0_tr, n0_tr, replace=False),
    rng.choice(idx_1_tr, n1_tr, replace=False)
])
rng.shuffle(gp_tr_idx)

# Test stratified subsample
idx_0_te  = np.where(y_test.values == 0)[0]
idx_1_te  = np.where(y_test.values == 1)[0]
n1_te     = int(GP_TEST_N * y_test.mean())
n0_te     = GP_TEST_N - n1_te
gp_te_idx = np.concatenate([
    rng.choice(idx_0_te, n0_te, replace=False),
    rng.choice(idx_1_te, n1_te, replace=False)
])
rng.shuffle(gp_te_idx)

# Scale — mandatory for both GPC and GPR
scaler      = StandardScaler()
X_gp_train  = scaler.fit_transform(X_train.iloc[gp_tr_idx])
y_gp_train  = y_train.iloc[gp_tr_idx].reset_index(drop=True)
X_gp_test   = scaler.transform(X_test.iloc[gp_te_idx])
y_gp_test   = y_test.iloc[gp_te_idx].reset_index(drop=True)

print(f"\nGP subsample — Train: {GP_TRAIN_N:,} | Test: {GP_TEST_N:,}")
print(f"  Train class balance: {pd.Series(y_gp_train).value_counts().to_dict()}")
print(f"  Test  class balance: {pd.Series(y_gp_test).value_counts().to_dict()}")

# Shared kernel
kernel = ConstantKernel(1.0, (1e-3, 1e3)) * Matern(
    length_scale=1.0,
    length_scale_bounds=(1e-2, 1e2),
    nu=1.5
)

# ─────────────────────────────────────────────
# 11. GPC — Gaussian Process Classifier
# ─────────────────────────────────────────────
print(f"\nFitting GPC …")
gpc = GaussianProcessClassifier(
    kernel=kernel,
    max_iter_predict=100,
    random_state=42,
    n_jobs=-1
)
gpc.fit(X_gp_train, y_gp_train)
print(f"  Optimised kernel: {gpc.kernel_}")

results.append(evaluate(
    f"GPC [{GP_TRAIN_N:,} train / {GP_TEST_N:,} test]",
    y_gp_train, gpc.predict(X_gp_train),
    y_gp_test,  gpc.predict(X_gp_test),
    prob_te=gpc.predict_proba(X_gp_test)[:, 1]
))

# ─────────────────────────────────────────────
# 12. GPR — Gaussian Process Regressor
#     Predicts floats → clipped/rounded for binary metrics
#     Also returns per-prediction uncertainty (std)
# ─────────────────────────────────────────────
print(f"\nFitting GPR …")
gpr = GaussianProcessRegressor(
    kernel=kernel,
    n_restarts_optimizer=5,    # refits kernel hyperparams 5 times → better optimum
    normalize_y=True,          # centers y → improves numerical stability on 0/1 target
    random_state=42
)
gpr.fit(X_gp_train, y_gp_train)
print(f"  Optimised kernel: {gpr.kernel_}")

# Predict with uncertainty
pred_tr_float               = gpr.predict(X_gp_train)
pred_te_float, pred_te_std  = gpr.predict(X_gp_test, return_std=True)

print(f"\n  GPR float output sample (first 10 test predictions):")
print(f"  Predicted : {np.round(pred_te_float[:10], 4)}")
print(f"  Std (unc) : {np.round(pred_te_std[:10],  4)}")
print(f"  Actual    : {y_gp_test.values[:10]}")

results.append(evaluate_gpr(
    f"GPR [{GP_TRAIN_N:,} train / {GP_TEST_N:,} test]",
    y_gp_train, pred_tr_float,
    y_gp_test,  pred_te_float,
    std_te=pred_te_std
))


# Train selected 3 models

In [ ]:
"""
LendingClub Default Forecasting Pipeline
─────────────────────────────────────────
Sections:
  1.  Data Loading & Merging        (2007–2020)
  2.  Log Transform + Cubic Detrend
  3.  Train / Test Split
  4.  Auto ARIMA                    (log-detrended)
  5.  ARIMA Train Fitted Values
  6.  ARIMA Rolling Walk-Forward    (test)
  7.  ARIMA Evaluation Metrics
  8.  ARIMA 12-Month Forecast
  9.  ARIMA Visualization
  10. ARIMA Save Outputs
  P1. Prophet Input Prep
  P2. Prophet Changepoints
  P3. Prophet Model Builders
  P4. Train Model A                 (with regressors)
  P5. Prophet Train Fitted Values
  P6. Prophet Rolling Walk-Forward  (test)
  P7. Prophet Evaluation Metrics
  P8. Prophet 12-Month Forecast     (Model B, no regressors)
  P9. Prophet Visualization
  P10.Prophet Components Plot
  P11.Prophet Save Outputs
  L1. LSTM Setup & Scaling
  L2. LSTM Reshape
  L3. LSTM Model
  L4. LSTM Training
  L5. LSTM Evaluation
  E1. Ensemble Bridge       (align ARIMA/Prophet/LSTM to same scale)
  E2. LSTM + ARIMA Ensemble
  E3. LSTM + Prophet Ensemble
  E4. ARIMA + Prophet Ensemble
  E5. Ensemble Comparison
"""

# ──────────────────────────────────────────────────────────────
# IMPORTS
# ──────────────────────────────────────────────────────────────
!pip install pmdarima

import warnings
warnings.filterwarnings('ignore')

import os
import random

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

from pmdarima import auto_arima
from prophet import Prophet
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.optimizers import Adam
import tensorflow.keras.backend as K
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.metrics import (
    mean_absolute_error, mean_squared_error, r2_score,
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, confusion_matrix
)

# ──────────────────────────────────────────────────────────────
# SHARED HELPERS
# ──────────────────────────────────────────────────────────────

def get_classification_report_ts(actual_counts, pred_counts, total_loans_series,
                                  threshold=None, label=""):
    actual_counts = np.array(actual_counts, dtype=float)
    pred_counts   = np.array(pred_counts,   dtype=float)
    loans         = np.array(total_loans_series, dtype=float)

    actual_rate = actual_counts / (loans + 1e-9)
    pred_rate   = pred_counts   / (loans + 1e-9)

    if threshold is None:
        threshold = np.median(actual_rate)

    y_true  = (actual_rate >= threshold).astype(int)
    y_pred  = (pred_rate   >= threshold).astype(int)
    y_score = pred_rate

    n_pos = y_true.sum()
    n_neg = (1 - y_true).sum()
    can_auc = (n_pos > 0) and (n_neg > 0)

    acc   = accuracy_score(y_true,  y_pred)
    prec  = precision_score(y_true, y_pred, zero_division=0)
    rec   = recall_score(y_true,    y_pred, zero_division=0)
    f1    = f1_score(y_true,        y_pred, zero_division=0)
    auc   = roc_auc_score(y_true,   y_score) if can_auc else float('nan')
    cm    = confusion_matrix(y_true, y_pred, labels=[0, 1])

    W_m, W_v = 20, 22
    sep = f"  ├{'─'*W_m}┼{'─'*W_v}┤"
    print(f"\n  ┌{'─'*W_m}┬{'─'*W_v}┐")
    print(f"  │ {('['+label+'] Classification'):<{W_m-2}} │ {'Value':>{W_v-2}} │")
    print(sep)
    print(f"  │ {'Threshold (rate)':<{W_m-2}} │ {threshold:>{W_v-2}.4f} │")
    print(f"  │ {'Positives (high)':<{W_m-2}} │ {n_pos:>{W_v-2}d} │")
    print(f"  │ {'Negatives (low)':<{W_m-2}} │ {n_neg:>{W_v-2}d} │")
    print(sep)
    print(f"  │ {'Accuracy':<{W_m-2}} │ {acc:>{W_v-2}.4f} │")
    print(f"  │ {'Precision':<{W_m-2}} │ {prec:>{W_v-2}.4f} │")
    print(f"  │ {'Recall':<{W_m-2}} │ {rec:>{W_v-2}.4f} │")
    print(f"  │ {'F1-Score':<{W_m-2}} │ {f1:>{W_v-2}.4f} │")
    print(f"  │ {'ROC-AUC':<{W_m-2}} │ {('N/A' if not can_auc else f'{auc:.4f}'):>{W_v-2}} │")
    print(f"  └{'─'*W_m}┴{'─'*W_v}┘")
    print(f"  Confusion matrix → TN={cm[0,0]} FP={cm[0,1]} FN={cm[1,0]} TP={cm[1,1]}")
    print(f"  Actual  labels : {y_true.tolist()}")
    print(f"  Predicted labels: {y_pred.tolist()}")

    return {
        'threshold': threshold, 'accuracy': acc, 'precision': prec,
        'recall': rec, 'f1': f1, 'auc': auc,
        'tn': cm[0,0], 'fp': cm[0,1], 'fn': cm[1,0], 'tp': cm[1,1],
    }


def get_report(actual, pred, label):
    actual, pred = np.array(actual), np.array(pred)
    mean_actual  = np.abs(actual).mean() + 1e-9
    r2    = r2_score(actual, pred)
    mae   = mean_absolute_error(actual, pred) / mean_actual
    rmse  = np.sqrt(mean_squared_error(actual, pred)) / mean_actual
    smape = np.mean(2 * np.abs(actual - pred) / (np.abs(actual) + np.abs(pred) + 1e-9)) * 100
    mape  = np.mean(np.abs((actual - pred) / (np.abs(actual) + 1e-9))) * 100

    def r2_label(v):
        if v >= 0.90: return "Excellent ✓"
        if v >= 0.75: return "Good ✓"
        if v >= 0.50: return "Moderate"
        if v >= 0.00: return "Weak (positive)"
        return "Poor (negative)"

    W_m, W_v = 18, 22
    sep = f"  ├{'─' * W_m}┼{'─' * W_v}┤"
    print(f"\n  ┌{'─' * W_m}┬{'─' * W_v}┐")
    print(f"  │ {('[' + label + ']'):<{W_m - 2}} │ {'Value':>{W_v - 2}} │")
    print(sep)
    print(f"  │ {'R²':<{W_m - 2}} │ {r2:>{W_v - 2}.4f} │  → {r2_label(r2)}")
    print(f"  │ {'MAE (norm)':<{W_m - 2}} │ {mae:>{W_v - 2}.4f} │")
    print(f"  │ {'RMSE (norm)':<{W_m - 2}} │ {rmse:>{W_v - 2}.4f} │")
    print(f"  │ {'sMAPE':<{W_m - 2}} │ {smape:>{W_v - 3}.4f}% │")
    print(f"  │ {'MAPE':<{W_m - 2}} │ {mape:>{W_v - 3}.4f}% │")
    print(f"  └{'─' * W_m}┴{'─' * W_v}┘")
    return {'r2': r2, 'mae': mae, 'rmse': rmse, 'smape': smape, 'mape': mape}


def fmt_axis(ax, interval=12):
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
    ax.xaxis.set_major_locator(mdates.MonthLocator(interval=interval))
    plt.setp(ax.xaxis.get_majorticklabels(), rotation=45)


def batch_predict(model, X, batch_size=10000):
    preds = []
    for i in range(0, len(X), batch_size):
        preds.append(model.predict(X[i:i+batch_size], verbose=0))
    return np.vstack(preds).ravel()


def evaluate_model(y_train, y_train_pred, y_train_pred_proba,
                   y_test,  y_test_pred,  y_test_pred_proba, model_name):
    def reg_metrics(y_true, y_prob):
        y_true = np.array(y_true, dtype=float)
        y_prob = np.array(y_prob, dtype=float)
        r2   = r2_score(y_true, y_prob)
        mae  = mean_absolute_error(y_true, y_prob)
        rmse = np.sqrt(mean_squared_error(y_true, y_prob))

        pos_mask = y_true == 1
        if pos_mask.sum() > 0:
            y_pos  = y_true[pos_mask]
            p_pos  = y_prob[pos_mask]
            smape  = np.mean(2 * np.abs(y_pos - p_pos) /
                             (np.abs(y_pos) + np.abs(p_pos) + 1e-9)) * 100
            mape   = np.mean(np.abs((y_pos - p_pos) /
                                    (np.abs(y_pos) + 1e-9))) * 100
        else:
            smape, mape = np.nan, np.nan
        return r2, mae, rmse, smape, mape

    tr_r2, tr_mae, tr_rmse, tr_smape, tr_mape = reg_metrics(y_train, y_train_pred_proba)
    te_r2, te_mae, te_rmse, te_smape, te_mape = reg_metrics(y_test,  y_test_pred_proba)

    tr_acc  = accuracy_score(y_train,  y_train_pred)
    tr_prec = precision_score(y_train, y_train_pred, zero_division=0)
    tr_rec  = recall_score(y_train,    y_train_pred, zero_division=0)
    tr_f1   = f1_score(y_train,        y_train_pred, zero_division=0)
    tr_auc  = roc_auc_score(y_train,   y_train_pred_proba)

    te_acc  = accuracy_score(y_test,   y_test_pred)
    te_prec = precision_score(y_test,  y_test_pred, zero_division=0)
    te_rec  = recall_score(y_test,     y_test_pred, zero_division=0)
    te_f1   = f1_score(y_test,         y_test_pred, zero_division=0)
    te_auc  = roc_auc_score(y_test,    y_test_pred_proba)
    te_cm   = confusion_matrix(y_test, y_test_pred)

    W_m, W_v = 18, 12
    title_w  = W_m + W_v * 3 + 2
    sep_top  = f"  ├{'─' * W_m}┬{'─' * W_v}┬{'─' * W_v}┬{'─' * W_v}┤"
    sep_mid  = f"  ├{'─' * W_m}┼{'─' * W_v}┼{'─' * W_v}┼{'─' * W_v}┤"
    sep_bot  = f"  └{'─' * W_m}┴{'─' * W_v}┴{'─' * W_v}┴{'─' * W_v}┘"

    def row(name, tr, te):
        if np.isnan(tr) or np.isnan(te):
            tr_s  = f"{'N/A':>{W_v - 2}}"
            te_s  = f"{'N/A':>{W_v - 2}}"
            dif_s = f"{'N/A':>{W_v - 2}}"
        else:
            diff  = tr - te
            tr_s  = f"{tr:>{W_v - 2}.4f}"
            te_s  = f"{te:>{W_v - 2}.4f}"
            dif_s = f"{diff:>{W_v - 2}.4f}"
        return (f"  │ {name:<{W_m - 2}} │ {tr_s} │ {te_s} │ {dif_s} │")

    def section(title):
        return (f"  │ {title:<{W_m - 2}} │ {'':>{W_v - 2}} │"
                f" {'':>{W_v - 2}} │ {'':>{W_v - 2}} │")

    print(f"\n  ┌{'─' * title_w}┐")
    print(f"  │ {(model_name + ' — Evaluation'):<{title_w}} │")
    print(sep_top)
    print(f"  │ {'Metric':<{W_m - 2}} │ {'Train':>{W_v - 2}} │ {'Test':>{W_v - 2}} │ {'Difference':>{W_v - 2}} │")
    print(sep_mid)
    print(section('── Regression ──'))
    print(row('R²',        tr_r2,    te_r2))
    print(row('MAE (0-1)',  tr_mae,   te_mae))
    print(row('RMSE (0-1)', tr_rmse,  te_rmse))
    print(row('sMAPE (%)',  tr_smape, te_smape))
    print(row('MAPE (%)',   tr_mape,  te_mape))
    print(sep_mid)
    print(section('── Classification ──'))
    print(row('Accuracy',  tr_acc,  te_acc))
    print(row('Precision', tr_prec, te_prec))
    print(row('Recall',    tr_rec,  te_rec))
    print(row('F1-Score',  tr_f1,   te_f1))
    print(row('ROC-AUC',   tr_auc,  te_auc))
    print(sep_bot)
    print(f"  Confusion Matrix (Test): TN={te_cm[0,0]:,} | FP={te_cm[0,1]:,} | FN={te_cm[1,0]:,} | TP={te_cm[1,1]:,}")
    print(f"  Default Rate — Train: {y_train.mean():.4f}  predicted: {y_train_pred.mean():.4f} │ "
          f"Test: {y_test.mean():.4f}  predicted: {y_test_pred.mean():.4f}")

    return {
        'Model':           model_name,
        'Train_R2':        tr_r2,    'Test_R2':        te_r2,
        'Train_MAE':       tr_mae,   'Test_MAE':       te_mae,
        'Train_RMSE':      tr_rmse,  'Test_RMSE':      te_rmse,
        'Train_sMAPE':     tr_smape, 'Test_sMAPE':     te_smape,
        'Train_MAPE':      tr_mape,  'Test_MAPE':      te_mape,
        'Train_Accuracy':  tr_acc,   'Test_Accuracy':  te_acc,
        'Train_Precision': tr_prec,  'Test_Precision': te_prec,
        'Train_Recall':    tr_rec,   'Test_Recall':    te_rec,
        'Train_F1':        tr_f1,    'Test_F1':        te_f1,
        'Train_ROC_AUC':   tr_auc,   'Test_ROC_AUC':   te_auc,
        'Overfitting_Gap': tr_acc - te_acc,
    }


def focal_loss(gamma=2.0, alpha=0.75):
    def loss_fn(y_true, y_pred):
        y_pred  = K.clip(y_pred, K.epsilon(), 1.0 - K.epsilon())
        p_t     = tf.where(tf.equal(y_true, 1), y_pred, 1 - y_pred)
        alpha_t = tf.where(tf.equal(y_true, 1), alpha, 1 - alpha)
        fl      = -alpha_t * K.pow(1.0 - p_t, gamma) * K.log(p_t)
        return K.mean(fl)
    return loss_fn


# ══════════════════════════════════════════════════════════════
# 1. DATA LOADING & MERGING  (2007–2020)
# ══════════════════════════════════════════════════════════════
print("--- Step 1: Merging Datasets ---")

df_old = pd.read_csv('/kaggle/input/datasets/thandarphyo/ze-data/LC_loans_granting_model_dataset (4).csv')
df_old['issue_d'] = pd.to_datetime(df_old['issue_d'], errors='coerce')
df_old = df_old.dropna(subset=['issue_d'])
df_old = df_old[df_old['issue_d'] < '2016-01-01']

print(f"  df_old columns : {df_old.columns.tolist()}")
print(f"  df_old shape   : {df_old.shape}")
print(f"  df_old range   : {df_old['issue_d'].min()} → {df_old['issue_d'].max()}")

if 'Default' not in df_old.columns:
    raise ValueError(
        f"'Default' column not found in df_old.\n"
        f"Available columns: {df_old.columns.tolist()}"
    )

df = pd.read_csv('/kaggle/working/LendingClub_Cleaned_Final.csv')
df['issue_d'] = pd.to_datetime(df['issue_d'], errors='coerce')
df = df.dropna(subset=['issue_d'])

ts_old = (
    df_old.groupby(df_old['issue_d'].dt.to_period('M'))
    .agg(total_loans=('Default', 'count'), total_defaults=('Default', 'sum'))
    .reset_index()
    .rename(columns={'issue_d': 'ym'})
)

ts_new = (
    df.groupby(df['issue_d'].dt.to_period('M'))
    .agg(total_loans=('Default', 'count'), total_defaults=('Default', 'sum'))
    .reset_index()
    .rename(columns={'issue_d': 'ym'})
)

print(f"\n  ts_old : {len(ts_old)} months  ({ts_old['ym'].min()} → {ts_old['ym'].max()})")
print(f"  ts_new : {len(ts_new)} months  ({ts_new['ym'].min()} → {ts_new['ym'].max()})")

monthly_ts = pd.concat([ts_old, ts_new], axis=0, ignore_index=True)
monthly_ts['month'] = monthly_ts['ym'].dt.to_timestamp()
monthly_ts = monthly_ts.sort_values('month').reset_index(drop=True)

dupes = monthly_ts['month'].duplicated().sum()
if dupes > 0:
    print(f"\n  ⚠ {dupes} duplicate months found — merging by summing...")
    monthly_ts = (
        monthly_ts.groupby('month', as_index=False)
        .agg(total_loans=('total_loans', 'sum'), total_defaults=('total_defaults', 'sum'))
    )
    monthly_ts = monthly_ts.sort_values('month').reset_index(drop=True)
    print(f"  After dedup: {len(monthly_ts)} months")

monthly_ts = monthly_ts[monthly_ts['total_loans'] >= 1000].reset_index(drop=True)
monthly_ts['total_defaults_smooth'] = (
    monthly_ts['total_defaults']
    .rolling(window=3, min_periods=1, center=False)
    .mean()
)

covid_start = pd.Timestamp('2020-03-01')
ts_model    = monthly_ts[monthly_ts['month'] < covid_start].copy().reset_index(drop=True)

series  = ts_model['total_defaults_smooth'].values
n_total = len(series)

print(f"\n  After filtering  : {len(monthly_ts)} months total")
print(f"  Pre-COVID series : {n_total} months")
print(f"  Period : {ts_model['month'].iloc[0].strftime('%Y-%m')} → {ts_model['month'].iloc[-1].strftime('%Y-%m')}")
print(f"  Avg total defaults : {series.mean():.1f}")
print(f"  Range  : {series.min():.0f} – {series.max():.0f}")

if n_total < 30:
    raise ValueError(f"Only {n_total} months after filtering — need ≥30.")


# ══════════════════════════════════════════════════════════════
# 2. LOG TRANSFORM + CUBIC DETRENDING
# ══════════════════════════════════════════════════════════════
print("\n--- Step 2: Log Transform + Cubic Detrending ---")

log_series = np.log(series)

t  = np.arange(n_total).reshape(-1, 1)
t3 = np.column_stack([t, t**2, t**3])

cubic_reg     = LinearRegression().fit(t3, log_series)
trend_fit     = cubic_reg.predict(t3)
r2_trend      = r2_score(log_series, trend_fit)
log_detrended = log_series - trend_fit

print(f"  Cubic trend R² (log-space) : {r2_trend:.4f}")
print(f"  Trend direction (log)      : {trend_fit[0]:.3f} → {trend_fit[-1]:.3f}")
print(f"  Original log-std           : {log_series.std():.4f}")
print(f"  Detrended log-std          : {log_detrended.std():.4f}")
print(f"  Variance explained         : {(1 - log_detrended.var() / log_series.var()) * 100:.1f}%")


# ══════════════════════════════════════════════════════════════
# 3. TRAIN / TEST SPLIT
# ══════════════════════════════════════════════════════════════
print("\n--- Step 3: Train / Test Split ---")

test_size  = 12
train_size = n_total - test_size

if train_size < 24:
    raise ValueError(f"Train size = {train_size} months — need ≥24.")

train_detrended = log_detrended[:train_size]
test_detrended  = log_detrended[train_size:]
trend_train     = trend_fit[:train_size]

train_series = series[:train_size]
test_series  = series[train_size:]
train_ts     = ts_model.iloc[:train_size].reset_index(drop=True)
test_ts      = ts_model.iloc[train_size:].reset_index(drop=True)

print(f"  Train : {train_ts['month'].iloc[0].strftime('%Y-%m')} → "
      f"{train_ts['month'].iloc[-1].strftime('%Y-%m')} ({train_size} months)")
print(f"  Test  : {test_ts['month'].iloc[0].strftime('%Y-%m')} → "
      f"{test_ts['month'].iloc[-1].strftime('%Y-%m')} ({test_size} months)")


# ══════════════════════════════════════════════════════════════
# 4. AUTO ARIMA ON LOG-DETRENDED SERIES
# ══════════════════════════════════════════════════════════════
print("\n--- Step 4: Training Auto ARIMA (log-detrended) ---")

use_seasonal = train_size >= 36
m_val        = 12 if use_seasonal else 1
print(f"  Seasonal : {'YES  m=12' if use_seasonal else 'NO — train < 36 months'}")

model_arima = auto_arima(
    train_detrended,
    seasonal=use_seasonal,
    m=m_val,
    max_p=4, max_q=4,
    max_P=2, max_Q=2,
    d=None, D=None,
    information_criterion='aic',
    stepwise=True,
    suppress_warnings=True,
    error_action='ignore',
    trace=True
)

order_str = f"SARIMA{model_arima.order}x{model_arima.seasonal_order}"
print(f"\n  Best model : {order_str}")
print(f"  AIC : {model_arima.aic():.4f}  |  BIC : {model_arima.bic():.4f}")
print(model_arima.summary())


# ══════════════════════════════════════════════════════════════
# 5. ARIMA TRAIN FITTED VALUES
# ══════════════════════════════════════════════════════════════
print("\n--- Step 5: Train Fitted Values ---")

train_preds_arima = np.exp(model_arima.predict_in_sample()[:train_size] + trend_train)
print(f"  Fitted values: {len(train_preds_arima)} months ✓")


# ══════════════════════════════════════════════════════════════
# 6. ARIMA ROLLING WALK-FORWARD VALIDATION
# ══════════════════════════════════════════════════════════════
print("\n--- Step 6: Rolling Walk-Forward Test Evaluation ---")

history_det    = list(train_detrended)
test_preds_det = []

for i in range(test_size):
    try:
        temp_model = auto_arima(
            history_det,
            order=model_arima.order,
            seasonal_order=model_arima.seasonal_order,
            suppress_warnings=True,
            error_action='ignore'
        )
        p_det = float(temp_model.predict(n_periods=1)[0])
    except Exception as e:
        print(f"  ⚠ Step {i+1}: refit failed ({e}) — using last value")
        p_det = history_det[-1]

    t_idx      = train_size + i
    t_val      = float(cubic_reg.predict([[t_idx, t_idx**2, t_idx**3]])[0])
    pred_final = np.exp(p_det + t_val)

    test_preds_det.append(p_det)
    history_det.append(test_detrended[i])

    month_str = test_ts['month'].iloc[i].strftime('%Y-%m')
    print(f"  {month_str} | actual={test_series[i]:.1f}  pred={pred_final:.1f}  "
          f"err={test_series[i] - pred_final:+.1f}")

test_preds_det   = np.array(test_preds_det)
t_test_idx       = np.arange(train_size, n_total).reshape(-1, 1)
t_test_feat      = np.column_stack([t_test_idx, t_test_idx**2, t_test_idx**3])
trend_test_vals  = cubic_reg.predict(t_test_feat)
test_preds_arima = np.exp(test_preds_det + trend_test_vals)


# ══════════════════════════════════════════════════════════════
# 7. ARIMA EVALUATION METRICS
# ══════════════════════════════════════════════════════════════
print("\n--- Step 7: Evaluation Metrics ---")

m_arima_train = get_report(train_series, train_preds_arima, "TRAIN (In-Sample)")
m_arima_test  = get_report(test_series,  test_preds_arima,  "TEST  (Rolling Walk-Forward)")

print(f"\n  {'':=<50}")
print(f"  {'METRIC':<15} | {'TRAIN':>12} | {'TEST':>12}")
print(f"  {'-'*44}")
for metric, label, unit in [
    ('r2',    'R-Squared', ''),
    ('mae',   'MAE',       ''),
    ('rmse',  'RMSE',      ''),
    ('smape', 'sMAPE (%)', '%'),
    ('mape',  'MAPE (%)',  '%'),
]:
    print(f"  {label:<15} | {m_arima_train[metric]:>11.4f}{unit} | "
          f"{m_arima_test[metric]:>11.4f}{unit}")
print(f"  {'':=<44}")

naive_mae = np.mean(np.abs(np.diff(test_series)))
print(f"\n  Naive baseline MAE : {naive_mae:.2f} defaults")
print(f"  ARIMA test MAE     : {m_arima_test['mae']:.2f} defaults")
print(f"  → ARIMA {'✓ BEATS' if m_arima_test['mae'] < naive_mae else '✗ does NOT beat'} naive")

arima_threshold = np.median(
    ts_model['total_defaults_smooth'].values /
    (ts_model['total_loans'].values + 1e-9)
)
print("\n  ── ARIMA Classification (train) ──")
cm_arima_train = get_classification_report_ts(
    actual_counts      = train_series,
    pred_counts        = train_preds_arima,
    total_loans_series = train_ts['total_loans'].values,
    threshold          = arima_threshold,
    label              = "ARIMA TRAIN"
)

print("\n  ── ARIMA Classification (test) ──")
cm_arima_test = get_classification_report_ts(
    actual_counts      = test_series,
    pred_counts        = test_preds_arima,
    total_loans_series = test_ts['total_loans'].values,
    threshold          = arima_threshold,
    label              = "ARIMA TEST"
)


# ══════════════════════════════════════════════════════════════
# 8. ARIMA 12-MONTH FORECAST
# ══════════════════════════════════════════════════════════════
print("\n--- Step 8: 12-Month Forecast ---")

final_arima = auto_arima(
    log_detrended,
    order=model_arima.order,
    seasonal_order=model_arima.seasonal_order,
    suppress_warnings=True,
    error_action='ignore'
)

fc_det, conf_int = final_arima.predict(n_periods=12, return_conf_int=True, alpha=0.05)

t_fc_idx  = np.arange(n_total, n_total + 12).reshape(-1, 1)
t_fc_feat = np.column_stack([t_fc_idx, t_fc_idx**2, t_fc_idx**3])
trend_fc  = cubic_reg.predict(t_fc_feat)

fc_values = np.exp(fc_det          + trend_fc)
fc_lower  = np.exp(conf_int[:, 0]  + trend_fc)
fc_upper  = np.exp(conf_int[:, 1]  + trend_fc)

last_date             = ts_model['month'].iloc[-1]
arima_forecast_dates  = pd.date_range(
    start=last_date + pd.DateOffset(months=1), periods=12, freq='MS'
)
arima_forecast_df = pd.DataFrame({
    'month'             : arima_forecast_dates,
    'forecast_defaults' : fc_values,
    'lower_95ci'        : fc_lower,
    'upper_95ci'        : fc_upper
})

print(f"\n  {'Month':<12} {'Forecast':>12} {'Lower 95%':>12} {'Upper 95%':>12}")
print(f"  {'-'*50}")
for _, row in arima_forecast_df.iterrows():
    print(f"  {row['month'].strftime('%Y-%m'):<12}"
          f" {row['forecast_defaults']:>12.1f}"
          f" {row['lower_95ci']:>12.1f}"
          f" {row['upper_95ci']:>12.1f}")
print(f"\n  Avg={fc_values.mean():.1f}  Min={fc_values.min():.1f}  Max={fc_values.max():.1f}")


# ══════════════════════════════════════════════════════════════
# 9. ARIMA VISUALIZATION
# ══════════════════════════════════════════════════════════════
print("\n--- Step 9: Visualizations ---")

fig, axes = plt.subplots(5, 1, figsize=(16, 28))
fig.suptitle(
    f'LendingClub Total Defaults — Log-ARIMA + Cubic Detrending\n'
    f'Model: {order_str} | 2007–2020',
    fontsize=14, fontweight='bold', y=0.99
)

ax = axes[0]
ax.plot(monthly_ts['month'], monthly_ts['total_defaults'],
        color='lightsteelblue', lw=1, alpha=0.6, label='Raw Defaults')
ax.plot(ts_model['month'], series,
        color='steelblue', lw=2, label='Smoothed (pre-COVID)')
ax.plot(ts_model['month'], np.exp(trend_fit),
        color='red', lw=1.5, ls='--', label=f'Cubic Trend (log R²={r2_trend:.4f})')
ax.axvspan(covid_start, monthly_ts['month'].max() + pd.DateOffset(months=1),
           color='red', alpha=0.07, label='COVID (excluded)')
ax.axvline(train_ts['month'].iloc[-1], color='orange', ls='--', lw=1.5,
           label='Train/Test Split')
ax.set_title('Full History + Cubic Trend', fontsize=12, fontweight='bold')
ax.set_ylabel('Total Defaults'); ax.grid(True, alpha=0.3); ax.legend(fontsize=9)
fmt_axis(ax, 12)

ax = axes[1]
ax.plot(ts_model['month'], log_detrended,
        color='purple', lw=2, marker='o', ms=2, label='Log-Detrended Series')
ax.axhline(0, color='black', ls='--', lw=1)
ax.axvline(train_ts['month'].iloc[-1], color='orange', ls='--', lw=1.5,
           label='Train/Test Split')
ax.set_title('Log-Detrended Series', fontsize=12, fontweight='bold')
ax.set_ylabel('log(Defaults) residual'); ax.grid(True, alpha=0.3); ax.legend(fontsize=9)
fmt_axis(ax, 12)

ax = axes[2]
ax.plot(train_ts['month'], train_series,
        color='steelblue', lw=2, marker='o', ms=3, label='Actual (Train)')
ax.plot(train_ts['month'], train_preds_arima,
        color='green', lw=1.5, ls='--', label=f'Fitted  R²={m_arima_train["r2"]:.4f}')
ax.set_title(
    f'Train — Actual vs Fitted | R²={m_arima_train["r2"]:.4f} | MAE={m_arima_train["mae"]:.2f}',
    fontsize=11, fontweight='bold'
)
ax.set_ylabel('Total Defaults'); ax.grid(True, alpha=0.3); ax.legend(fontsize=9)
fmt_axis(ax, 6)

ax = axes[3]
ax.plot(test_ts['month'], test_series,
        color='steelblue', lw=2, marker='o', ms=5, label='Actual (Test)')
ax.plot(test_ts['month'], test_preds_arima,
        color='darkorange', lw=1.5, ls='--', marker='s', ms=5,
        label=f'Rolling Pred  R²={m_arima_test["r2"]:.4f}')
ax.set_title(
    f'Test — Actual vs Rolling Forecast | R²={m_arima_test["r2"]:.4f} | '
    f'MAE(norm)={m_arima_test["mae"]:.4f}  MAPE={m_arima_test["mape"]:.2f}%',
    fontsize=11, fontweight='bold'
)
ax.set_ylabel('Total Defaults'); ax.grid(True, alpha=0.3); ax.legend(fontsize=9)
fmt_axis(ax, 1)

ax = axes[4]
ctx = ts_model.tail(18)
ax.plot(ctx['month'], ctx['total_defaults_smooth'],
        color='steelblue', lw=2, marker='o', ms=4, label='Historical (last 18 months)')
ax.plot(arima_forecast_df['month'], arima_forecast_df['forecast_defaults'],
        color='crimson', lw=2, ls='--', marker='D', ms=5, label='12-Month Forecast')
ax.fill_between(arima_forecast_df['month'],
                arima_forecast_df['lower_95ci'], arima_forecast_df['upper_95ci'],
                color='crimson', alpha=0.15, label='95% CI')
ax.axvline(last_date, color='gray', ls=':', lw=1.5, label='Forecast Start')
ax.set_title('12-Month Ahead Forecast', fontsize=12, fontweight='bold')
ax.set_ylabel('Total Defaults'); ax.grid(True, alpha=0.3); ax.legend(fontsize=9)
fmt_axis(ax, 1)

plt.tight_layout(rect=[0, 0, 1, 0.98])
plt.savefig('/kaggle/working/arima_final.png', dpi=120, bbox_inches='tight')
print("  ✓ Saved: arima_final.png")
plt.show()


# ══════════════════════════════════════════════════════════════
# 10. ARIMA SAVE OUTPUTS
# ══════════════════════════════════════════════════════════════
arima_forecast_df.to_csv('/kaggle/working/arima_forecast.csv', index=False)

pd.DataFrame({
    'month'              : list(train_ts['month'])    + list(test_ts['month']),
    'actual_defaults'    : list(train_series)         + list(test_series),
    'predicted_defaults' : list(train_preds_arima)    + list(test_preds_arima),
    'set'                : ['train'] * train_size     + ['test']  * test_size
}).to_csv('/kaggle/working/arima_fitted_vs_actual.csv', index=False)

print("  ✓ Saved: arima_forecast.csv")
print("  ✓ Saved: arima_fitted_vs_actual.csv")
print("\n✓ ARIMA pipeline complete!")


# ══════════════════════════════════════════════════════════════
# P1. PROPHET — INPUT PREP
# ══════════════════════════════════════════════════════════════
print("\n--- Prophet P1: Preparing Input ---")

base_df = ts_model[['month', 'total_defaults_smooth']].rename(
    columns={'month': 'ds', 'total_defaults_smooth': 'y'}
).copy()
base_df['y'] = np.log(base_df['y'])

base_df['momentum_3m'] = base_df['y'].diff(3)
base_df['momentum_6m'] = base_df['y'].diff(6)
base_df['yoy_change']  = base_df['y'].diff(12)

prophet_df = base_df.dropna().reset_index(drop=True)

PROPHET_REG_COLS = ['momentum_3m', 'momentum_6m', 'yoy_change']
prophet_reg_scaler = {}
for col in PROPHET_REG_COLS:
    mu  = prophet_df[col].mean()
    std = prophet_df[col].std() + 1e-9
    prophet_df[col] = (prophet_df[col] - mu) / std
    prophet_reg_scaler[col] = {'mu': mu, 'std': std}

base_df_noreg = ts_model[['month', 'total_defaults_smooth']].rename(
    columns={'month': 'ds', 'total_defaults_smooth': 'y'}
).copy()
base_df_noreg['y'] = np.log(base_df_noreg['y'])

n_total_reg   = len(prophet_df)
n_total_noreg = len(base_df_noreg)
p_test_size   = 12
p_train_size  = n_total_reg - p_test_size

p_train_df = prophet_df.iloc[:p_train_size].reset_index(drop=True)
p_test_df  = prophet_df.iloc[p_train_size:].reset_index(drop=True)

p_train_series = np.exp(p_train_df['y'].values)
p_test_series  = np.exp(p_test_df['y'].values)

log_raw_full = np.log(ts_model['total_defaults_smooth'].values)
start_offset = len(ts_model) - len(prophet_df)

print(f"  Model A (with regressors) — {n_total_reg} months after lag drop")
print(f"  Model B (forecast only)   — {n_total_noreg} months")
print(f"  Train : {p_train_df['ds'].iloc[0].strftime('%Y-%m')} → "
      f"{p_train_df['ds'].iloc[-1].strftime('%Y-%m')} ({p_train_size} months)")
print(f"  Test  : {p_test_df['ds'].iloc[0].strftime('%Y-%m')} → "
      f"{p_test_df['ds'].iloc[-1].strftime('%Y-%m')} ({p_test_size} months)")


# ══════════════════════════════════════════════════════════════
# P2. PROPHET CHANGEPOINTS
# ══════════════════════════════════════════════════════════════
print("\n--- Prophet P2: Changepoints ---")

known_changepoints = ['2013-01-01', '2015-06-01', '2018-08-01']
print(f"  Changepoints : {known_changepoints}")


# ══════════════════════════════════════════════════════════════
# P3. PROPHET MODEL BUILDERS
# ══════════════════════════════════════════════════════════════
def build_prophet_with_regs(changepoints=None):
    m = Prophet(
        growth='linear',
        yearly_seasonality=10,
        weekly_seasonality=False,
        daily_seasonality=False,
        changepoint_prior_scale=0.5,
        seasonality_prior_scale=10,
        seasonality_mode='additive',
        changepoints=changepoints,
        interval_width=0.95
    )
    for col in PROPHET_REG_COLS:
        m.add_regressor(col, standardize=False)
    return m


def build_prophet_noreg(changepoints=None):
    return Prophet(
        growth='linear',
        yearly_seasonality=10,
        weekly_seasonality=False,
        daily_seasonality=False,
        changepoint_prior_scale=0.5,
        seasonality_prior_scale=10,
        seasonality_mode='additive',
        changepoints=changepoints,
        interval_width=0.95
    )


# ══════════════════════════════════════════════════════════════
# P4. TRAIN MODEL A
# ══════════════════════════════════════════════════════════════
print("\n--- Prophet P4: Training Model A (with regressors) ---")

model_A = build_prophet_with_regs(changepoints=known_changepoints)
model_A.fit(p_train_df)
print("  ✓ Model A fitted")

train_fc_raw = model_A.predict(p_train_df)
log_residuals_A = p_train_df['y'].values - train_fc_raw['yhat'].values
sigma2_A = np.var(log_residuals_A)
print(f"  Log-space σ² : {sigma2_A:.6f}  (bias corr: exp(+{sigma2_A/2:.6f}))")


# ══════════════════════════════════════════════════════════════
# P5. PROPHET TRAIN FITTED VALUES
# ══════════════════════════════════════════════════════════════
print("\n--- Prophet P5: Train Fitted Values ---")

train_preds_prophet = np.exp(train_fc_raw['yhat'].values + sigma2_A / 2)
print(f"  Fitted values: {len(train_preds_prophet)} months ✓")


# ══════════════════════════════════════════════════════════════
# P6. PROPHET ROLLING WALK-FORWARD TEST EVALUATION
# ══════════════════════════════════════════════════════════════
print("\n--- Prophet P6: Rolling Walk-Forward Test Evaluation ---")

test_preds_prophet = []

for i in range(p_test_size):
    window_end_prop = p_train_size + i
    window_end_raw  = start_offset + window_end_prop

    log_win  = log_raw_full[:window_end_raw + 1]
    mom3_raw = pd.Series(log_win).diff(3).values
    mom6_raw = pd.Series(log_win).diff(6).values
    yoy_raw  = pd.Series(log_win).diff(12).values

    hist_df = prophet_df.iloc[:window_end_prop].copy()

    fold_scaler = {}
    for col, raw_arr in [
        ('momentum_3m', mom3_raw),
        ('momentum_6m', mom6_raw),
        ('yoy_change',  yoy_raw),
    ]:
        raw_win  = raw_arr[start_offset: start_offset + window_end_prop]
        valid    = raw_win[~np.isnan(raw_win)]
        mu_fold  = valid.mean()
        std_fold = valid.std() + 1e-9
        hist_df[col]     = (raw_win - mu_fold) / std_fold
        fold_scaler[col] = {'mu': mu_fold, 'std': std_fold}

    history_end = hist_df['ds'].iloc[-1]
    active_cps  = [cp for cp in known_changepoints if pd.Timestamp(cp) <= history_end]

    m = build_prophet_with_regs(changepoints=active_cps if active_cps else None)
    m.fit(hist_df)

    fc_hist  = m.predict(hist_df)
    s2_fold  = np.var(hist_df['y'].values - fc_hist['yhat'].values)

    future_one = p_test_df[['ds']].iloc[[i]].copy()
    next_idx   = window_end_raw

    for col, raw_arr in [
        ('momentum_3m', mom3_raw),
        ('momentum_6m', mom6_raw),
        ('yoy_change',  yoy_raw),
    ]:
        raw_val = (
            raw_arr[next_idx]
            if next_idx < len(raw_arr) and not np.isnan(raw_arr[next_idx])
            else 0.0
        )
        future_one[col] = (raw_val - fold_scaler[col]['mu']) / fold_scaler[col]['std']

    fc_one     = m.predict(future_one)
    pred_final = np.exp(float(fc_one['yhat'].values[0]) + s2_fold / 2)
    test_preds_prophet.append(pred_final)

    month_str = p_test_df['ds'].iloc[i].strftime('%Y-%m')
    print(f"  {month_str} | actual={p_test_series[i]:.1f}  pred={pred_final:.1f}  "
          f"err={p_test_series[i] - pred_final:+.1f}")

test_preds_prophet = np.array(test_preds_prophet)

fc_test_A  = model_A.predict(p_test_df)
test_lower = np.exp(fc_test_A['yhat_lower'].values + sigma2_A / 2)
test_upper = np.exp(fc_test_A['yhat_upper'].values + sigma2_A / 2)


# ══════════════════════════════════════════════════════════════
# P7. PROPHET EVALUATION METRICS
# ══════════════════════════════════════════════════════════════
print("\n--- Prophet P7: Evaluation Metrics ---")

m_prophet_train = get_report(p_train_series, train_preds_prophet, "TRAIN (In-Sample)")
m_prophet_test  = get_report(p_test_series,  test_preds_prophet,  "TEST  (Rolling Walk-Forward)")

print(f"\n  {'':=<50}")
print(f"  {'METRIC':<15} | {'TRAIN':>12} | {'TEST':>12}")
print(f"  {'-'*44}")
for metric, label, unit in [
    ('r2',    'R-Squared', ''),
    ('mae',   'MAE',       ''),
    ('rmse',  'RMSE',      ''),
    ('smape', 'sMAPE (%)', '%'),
    ('mape',  'MAPE (%)',  '%'),
]:
    print(f"  {label:<15} | {m_prophet_train[metric]:>11.4f}{unit} | "
          f"{m_prophet_test[metric]:>11.4f}{unit}")
print(f"  {'':=<44}")

naive_mae = np.mean(np.abs(np.diff(p_test_series)))
print(f"\n  Naive baseline MAE : {naive_mae:.2f} defaults")
print(f"  Prophet test MAE   : {m_prophet_test['mae']:.2f} defaults")
print(f"  → Prophet {'✓ BEATS' if m_prophet_test['mae'] < naive_mae else '✗ does NOT beat'} naive")

prophet_train_loans = (
    ts_model.set_index('month')
    .reindex(p_train_df['ds'].values)['total_loans']
    .fillna(ts_model['total_loans'].median())
    .values
)
prophet_test_loans = (
    ts_model.set_index('month')
    .reindex(p_test_df['ds'].values)['total_loans']
    .fillna(ts_model['total_loans'].median())
    .values
)

print("\n  ── Prophet Classification (train) ──")
cm_prophet_train = get_classification_report_ts(
    actual_counts      = p_train_series,
    pred_counts        = train_preds_prophet,
    total_loans_series = prophet_train_loans,
    threshold          = arima_threshold,
    label              = "PROPHET TRAIN"
)

print("\n  ── Prophet Classification (test) ──")
cm_prophet_test = get_classification_report_ts(
    actual_counts      = p_test_series,
    pred_counts        = test_preds_prophet,
    total_loans_series = prophet_test_loans,
    threshold          = arima_threshold,
    label              = "PROPHET TEST"
)


# ══════════════════════════════════════════════════════════════
# P8. PROPHET 12-MONTH FORECAST  (Model B)
# ══════════════════════════════════════════════════════════════
print("\n--- Prophet P8: 12-Month Forecast (Model B, no regressors) ---")

model_B = build_prophet_noreg(changepoints=known_changepoints)
model_B.fit(base_df_noreg)

fc_B_train = model_B.predict(base_df_noreg[['ds']])
sigma2_B   = np.var(base_df_noreg['y'].values - fc_B_train['yhat'].values)

p_last_date       = base_df_noreg['ds'].iloc[-1]
prophet_fc_dates  = pd.date_range(
    start=p_last_date + pd.DateOffset(months=1), periods=12, freq='MS'
)
future_df = pd.DataFrame({'ds': prophet_fc_dates})

fc_out      = model_B.predict(future_df)
p_fc_values = np.exp(fc_out['yhat'].values       + sigma2_B / 2)
p_fc_lower  = np.exp(fc_out['yhat_lower'].values + sigma2_B / 2)
p_fc_upper  = np.exp(fc_out['yhat_upper'].values + sigma2_B / 2)

prophet_forecast_df = pd.DataFrame({
    'month'             : prophet_fc_dates,
    'forecast_defaults' : p_fc_values,
    'lower_95ci'        : p_fc_lower,
    'upper_95ci'        : p_fc_upper
})

print(f"\n  {'Month':<12} {'Forecast':>12} {'Lower 95%':>12} {'Upper 95%':>12}")
print(f"  {'-'*50}")
for _, row in prophet_forecast_df.iterrows():
    print(f"  {row['month'].strftime('%Y-%m'):<12}"
          f" {row['forecast_defaults']:>12.1f}"
          f" {row['lower_95ci']:>12.1f}"
          f" {row['upper_95ci']:>12.1f}")
print(f"\n  Avg={p_fc_values.mean():.1f}  Min={p_fc_values.min():.1f}  Max={p_fc_values.max():.1f}")


# ══════════════════════════════════════════════════════════════
# P9. PROPHET VISUALIZATION
# ══════════════════════════════════════════════════════════════
print("\n--- Prophet P9: Visualizations ---")

fig, axes = plt.subplots(5, 1, figsize=(16, 28))
fig.suptitle(
    'LendingClub Total Defaults — Prophet + Momentum Regressors\n'
    'Model A (eval) + Model B (forecast) | 2007–2020',
    fontsize=14, fontweight='bold', y=0.99
)

p_train_dates = p_train_df['ds'].values
p_test_dates  = p_test_df['ds'].values

ax = axes[0]
ax.plot(monthly_ts['month'], monthly_ts['total_defaults'],
        color='lightsteelblue', lw=1, alpha=0.6, label='Raw Defaults')
ax.plot(ts_model['month'], ts_model['total_defaults_smooth'],
        color='steelblue', lw=2, label='Smoothed (pre-COVID)')
ax.axvspan(covid_start, monthly_ts['month'].max() + pd.DateOffset(months=1),
           color='red', alpha=0.07, label='COVID (excluded)')
ax.axvline(p_train_df['ds'].iloc[-1], color='orange', ls='--', lw=1.5,
           label='Train/Test Split')
for cp in known_changepoints:
    ax.axvline(pd.Timestamp(cp), color='purple', lw=1, ls=':', alpha=0.7)
ax.set_title('Full History + Changepoints (purple)', fontsize=12, fontweight='bold')
ax.set_ylabel('Total Defaults'); ax.grid(True, alpha=0.3); ax.legend(fontsize=9)
fmt_axis(ax, 12)

ax = axes[1]
ax.plot(base_df_noreg['ds'], np.exp(base_df_noreg['y'].values),
        color='steelblue', lw=2, label='Actual')
ax.plot(base_df_noreg['ds'], np.exp(fc_B_train['trend'].values),
        color='red', lw=1.5, ls='--', label='Prophet Trend (Model B)')
for cp in known_changepoints:
    ax.axvline(pd.Timestamp(cp), color='purple', lw=1, ls=':', alpha=0.7)
ax.set_title('Prophet Trend + Changepoints (Model B)', fontsize=12, fontweight='bold')
ax.set_ylabel('Total Defaults'); ax.grid(True, alpha=0.3); ax.legend(fontsize=9)
fmt_axis(ax, 12)

ax = axes[2]
ax.plot(p_train_dates, p_train_series,
        color='steelblue', lw=2, marker='o', ms=3, label='Actual (Train)')
ax.plot(p_train_dates, train_preds_prophet,
        color='green', lw=1.5, ls='--', label=f'Fitted  R²={m_prophet_train["r2"]:.4f}')
ax.set_title(
    f'Train — Actual vs Fitted | R²={m_prophet_train["r2"]:.4f} | '
    f'MAE={m_prophet_train["mae"]:.2f}',
    fontsize=11, fontweight='bold'
)
ax.set_ylabel('Total Defaults'); ax.grid(True, alpha=0.3); ax.legend(fontsize=9)
fmt_axis(ax, 6)

ax = axes[3]
ax.plot(p_test_dates, p_test_series,
        color='steelblue', lw=2, marker='o', ms=5, label='Actual (Test)')
ax.plot(p_test_dates, test_preds_prophet,
        color='darkorange', lw=1.5, ls='--', marker='s', ms=5,
        label=f'Rolling Pred  R²={m_prophet_test["r2"]:.4f}')
ax.fill_between(p_test_dates, test_lower, test_upper,
                color='darkorange', alpha=0.15, label='95% CI (single-pass)')
ax.set_title(
    f'Test — Actual vs Rolling Forecast | R²={m_prophet_test["r2"]:.4f} | '
    f'MAE(norm)={m_prophet_test["mae"]:.4f}  MAPE={m_prophet_test["mape"]:.2f}%',
    fontsize=11, fontweight='bold'
)
ax.set_ylabel('Total Defaults'); ax.grid(True, alpha=0.3); ax.legend(fontsize=9)
fmt_axis(ax, 1)

ax = axes[4]
ctx = ts_model.tail(18)
ax.plot(ctx['month'], ctx['total_defaults_smooth'],
        color='steelblue', lw=2, marker='o', ms=4, label='Historical (last 18 months)')
ax.plot(prophet_forecast_df['month'], prophet_forecast_df['forecast_defaults'],
        color='crimson', lw=2, ls='--', marker='D', ms=5,
        label='12-Month Forecast (Model B)')
ax.fill_between(prophet_forecast_df['month'],
                prophet_forecast_df['lower_95ci'], prophet_forecast_df['upper_95ci'],
                color='crimson', alpha=0.15, label='95% CI')
ax.axvline(p_last_date, color='gray', ls=':', lw=1.5, label='Forecast Start')
ax.set_title('12-Month Ahead Forecast (Model B — no regressors)',
             fontsize=12, fontweight='bold')
ax.set_ylabel('Total Defaults'); ax.grid(True, alpha=0.3); ax.legend(fontsize=9)
fmt_axis(ax, 1)

plt.tight_layout(rect=[0, 0, 1, 0.98])
plt.savefig('/kaggle/working/prophet_final.png', dpi=120, bbox_inches='tight')
print("  ✓ Saved: prophet_final.png")
plt.show()


# ══════════════════════════════════════════════════════════════
# P10. PROPHET COMPONENTS PLOT
# ══════════════════════════════════════════════════════════════
fig_comp = model_B.plot_components(fc_B_train)
fig_comp.suptitle('Prophet Components (Model B)', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('/kaggle/working/prophet_components.png', dpi=120, bbox_inches='tight')
print("  ✓ Saved: prophet_components.png")
plt.show()


# ══════════════════════════════════════════════════════════════
# P11. PROPHET SAVE OUTPUTS
# ══════════════════════════════════════════════════════════════
prophet_forecast_df.to_csv('/kaggle/working/prophet_forecast.csv', index=False)

pd.DataFrame({
    'month'              : list(p_train_dates)       + list(p_test_dates),
    'actual_defaults'    : list(p_train_series)      + list(p_test_series),
    'predicted_defaults' : list(train_preds_prophet) + list(test_preds_prophet),
    'set'                : ['train'] * p_train_size  + ['test'] * p_test_size
}).to_csv('/kaggle/working/prophet_fitted_vs_actual.csv', index=False)

print("  ✓ Saved: prophet_forecast.csv")
print("  ✓ Saved: prophet_fitted_vs_actual.csv")
print("\n✓ Prophet pipeline complete!")


# ══════════════════════════════════════════════════════════════
# LSTM
# ══════════════════════════════════════════════════════════════
# ── Preserve monthly time-series references before LSTM
#    overwrites train_ts / test_ts with loan-level DataFrames ──
arima_test_ts     = test_ts.copy()
arima_test_series = test_series.copy()

print("=" * 100)
print("MODEL 7: LSTM  (IMPROVED — drop-in for unified pipeline)")
print("=" * 100)

SEED = 100
random.seed(SEED)
np.random.seed(SEED)
os.environ["PYTHONHASHSEED"] = str(SEED)
tf.random.set_seed(SEED)

df = pd.read_csv('/kaggle/working/LendingClub_Cleaned_Final.csv')
df['issue_d'] = pd.to_datetime(df['issue_d'], errors='coerce')
df = df.sort_values('issue_d').reset_index(drop=True)
print(f"✓ Loaded: {len(df):,} records")

# ── FIX: use isinstance() checks instead of 'not in globals()'
#         to guard against stale variables of the wrong type ──
if 'X_train' not in globals():
    cutoff_index = int(len(df) * 0.8)
    cutoff_date  = pd.to_datetime(df.loc[cutoff_index, 'issue_d'])
    train_ts     = df[df['issue_d'] <= cutoff_date]
    test_ts      = df[df['issue_d'] >  cutoff_date]
    X_train = train_ts.drop('Default', axis=1)
    X_test  = test_ts.drop('Default', axis=1)
    y_train = train_ts['Default']
    y_test  = test_ts['Default']
    print(f"✓ Split — Train:{len(X_train):,} | Test:{len(X_test):,}")
else:
    print(f"✓ Using existing split — Train:{len(X_train):,} | Test:{len(X_test):,}")

# ── KEY FIX: always reset accumulator containers to correct types ──
# Using isinstance() guards prevents stale dict/list from a prior
# interrupted run from silently causing AttributeError downstream.
if not isinstance(all_results, list) if 'all_results' in globals() else True:
    all_results = []
elif 'all_results' not in globals():
    all_results = []

if not isinstance(trained_models, dict) if 'trained_models' in globals() else True:
    trained_models = {}
elif 'trained_models' not in globals():
    trained_models = {}

if not isinstance(prediction_registry, dict) if 'prediction_registry' in globals() else True:
    prediction_registry = {}
elif 'prediction_registry' not in globals():
    prediction_registry = {}

# Cleaner equivalent of the above three blocks:
all_results         = all_results         if ('all_results'         in globals() and isinstance(all_results,         list)) else []
trained_models      = trained_models      if ('trained_models'      in globals() and isinstance(trained_models,      dict)) else {}
prediction_registry = prediction_registry if ('prediction_registry' in globals() and isinstance(prediction_registry, dict)) else {}

models_config = {'lstm': {'enabled': True, 'name': 'LSTM'}}

if models_config['lstm']['enabled']:
    print("\n" + "=" * 100)
    print("MODEL 7: LSTM")
    print("=" * 100)

    X_train_lstm_df = X_train.copy()
    X_test_lstm_df  = X_test.copy()

    for df_part in [X_train_lstm_df, X_test_lstm_df]:
        df_part['month_sin']      = np.sin(2 * np.pi * df_part['issue_d'].dt.month / 12)
        df_part['month_cos']      = np.cos(2 * np.pi * df_part['issue_d'].dt.month / 12)
        df_part['months_elapsed'] = (
            (df_part['issue_d'] - df_part['issue_d'].min()).dt.days / 30.44
        ).astype(np.float32)

    roll = y_train.shift(1).rolling(90, min_periods=10).mean().fillna(y_train.mean())
    X_train_lstm_df['rolling_default_3m'] = roll.values

    '''
    combined_y = pd.concat([y_train, y_test], ignore_index=True)
    roll_full  = combined_y.shift(1).rolling(90, min_periods=10).mean().fillna(y_train.mean())
    X_test_lstm_df['rolling_default_3m'] = roll_full.iloc[len(y_train):].values
    '''
    WINDOW = 90
    y_tr_tail    = y_train.values[-WINDOW:]
    y_combined_safe = np.concatenate([y_tr_tail, y_test.values])
    roll_safe = (
        pd.Series(y_combined_safe)
        .shift(1)
        .rolling(WINDOW, min_periods=10)
        .mean()
        .fillna(y_train.mean())
    )
    X_test_lstm_df['rolling_default_3m'] = roll_safe.values[WINDOW:]

    lstm_num_cols    = X_train_lstm_df.select_dtypes(include=[np.number]).columns.tolist()
    X_train_lstm_num = X_train_lstm_df[lstm_num_cols].values.astype(np.float32)
    X_test_lstm_num  = X_test_lstm_df[lstm_num_cols].values.astype(np.float32)

    scaler_lstm     = StandardScaler()
    X_train_lstm_sc = scaler_lstm.fit_transform(X_train_lstm_num)
    X_test_lstm_sc  = scaler_lstm.transform(X_test_lstm_num)

    time_steps   = 1
    X_train_lstm = X_train_lstm_sc.reshape(
        (X_train_lstm_sc.shape[0], time_steps, X_train_lstm_sc.shape[1]))
    X_test_lstm  = X_test_lstm_sc.reshape(
        (X_test_lstm_sc.shape[0],  time_steps, X_test_lstm_sc.shape[1]))

    neg_c, pos_c = np.bincount(y_train.astype(int))
    lstm_cw      = {0: 1.0, 1: neg_c / pos_c}
    print(f"  Class weight (minority) : {lstm_cw[1]:.2f}x")

    model_lstm = Sequential([
        LSTM(64, activation='tanh',
             input_shape=(time_steps, X_train_lstm.shape[2]),
             return_sequences=False),
        Dropout(0.2),
        Dense(32, activation='relu'),
        Dropout(0.2),
        Dense(16, activation='relu'),
        Dense(1,  activation='sigmoid')
    ])

    model_lstm.compile(
        optimizer=Adam(learning_rate=0.001),
        loss=focal_loss(gamma=2.0, alpha=0.75),
        metrics=['accuracy']
    )

    early_stop = EarlyStopping(
        monitor='val_loss', patience=5,
        restore_best_weights=True, verbose=0
    )

    print(f"Training with early stopping (patience=5)...\n")

    history_lstm = model_lstm.fit(
        X_train_lstm, y_train,
        batch_size=256,
        epochs=10,
        validation_data=(X_test_lstm, y_test),
        callbacks=[early_stop],
        class_weight=lstm_cw,
        verbose=0
    )

    print(f"✓ Training complete (Epochs: {len(history_lstm.history['loss'])})\n")

    y_train_pred_proba_lstm = batch_predict(model_lstm, X_train_lstm)
    y_test_pred_proba_lstm  = batch_predict(model_lstm, X_test_lstm)

    print(f"  Prob std — Train:{y_train_pred_proba_lstm.std():.4f} "
          f"| Test:{y_test_pred_proba_lstm.std():.4f}  (should be > 0.05)")

    y_train_pred_lstm = (y_train_pred_proba_lstm > 0.5).astype(int)
    y_test_pred_lstm  = (y_test_pred_proba_lstm  > 0.5).astype(int)

    trained_models['lstm'] = {
        'model'              : model_lstm,
        'y_train_pred'       : y_train_pred_lstm,
        'y_train_pred_proba' : y_train_pred_proba_lstm,
        'y_test_pred'        : y_test_pred_lstm,
        'y_test_pred_proba'  : y_test_pred_proba_lstm
    }

    print("=" * 100)
    print("LSTM - EVALUATION")
    print("=" * 100)
    result_lstm = evaluate_model(
        y_train, y_train_pred_lstm, y_train_pred_proba_lstm,
        y_test,  y_test_pred_lstm,  y_test_pred_proba_lstm,
        'LSTM'
    )
    all_results.append(result_lstm)           # ← now guaranteed to be a list
    prediction_registry['LSTM'] = y_test_pred_lstm

    print("\n✓ result_lstm      → appended to all_results")
    print("✓ model_lstm       → stored in trained_models['lstm']")
    print("✓ predictions      → stored in prediction_registry['LSTM']")


# ══════════════════════════════════════════════════════════════
# E1. ENSEMBLE BRIDGE
# ══════════════════════════════════════════════════════════════
print("\n--- E1: Ensemble Bridge ---")

arima_series = pd.Series(
    test_preds_arima,
    index=pd.PeriodIndex(arima_test_ts['month'].dt.to_period('M')),
    name='arima'
)

prophet_series = pd.Series(
    test_preds_prophet,
    index=pd.PeriodIndex(p_test_df['ds'].dt.to_period('M')),
    name='prophet'
)

lstm_test_months_idx = X_test['issue_d'].dt.to_period('M')
lstm_agg = (
    pd.DataFrame({
        'month'       : lstm_test_months_idx,
        'proba'       : y_test_pred_proba_lstm,
        'total_loans' : 1
    })
    .groupby('month')
    .agg(mean_proba=('proba', 'mean'), loan_count=('total_loans', 'count'))
    .reset_index()
)
lstm_agg['lstm_defaults'] = lstm_agg['mean_proba'] * lstm_agg['loan_count']

lstm_series = pd.Series(
    lstm_agg['lstm_defaults'].values,
    index=pd.PeriodIndex(lstm_agg['month']),
    name='lstm'
)

actual_series = pd.Series(
    arima_test_series,
    index=pd.PeriodIndex(arima_test_ts['month'].dt.to_period('M')),
    name='actual'
)

bridge = pd.DataFrame({
    'actual' : actual_series,
    'arima'  : arima_series,
    'prophet': prophet_series,
    'lstm'   : lstm_series,
}).dropna()

common_test_months = bridge.index.tolist()
actual_common      = bridge['actual'].values.astype(float)
arima_common       = bridge['arima'].values.astype(float)
prophet_common     = bridge['prophet'].values.astype(float)
lstm_raw           = bridge['lstm'].values.astype(float)

_lstm_mean    = lstm_raw.mean()
_actual_mean  = actual_common.mean()
_scale_factor = (_actual_mean / (_lstm_mean + 1e-9))
lstm_common   = lstm_raw * _scale_factor

print(f"  LSTM raw range     : {lstm_raw.min():.0f} – {lstm_raw.max():.0f}  (mean={_lstm_mean:.1f})")
print(f"  LSTM scale factor  : {_scale_factor:.4f}  (actual_mean/lstm_mean)")
print(f"  LSTM scaled range  : {lstm_common.min():.0f} – {lstm_common.max():.0f}  (mean={lstm_common.mean():.1f})")

print(f"  Common test months : {len(common_test_months)}")
print(f"  Period             : {common_test_months[0]} → {common_test_months[-1]}")
print(f"  Actual  range      : {actual_common.min():.0f} – {actual_common.max():.0f}")
print(f"  ARIMA   range      : {arima_common.min():.0f} – {arima_common.max():.0f}")
print(f"  Prophet range      : {prophet_common.min():.0f} – {prophet_common.max():.0f}")
print(f"  LSTM    range      : {lstm_common.min():.0f} – {lstm_common.max():.0f}")

if len(common_test_months) == 0:
    raise ValueError("No overlapping months found — check that all three model "
                     "test windows cover the same period.")

'''
# ══════════════════════════════════════════════════════════════
# LITERATURE-BASED ENSEMBLE METHODS
# ══════════════════════════════════════════════════════════════
from sklearn.linear_model import Ridge

print("=" * 80)
print("LITERATURE-BASED ENSEMBLE METHODS")
print("=" * 80)

n = len(actual_common)
x_dates = [pd.Period(m).to_timestamp() for m in common_test_months]

all_preds   = np.column_stack([arima_common, prophet_common, lstm_common])
model_names = ['ARIMA', 'Prophet', 'LSTM']

results = {}

# METHOD 1: SIMPLE AVERAGE
print("\n" + "=" * 70)
print("METHOD 1: SIMPLE AVERAGE  [Makridakis et al. 2020]")
print("=" * 70)
ens_simple = all_preds.mean(axis=1)
results['Simple Average'] = get_report(actual_common, ens_simple, "Simple Average (all 3 models)")

# METHOD 2: BATES-GRANGER
print("\n" + "=" * 70)
print("METHOD 2: BATES-GRANGER INVERSE-MSE WEIGHTS  [Bates & Granger 1969]")
print("=" * 70)
mse_each   = np.mean((all_preds - actual_common.reshape(-1, 1)) ** 2, axis=0)
inv_mse    = 1.0 / (mse_each + 1e-9)
bg_weights = inv_mse / inv_mse.sum()
ens_bg = (all_preds * bg_weights).sum(axis=1)
print(f"\n  Bates-Granger weights:")
for name, mse, w in zip(model_names, mse_each, bg_weights):
    print(f"    {name:<10} MSE={mse:>10.2f}  weight={w:.4f}")
results['Bates-Granger'] = get_report(actual_common, ens_bg, "Bates-Granger Inverse-MSE")

# METHOD 3: VARIANCE-BASED
print("\n" + "=" * 70)
print("METHOD 3: VARIANCE-BASED WEIGHTING  [Timmermann 2006]")
print("=" * 70)
errors      = all_preds - actual_common.reshape(-1, 1)
var_each    = np.var(errors, axis=0, ddof=1)
inv_var     = 1.0 / (var_each + 1e-9)
var_weights = inv_var / inv_var.sum()
ens_var = (all_preds * var_weights).sum(axis=1)
print(f"\n  Variance-based weights:")
for name, var, w in zip(model_names, var_each, var_weights):
    print(f"    {name:<10} Var={var:>10.2f}  weight={w:.4f}")
results['Variance-Weighted'] = get_report(actual_common, ens_var, "Variance-Based Weighting")

# METHOD 4: TRIMMED MEAN
print("\n" + "=" * 70)
print("METHOD 4: TRIMMED MEAN  [Genre et al. 2013]")
print("=" * 70)
if all_preds.shape[1] >= 3:
    sorted_preds = np.sort(all_preds, axis=1)
    ens_trimmed  = sorted_preds[:, 1:-1].mean(axis=1)
    results['Trimmed Mean'] = get_report(actual_common, ens_trimmed, "Trimmed Mean (drop min/max per step)")
else:
    ens_trimmed = ens_simple
    print("  ⚠ Need ≥3 models for trimmed mean — using simple average")

# METHOD 5: MEDIAN
print("\n" + "=" * 70)
print("METHOD 5: MEDIAN ENSEMBLE  [Timmermann 2006]")
print("=" * 70)
ens_median = np.median(all_preds, axis=1)
results['Median'] = get_report(actual_common, ens_median, "Median Ensemble")

# METHOD 6: ZHANG HYBRID
print("\n" + "=" * 70)
print("METHOD 6: ZHANG HYBRID (ARIMA + LSTM on residuals)  [Zhang 2003]")
print("=" * 70)
arima_residuals = actual_common - arima_common
lstm_centered   = lstm_common - lstm_common.mean()
res_std         = arima_residuals.std() + 1e-9
lstm_scaled_res = lstm_centered * (res_std / (lstm_centered.std() + 1e-9))
ens_zhang = arima_common + lstm_scaled_res
results['Zhang Hybrid'] = get_report(actual_common, ens_zhang, "Zhang Hybrid (ARIMA + LSTM residual)")
print(f"\n  ARIMA residuals range : {arima_residuals.min():.1f} to {arima_residuals.max():.1f}")
print(f"  LSTM residual proxy   : {lstm_scaled_res.min():.1f} to {lstm_scaled_res.max():.1f}")

# METHOD 7: STACKING
print("\n" + "=" * 70)
print("METHOD 7: STACKING / SUPER LEARNER  [Wolpert 1992]")
print("=" * 70)
if n >= 5:
    stack_preds = np.zeros(n)
    for i in range(n):
        train_idx       = [j for j in range(n) if j != i]
        meta            = Ridge(alpha=1.0, fit_intercept=True)
        meta.fit(all_preds[train_idx], actual_common[train_idx])
        stack_preds[i]  = meta.predict(all_preds[[i]])[0]
    results['Stacking (LOO)'] = get_report(actual_common, stack_preds, "Stacking / Super Learner (LOO)")
    meta_final = Ridge(alpha=1.0, fit_intercept=True)
    meta_final.fit(all_preds, actual_common)
    print(f"\n  Meta-learner coefficients (Ridge α=1):")
    for name, coef in zip(model_names, meta_final.coef_):
        print(f"    {name:<10}  {coef:.4f}")
    print(f"    Intercept   {meta_final.intercept_:.4f}")
else:
    print(f"  ⚠ Only {n} test points — stacking skipped (need ≥5)")

# METHOD 8: RANK-BASED
print("\n" + "=" * 70)
print("METHOD 8: RANK-BASED WEIGHTING  [Genre et al. 2013]")
print("=" * 70)
mae_each     = np.mean(np.abs(all_preds - actual_common.reshape(-1, 1)), axis=0)
ranks        = mae_each.argsort().argsort() + 1
inv_rank     = 1.0 / ranks
rank_weights = inv_rank / inv_rank.sum()
ens_rank = (all_preds * rank_weights).sum(axis=1)
print(f"\n  Rank-based weights:")
for name, mae, rank, w in zip(model_names, mae_each, ranks, rank_weights):
    print(f"    {name:<10} MAE={mae:>8.2f}  rank={rank}  weight={w:.4f}")
results['Rank-Weighted'] = get_report(actual_common, ens_rank, "Rank-Based Weighting")

# ── COMPARISON TABLE ──────────────────────────────────────────
print("\n" + "=" * 80)
print("FINAL COMPARISON — ALL LITERATURE METHODS")
print("=" * 80)

results_all = {
    'ARIMA (solo)':   get_report(actual_common, arima_common,   "ARIMA solo"),
    'Prophet (solo)': get_report(actual_common, prophet_common, "Prophet solo"),
    'LSTM (cal)':     get_report(actual_common, lstm_common,    "LSTM calibrated solo"),
    **results
}

comp_df = pd.DataFrame([{'Method': k, **v} for k, v in results_all.items()])

print(f"\n  {'Method':<28} {'R²':>8} {'MAE':>8} {'RMSE':>8} {'sMAPE%':>8} {'MAPE%':>8}")
print(f"  {'-'*72}")
for _, row in comp_df.iterrows():
    marker = ' ←best' if row['mae'] == comp_df['mae'].min() else ''
    print(f"  {row['Method']:<28}"
          f" {row['r2']:>8.4f}"
          f" {row['mae']:>8.2f}"
          f" {row['rmse']:>8.2f}"
          f" {row['smape']:>8.4f}"
          f" {row['mape']:>8.4f}{marker}")

best_r2  = comp_df.loc[comp_df['r2'].idxmax(),  'Method']
best_mae = comp_df.loc[comp_df['mae'].idxmin(), 'Method']
print(f"\n  Best R²  : {best_r2}")
print(f"  Best MAE : {best_mae}")

# ── VISUALIZATION ─────────────────────────────────────────────
fig, axes = plt.subplots(3, 1, figsize=(16, 20))
fig.suptitle(
    'Literature-Based Ensemble Methods — LendingClub Monthly Defaults\n'
    f'{str(common_test_months[0])} → {str(common_test_months[-1])}',
    fontsize=14, fontweight='bold', y=0.99
)

ax = axes[0]
ax.plot(x_dates, actual_common, color='steelblue',  lw=2.5, marker='o', ms=6, label='Actual')
ax.plot(x_dates, ens_bg,        color='green',      lw=1.5, ls='--', marker='s', ms=4,
        label=f'Bates-Granger  MAE={results["Bates-Granger"]["mae"]:.1f}')
ax.plot(x_dates, ens_var,       color='purple',     lw=1.5, ls='--', marker='^', ms=4,
        label=f'Variance-Wtd   MAE={results["Variance-Weighted"]["mae"]:.1f}')
ax.plot(x_dates, ens_trimmed,   color='crimson',    lw=1.5, ls='-.', marker='D', ms=4,
        label=f'Trimmed Mean   MAE={results["Trimmed Mean"]["mae"]:.1f}')
ax.plot(x_dates, ens_median,    color='darkorange', lw=1.5, ls=':',  marker='v', ms=4,
        label=f'Median         MAE={results["Median"]["mae"]:.1f}')
ax.set_title('Classical Weighting Methods  [Bates-Granger 1969, Timmermann 2006, Genre 2013]',
             fontsize=11, fontweight='bold')
ax.set_ylabel('Total Defaults'); ax.grid(True, alpha=0.3); ax.legend(fontsize=9)
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
plt.setp(ax.xaxis.get_majorticklabels(), rotation=45)

ax = axes[1]
ax.plot(x_dates, actual_common, color='steelblue', lw=2.5, marker='o', ms=6, label='Actual')
ax.plot(x_dates, ens_zhang,     color='green',     lw=2, ls='--', marker='s', ms=5,
        label=f'Zhang Hybrid   MAE={results["Zhang Hybrid"]["mae"]:.1f}')
if 'Stacking (LOO)' in results:
    ax.plot(x_dates, stack_preds, color='crimson', lw=2, ls='-.', marker='D', ms=5,
            label=f'Stacking LOO   MAE={results["Stacking (LOO)"]["mae"]:.1f}')
ax.plot(x_dates, ens_rank,      color='darkorange', lw=2, ls=':',  marker='^', ms=5,
        label=f'Rank-Weighted  MAE={results["Rank-Weighted"]["mae"]:.1f}')
ax.set_title('Hybrid / Meta-Learning Methods  [Zhang 2003, Wolpert 1992, Genre 2013]',
             fontsize=11, fontweight='bold')
ax.set_ylabel('Total Defaults'); ax.grid(True, alpha=0.3); ax.legend(fontsize=9)
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
plt.setp(ax.xaxis.get_majorticklabels(), rotation=45)

ensemble_only = comp_df[~comp_df['Method'].isin(['ARIMA (solo)', 'Prophet (solo)', 'LSTM (cal)'])]
best_ens_name = ensemble_only.loc[ensemble_only['mae'].idxmin(), 'Method']

ens_map = {
    'Simple Average'   : ens_simple,
    'Bates-Granger'    : ens_bg,
    'Variance-Weighted': ens_var,
    'Trimmed Mean'     : ens_trimmed,
    'Median'           : ens_median,
    'Zhang Hybrid'     : ens_zhang,
    'Rank-Weighted'    : ens_rank,
}
if 'Stacking (LOO)' in results:
    ens_map['Stacking (LOO)'] = stack_preds

best_ens_pred = ens_map.get(best_ens_name, ens_bg)

ax = axes[2]
ax.plot(x_dates, actual_common,  color='steelblue', lw=2.5, marker='o', ms=6, label='Actual')
ax.plot(x_dates, arima_common,   color='green',     lw=1.5, ls='--', marker='s', ms=4,
        label=f'ARIMA          MAE={results_all["ARIMA (solo)"]["mae"]:.1f}')
ax.plot(x_dates, prophet_common, color='orange',    lw=1.5, ls='--', marker='^', ms=4,
        label=f'Prophet        MAE={results_all["Prophet (solo)"]["mae"]:.1f}')
ax.plot(x_dates, best_ens_pred,  color='crimson',   lw=2.5, ls='-',  marker='D', ms=6,
        label=f'Best: {best_ens_name}  MAE={results_all[best_ens_name]["mae"]:.1f}')
ax.set_title(f'Best Ensemble vs Solo Models — Winner: {best_ens_name}',
             fontsize=11, fontweight='bold')
ax.set_ylabel('Total Defaults'); ax.grid(True, alpha=0.3); ax.legend(fontsize=9)
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
plt.setp(ax.xaxis.get_majorticklabels(), rotation=45)

plt.tight_layout(rect=[0, 0, 1, 0.98])
plt.savefig('/kaggle/working/ensemble_literature.png', dpi=120, bbox_inches='tight')
print("\n  ✓ Saved: ensemble_literature.png")
plt.show()

comp_df.to_csv('/kaggle/working/ensemble_literature_comparison.csv', index=False)

pd.DataFrame({
    'month'         : [str(m) for m in common_test_months],
    'actual'        : actual_common,
    'arima'         : arima_common,
    'prophet'       : prophet_common,
    'lstm_cal'      : lstm_common,
    'simple_avg'    : ens_simple,
    'bates_granger' : ens_bg,
    'variance_wtd'  : ens_var,
    'trimmed_mean'  : ens_trimmed,
    'median'        : ens_median,
    'zhang_hybrid'  : ens_zhang,
    'rank_weighted' : ens_rank,
    **({'stacking_loo': stack_preds} if 'Stacking (LOO)' in results else {})
}).to_csv('/kaggle/working/ensemble_literature_predictions.csv', index=False)

print("  ✓ Saved: ensemble_literature_comparison.csv")
print("  ✓ Saved: ensemble_literature_predictions.csv")
print("\n✓ Literature-based ensemble section complete!")


# ══════════════════════════════════════════════════════════════
# POST-2020 RESEARCH ENSEMBLE METHODS
# ══════════════════════════════════════════════════════════════
from scipy.stats import pearsonr

print("\n" + "=" * 80)
print("POST-2020 RESEARCH ENSEMBLE METHODS")
print("=" * 80)
print("""
  [A] Bertsimas & Boussioux (2023) — Adaptive Robust Optimization (ARO)
      DOI: https://doi.org/10.48550/arXiv.2304.04308
  [B] Patel & Wikner (2023) — Attention-Based Ensemble Pooling
      DOI: https://doi.org/10.48550/arXiv.2310.16231
  [C] Montero-Manso et al. (2020) — FFORMA (Feature-Based Model Averaging)
      DOI: https://doi.org/10.1016/j.ijforecast.2019.02.011
  [D] Saadallah & Jakobs (2023) — Online Hedge / Multiplicative Weights
      DOI: https://doi.org/10.1007/978-3-031-43424-2_10
""")

n        = len(actual_common)
x_dates  = [pd.Period(m).to_timestamp() for m in common_test_months]
P        = np.column_stack([arima_common, prophet_common, lstm_common])
mnames   = ['ARIMA', 'Prophet', 'LSTM']
post_results = {}

# METHOD A: ARO
print("\n" + "=" * 70)
print("METHOD A: ADAPTIVE ROBUST OPTIMIZATION  [Bertsimas & Boussioux 2023]")
print("  DOI: https://doi.org/10.48550/arXiv.2304.04308")
print("=" * 70)

aro_preds   = np.zeros(n)
cum_sq_err  = np.ones(3) * 1e-6

for t in range(n):
    inv_regret = 1.0 / (cum_sq_err + 1e-9)
    w_aro      = inv_regret / inv_regret.sum()
    aro_preds[t] = (P[t] * w_aro).sum()
    err_sq          = (P[t] - actual_common[t]) ** 2
    cum_sq_err     += err_sq

post_results["ARO (Bertsimas 2023)"] = get_report(
    actual_common, aro_preds, "ARO — Adaptive Robust Optimization"
)
print(f"  Final adaptive weights: "
      + "  ".join(f"{m}={w:.4f}" for m, w in zip(mnames, w_aro)))

# METHOD B: ATTENTION-BASED
print("\n" + "=" * 70)
print("METHOD B: ATTENTION-BASED ENSEMBLE POOLING  [Patel & Wikner 2023]")
print("  DOI: https://doi.org/10.48550/arXiv.2310.16231")
print("=" * 70)

ATTN_WINDOW = max(3, n // 3)
attn_preds  = np.zeros(n)
attn_weights_log = []

for t in range(n):
    if t == 0:
        w_attn = np.ones(3) / 3
    else:
        win_start = max(0, t - ATTN_WINDOW)
        win_actual = actual_common[win_start:t]
        win_preds  = P[win_start:t]
        rolling_mae = np.mean(np.abs(win_preds - win_actual.reshape(-1, 1)), axis=0)
        logits = -rolling_mae / (rolling_mae.std() + 1e-9)
        exp_l  = np.exp(logits - logits.max())
        w_attn = exp_l / exp_l.sum()

    attn_preds[t] = (P[t] * w_attn).sum()
    attn_weights_log.append(w_attn.copy())

attn_weights_log = np.array(attn_weights_log)
post_results["Attention Pool (Patel 2023)"] = get_report(
    actual_common, attn_preds, "Attention-Based Ensemble Pooling"
)
print(f"  Avg attention weights across steps:")
for m, w in zip(mnames, attn_weights_log.mean(axis=0)):
    print(f"    {m:<10} {w:.4f}")

# METHOD C: FFORMA
print("\n" + "=" * 70)
print("METHOD C: FFORMA — FEATURE-BASED MODEL AVERAGING")
print("  [Montero-Manso et al. 2020]")
print("  DOI: https://doi.org/10.1016/j.ijforecast.2019.02.011")
print("=" * 70)

def ts_features(series):
    s   = np.array(series, dtype=float)
    n_s = len(s)
    if n_s < 4:
        return np.zeros(5)
    t_idx = np.arange(n_s)
    slope, intercept = np.polyfit(t_idx, s, 1)
    fitted    = slope * t_idx + intercept
    ss_res    = np.sum((s - fitted) ** 2)
    ss_tot    = np.sum((s - s.mean()) ** 2) + 1e-9
    trend_str = max(0.0, 1 - ss_res / ss_tot)
    if n_s > 2:
        acf1, _ = pearsonr(s[:-1], s[1:])
    else:
        acf1 = 0.0
    vol_ratio = np.std(np.diff(s)) / (np.std(s) + 1e-9)
    fft_mag   = np.abs(np.fft.rfft(s - s.mean()))
    sp_ent    = -(fft_mag / (fft_mag.sum() + 1e-9) *
                  np.log(fft_mag / (fft_mag.sum() + 1e-9) + 1e-12)).sum()
    sp_ent   /= np.log(len(fft_mag) + 1)
    skew = float(pd.Series(s).skew()) if n_s >= 3 else 0.0
    return np.array([trend_str, acf1, vol_ratio, sp_ent, skew])

fforma_preds = np.zeros(n)
train_hist   = list(arima_test_series[:len(arima_test_series) - n])

if len(train_hist) < 4:
    train_hist = list(ts_model["total_defaults_smooth"].values[:-n])

feature_rows  = []
target_errors = []

for t in range(n):
    context = np.array(train_hist + list(actual_common[:t]) if t > 0 else train_hist)
    feat    = ts_features(context[-24:] if len(context) >= 24 else context)
    feature_rows.append(feat)
    target_errors.append(np.abs(P[t] - actual_common[t]))

X_fforma = np.array(feature_rows)
E_fforma = np.array(target_errors)

from sklearn.linear_model import Ridge as _Ridge
from sklearn.preprocessing import StandardScaler as _SC

sc_fforma = _SC()
X_fforma_sc = sc_fforma.fit_transform(X_fforma)

for t in range(n):
    loo_idx  = [j for j in range(n) if j != t]
    if len(loo_idx) == 0:
        w_ff = np.ones(3) / 3
    else:
        meta_ff = _Ridge(alpha=1.0)
        meta_ff.fit(X_fforma_sc[loo_idx], E_fforma[loo_idx])
        pred_err = meta_ff.predict(X_fforma_sc[[t]])[0]
        pred_err = np.clip(pred_err, 1e-9, None)
        inv_err  = 1.0 / pred_err
        w_ff     = inv_err / inv_err.sum()
    fforma_preds[t] = (P[t] * w_ff).sum()

post_results["FFORMA (Montero-Manso 2020)"] = get_report(
    actual_common, fforma_preds, "FFORMA Feature-Based Model Averaging"
)

# METHOD D: HEDGE / MULTIPLICATIVE WEIGHTS
print("\n" + "=" * 70)
print("METHOD D: ONLINE HEDGE / MULTIPLICATIVE WEIGHTS")
print("  [Saadallah & Jakobs, ECML PKDD 2023]")
print("  DOI: https://doi.org/10.1007/978-3-031-43424-2_10")
print("=" * 70)

ETA = 0.5
hedge_w     = np.ones(3) / 3.0
hedge_preds = np.zeros(n)
hedge_w_log = []

for t in range(n):
    hedge_preds[t] = (P[t] * hedge_w).sum()
    hedge_w_log.append(hedge_w.copy())
    losses  = np.abs(P[t] - actual_common[t])
    losses /= (losses.max() + 1e-9)
    hedge_w  = hedge_w * np.exp(-ETA * losses)
    hedge_w /= hedge_w.sum()

hedge_w_log = np.array(hedge_w_log)
post_results["Hedge/MWU (Saadallah 2023)"] = get_report(
    actual_common, hedge_preds, "Online Hedge / Multiplicative Weights"
)
print(f"  η (learning rate) : {ETA}")
print(f"  Final weights after {n} steps:")
for m, w in zip(mnames, hedge_w):
    print(f"    {m:<10} {w:.4f}")

# ── FULL COMPARISON TABLE ─────────────────────────────────────
print("\n" + "=" * 80)
print("FULL COMPARISON — POST-2020 + LITERATURE + SOLO MODELS")
print("=" * 80)

all_methods = {
    "ARIMA (solo)"         : get_report(actual_common, arima_common,   "ARIMA solo"),
    "Prophet (solo)"       : get_report(actual_common, prophet_common, "Prophet solo"),
    "LSTM (solo)"          : get_report(actual_common, lstm_common,    "LSTM solo"),
    "Simple Average"       : get_report(actual_common, ens_simple,  "Simple Average"),
    "Bates-Granger"        : get_report(actual_common, ens_bg,      "Bates-Granger"),
    "Variance-Weighted"    : get_report(actual_common, ens_var,     "Variance-Weighted"),
    "Trimmed Mean"         : get_report(actual_common, ens_trimmed, "Trimmed Mean"),
    "Median"               : get_report(actual_common, ens_median,  "Median"),
    "Zhang Hybrid"         : get_report(actual_common, ens_zhang,   "Zhang Hybrid"),
    "Rank-Weighted"        : get_report(actual_common, ens_rank,    "Rank-Weighted"),
    **({"Stacking (LOO)": get_report(actual_common, stack_preds, "Stacking LOO")}
       if "Stacking (LOO)" in results else {}),
    **{k: get_report(actual_common, v_arr, k)
       for k, v_arr in [
           ("ARO (Bertsimas 2023)",       aro_preds),
           ("Attention Pool (Patel 2023)", attn_preds),
           ("FFORMA (Montero-Manso 2020)", fforma_preds),
           ("Hedge/MWU (Saadallah 2023)",  hedge_preds),
       ]},
}

full_df = pd.DataFrame([{"Method": k, **v} for k, v in all_methods.items()])
best_mae_val = full_df["mae"].min()

W_m, W_v = 30, 10
print(f"\n  {'Method':<{W_m}} {'R²':>{W_v}} {'MAE(norm)':>{W_v}} {'RMSE(norm)':>{W_v}} {'sMAPE%':>{W_v}} {'MAPE%':>{W_v}}")
print(f"  {'-'* (W_m + W_v * 5 + 4)}")
for _, r in full_df.iterrows():
    tag = " ◄ BEST" if abs(r["mae"] - best_mae_val) < 1e-9 else ""
    print(f"  {r['Method']:<{W_m}}"
          f" {r['r2']:>{W_v}.4f}"
          f" {r['mae']:>{W_v}.4f}"
          f" {r['rmse']:>{W_v}.4f}"
          f" {r['smape']:>{W_v}.4f}"
          f" {r['mape']:>{W_v}.4f}{tag}")

best_r2_name  = full_df.loc[full_df["r2"].idxmax(),  "Method"]
best_mae_name = full_df.loc[full_df["mae"].idxmin(), "Method"]
print(f"\n  Best R²       : {best_r2_name}")
print(f"  Best MAE(norm): {best_mae_name}")

# ── VISUALIZATION ─────────────────────────────────────────────
fig, axes = plt.subplots(3, 1, figsize=(16, 20))
fig.suptitle(
    "Post-2020 Research Ensemble Methods — LendingClub Monthly Defaults\n"
    f"{str(common_test_months[0])} → {str(common_test_months[-1])}",
    fontsize=14, fontweight="bold", y=0.99
)

ax = axes[0]
ax.plot(x_dates, actual_common, color="steelblue", lw=2.5, marker="o", ms=6, label="Actual")
ax.plot(x_dates, aro_preds,   color="green",     lw=2, ls="--", marker="s", ms=5,
        label=f"ARO (Bertsimas 2023)    MAE={post_results['ARO (Bertsimas 2023)']['mae']:.4f}")
ax.plot(x_dates, attn_preds,  color="purple",    lw=2, ls="-.", marker="^", ms=5,
        label=f"Attention Pool (Patel 2023) MAE={post_results['Attention Pool (Patel 2023)']['mae']:.4f}")
ax.set_title("Adaptive Robust Optimization vs Attention-Based Pooling\n"
             "[Bertsimas & Boussioux 2023 | Patel & Wikner 2023]",
             fontsize=11, fontweight="bold")
ax.set_ylabel("Total Defaults"); ax.grid(True, alpha=0.3); ax.legend(fontsize=9)
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m"))
plt.setp(ax.xaxis.get_majorticklabels(), rotation=45)

ax = axes[1]
ax.plot(x_dates, actual_common, color="steelblue", lw=2.5, marker="o", ms=6, label="Actual")
ax.plot(x_dates, fforma_preds,  color="darkorange", lw=2, ls="--", marker="D", ms=5,
        label=f"FFORMA (Montero-Manso 2020) MAE={post_results['FFORMA (Montero-Manso 2020)']['mae']:.4f}")
ax.plot(x_dates, hedge_preds,   color="crimson",    lw=2, ls="-.", marker="v", ms=5,
        label=f"Hedge/MWU (Saadallah 2023) MAE={post_results['Hedge/MWU (Saadallah 2023)']['mae']:.4f}")
ax.set_title("Feature-Based Averaging vs Online Multiplicative Weights\n"
             "[Montero-Manso et al. 2020 | Saadallah & Jakobs 2023]",
             fontsize=11, fontweight="bold")
ax.set_ylabel("Total Defaults"); ax.grid(True, alpha=0.3); ax.legend(fontsize=9)
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m"))
plt.setp(ax.xaxis.get_majorticklabels(), rotation=45)

post_map = {
    "ARO (Bertsimas 2023)"        : aro_preds,
    "Attention Pool (Patel 2023)" : attn_preds,
    "FFORMA (Montero-Manso 2020)" : fforma_preds,
    "Hedge/MWU (Saadallah 2023)"  : hedge_preds,
}
post_only_df = full_df[full_df["Method"].isin(post_map.keys())]
best_post_name = post_only_df.loc[post_only_df["mae"].idxmin(), "Method"]
best_post_pred = post_map[best_post_name]

lit_map = {
    "Simple Average": ens_simple, "Bates-Granger": ens_bg,
    "Variance-Weighted": ens_var, "Trimmed Mean": ens_trimmed,
    "Median": ens_median, "Zhang Hybrid": ens_zhang, "Rank-Weighted": ens_rank,
}
if "Stacking (LOO)" in results:
    lit_map["Stacking (LOO)"] = stack_preds
lit_only_df = full_df[full_df["Method"].isin(lit_map.keys())]
best_lit_name = lit_only_df.loc[lit_only_df["mae"].idxmin(), "Method"]
best_lit_pred = lit_map[best_lit_name]

ax = axes[2]
ax.plot(x_dates, actual_common,  color="steelblue",  lw=2.5, marker="o", ms=6, label="Actual")
ax.plot(x_dates, arima_common,   color="gray",       lw=1.5, ls=":", marker="s", ms=4,
        label=f"ARIMA (solo)       MAE={all_methods['ARIMA (solo)']['mae']:.4f}")
ax.plot(x_dates, prophet_common, color="sandybrown", lw=1.5, ls=":", marker="^", ms=4,
        label=f"Prophet (solo)     MAE={all_methods['Prophet (solo)']['mae']:.4f}")
ax.plot(x_dates, best_lit_pred,  color="green",      lw=2,   ls="--", marker="D", ms=5,
        label=f"Best Lit: {best_lit_name}  MAE={all_methods[best_lit_name]['mae']:.4f}")
ax.plot(x_dates, best_post_pred, color="crimson",    lw=2.5, ls="-",  marker="*", ms=8,
        label=f"Best Post-2020: {best_post_name}  MAE={all_methods[best_post_name]['mae']:.4f}")
ax.set_title(f"Best Post-2020 ({best_post_name}) vs Best Literature ({best_lit_name}) vs Solo Models",
             fontsize=11, fontweight="bold")
ax.set_ylabel("Total Defaults"); ax.grid(True, alpha=0.3); ax.legend(fontsize=9)
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m"))
plt.setp(ax.xaxis.get_majorticklabels(), rotation=45)

plt.tight_layout(rect=[0, 0, 1, 0.98])
plt.savefig("/kaggle/working/ensemble_post2020.png", dpi=120, bbox_inches="tight")
print("\n  ✓ Saved: ensemble_post2020.png")
plt.show()

fig2, axes2 = plt.subplots(2, 1, figsize=(14, 10))
fig2.suptitle("Dynamic Weight Evolution — Post-2020 Ensemble Methods",
              fontsize=13, fontweight="bold")

ax = axes2[0]
for i, m in enumerate(mnames):
    ax.plot(x_dates, attn_weights_log[:, i], marker="o", ms=5, lw=1.5, label=m)
ax.set_title("Attention-Based Pooling Weights per Step  [Patel & Wikner 2023]",
             fontsize=11, fontweight="bold")
ax.set_ylabel("Weight"); ax.set_ylim(0, 1); ax.grid(True, alpha=0.3); ax.legend(fontsize=9)
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m"))
plt.setp(ax.xaxis.get_majorticklabels(), rotation=45)

ax = axes2[1]
for i, m in enumerate(mnames):
    ax.plot(x_dates, hedge_w_log[:, i], marker="s", ms=5, lw=1.5, label=m)
ax.set_title(f"Hedge/MWU Weights per Step (η={ETA})  [Saadallah & Jakobs 2023]",
             fontsize=11, fontweight="bold")
ax.set_ylabel("Weight"); ax.set_ylim(0, 1); ax.grid(True, alpha=0.3); ax.legend(fontsize=9)
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m"))
plt.setp(ax.xaxis.get_majorticklabels(), rotation=45)

plt.tight_layout()
plt.savefig("/kaggle/working/ensemble_post2020_weights.png", dpi=120, bbox_inches="tight")
print("  ✓ Saved: ensemble_post2020_weights.png")
plt.show()

full_df.to_csv("/kaggle/working/ensemble_full_comparison.csv", index=False)

pd.DataFrame({
    "month"              : [str(m) for m in common_test_months],
    "actual"             : actual_common,
    "arima"              : arima_common,
    "prophet"            : prophet_common,
    "lstm_cal"           : lstm_common,
    "simple_avg"         : ens_simple,
    "bates_granger"      : ens_bg,
    "variance_wtd"       : ens_var,
    "trimmed_mean"       : ens_trimmed,
    "median"             : ens_median,
    "zhang_hybrid"       : ens_zhang,
    "rank_weighted"      : ens_rank,
    **({ "stacking_loo" : stack_preds } if "Stacking (LOO)" in results else {}),
    "aro_bertsimas2023"         : aro_preds,
    "attention_patel2023"       : attn_preds,
    "fforma_montero2020"        : fforma_preds,
    "hedge_saadallah2023"       : hedge_preds,
}).to_csv("/kaggle/working/ensemble_all_predictions.csv", index=False)

print("  ✓ Saved: ensemble_full_comparison.csv")
print("  ✓ Saved: ensemble_all_predictions.csv")
print("\n✓ Post-2020 ensemble section complete!")

'''
# ══════════════════════════════════════════════════════════════
# LINEAR + NONLINEAR HYBRID ENSEMBLE METHODS
# ══════════════════════════════════════════════════════════════
from sklearn.neighbors import KNeighborsRegressor
from sklearn.preprocessing import StandardScaler as _SC2
from sklearn.linear_model import Ridge as _Ridge2

print("\n" + "=" * 80)
print("LINEAR + NONLINEAR HYBRID ENSEMBLE METHODS  (Post-2020)")
print("=" * 80)
print("""
  [LN-A] Santos Júnior et al. (2023) — ERF: Ensemble Residual Forecasting
         DOI: https://doi.org/10.1016/j.ins.2023.119600
  [LN-B] Cruz Medina & de Oliveira (2024) — KNN-Local Linear/Nonlinear Mixing
         DOI: https://doi.org/10.1016/j.neucom.2024.129235
  [LN-C] Li et al. (2025) — Multi-Scale Adaptive Fusion (ARIMA + LSTM)
         DOI: https://doi.org/10.3390/e27070695
  [LN-D] Yilmaz & Sofuoglu (2025) — Prophet-LSTM Residual Hybrid
         DOI: https://doi.org/10.1038/s41598-025-27510-y
""")

ln_results = {}
n       = len(actual_common)
x_dates = [pd.Period(m).to_timestamp() for m in common_test_months]

lin_pred  = arima_common
lin2_pred = prophet_common
nl_pred   = lstm_common

# METHOD LN-A: ERF
print("\n" + "=" * 70)
print("METHOD LN-A: ERF — Ensemble Residual Forecasting")
print("  [Santos Júnior et al., Information Sciences, 2023]")
print("  DOI: https://doi.org/10.1016/j.ins.2023.119600")
print("=" * 70)

arima_res = actual_common - lin_pred

lstm_cen    = nl_pred - nl_pred.mean()
res_scale   = arima_res.std() / (lstm_cen.std() + 1e-9)
nl_res1     = lstm_cen * res_scale

nl_res2 = prophet_common - lin_pred

def nadaraya_watson(x_train, y_train, x_test, h=1.5):
    preds = []
    for xt in x_test:
        w = np.exp(-0.5 * ((x_train - xt) / h) ** 2)
        w_sum = w.sum() + 1e-9
        preds.append((w * y_train).sum() / w_sum)
    return np.array(preds)

t_idx    = np.arange(n, dtype=float)
nl_res3  = nadaraya_watson(t_idx, arima_res, t_idx, h=2.0)

residual_ensemble = (nl_res1 + nl_res2 + nl_res3) / 3.0
erf_pred          = lin_pred + residual_ensemble

ln_results["ERF (Santos Jr. 2023)"] = get_report(
    actual_common, erf_pred,
    "ERF: ARIMA + Ensemble(Residual Corrections)"
)
print(f"\n  Residual ensemble component stats:")
print(f"    LSTM-scaled : mean={nl_res1.mean():.2f}  std={nl_res1.std():.2f}")
print(f"    Prophet-gap : mean={nl_res2.mean():.2f}  std={nl_res2.std():.2f}")
print(f"    Kernel-reg  : mean={nl_res3.mean():.2f}  std={nl_res3.std():.2f}")
'''

# METHOD LN-B: KNN-LOCAL
print("\n" + "=" * 70)
print("METHOD LN-B: KNN-LOCAL Linear/Nonlinear Mixing")
print("  [Cruz Medina & de Oliveira, Neurocomputing, 2024]")
print("  DOI: https://doi.org/10.1016/j.neucom.2024.129235")
print("=" * 70)

K_NEIGHBORS = max(3, n // 3)
knn_preds   = np.zeros(n)
knn_alpha   = np.zeros(n)

Xq = np.column_stack([lin_pred, nl_pred])
sc_knn = _SC2()
Xq_sc  = sc_knn.fit_transform(Xq)

for t in range(n):
    loo_idx = [j for j in range(n) if j != t]
    if len(loo_idx) < K_NEIGHBORS:
        alpha_t = 0.5
    else:
        dists   = np.linalg.norm(Xq_sc[loo_idx] - Xq_sc[t], axis=1)
        nn_idx  = np.array(loo_idx)[np.argsort(dists)[:K_NEIGHBORS]]
        X_loc = Xq[nn_idx]
        y_loc = actual_common[nn_idx]
        lr    = _Ridge2(alpha=1.0, fit_intercept=True)
        lr.fit(X_loc, y_loc)
        c     = lr.coef_
        alpha_t = abs(c[0]) / (abs(c[0]) + abs(c[1]) + 1e-9)

    knn_alpha[t]  = alpha_t
    knn_preds[t]  = alpha_t * lin_pred[t] + (1 - alpha_t) * nl_pred[t]

ln_results["KNN-Local Hybrid (Cruz Medina 2024)"] = get_report(
    actual_common, knn_preds, "KNN-Local Linear/Nonlinear Mixing"
)
print(f"\n  Per-step local mixing weight α (linear share):")
for t, (m, a) in enumerate(zip(common_test_months, knn_alpha)):
    print(f"    {str(m)}  α_lin={a:.3f}  α_nl={1-a:.3f}")
print(f"  Mean α_lin={knn_alpha.mean():.3f}  Mean α_nl={1-knn_alpha.mean():.3f}")

# METHOD LN-C: MULTI-SCALE ADAPTIVE
print("\n" + "=" * 70)
print("METHOD LN-C: MULTI-SCALE ADAPTIVE FUSION  [Li et al., Entropy 2025]")
print("  DOI: https://doi.org/10.3390/e27070695")
print("=" * 70)

arima_res_c = actual_common - lin_pred

lstm_cen_c    = nl_pred - nl_pred.mean()
scale_c       = arima_res_c.std() / (lstm_cen_c.std() + 1e-9)
lstm_res_corr = lstm_cen_c * scale_c

stage1 = lin_pred + lstm_res_corr

mse_arima_loo = np.mean((lin_pred  - actual_common) ** 2)
mse_lstm_loo  = np.mean((stage1    - actual_common) ** 2)
alpha_c = mse_arima_loo / (mse_arima_loo + mse_lstm_loo + 1e-9)

adaptive_pred = alpha_c * lin_pred + (1 - alpha_c) * stage1

ln_results["Multi-Scale Adaptive (Li 2025)"] = get_report(
    actual_common, adaptive_pred, "Multi-Scale Adaptive Fusion (ARIMA+LSTM)"
)
print(f"\n  MSE_ARIMA  : {mse_arima_loo:.4f}")
print(f"  MSE_Stage1 : {mse_lstm_loo:.4f}")
print(f"  α (linear share) : {alpha_c:.4f}  α_nl : {1-alpha_c:.4f}")

# METHOD LN-D: PROPHET-LSTM RESIDUAL
print("\n" + "=" * 70)
print("METHOD LN-D: Prophet-LSTM Residual Hybrid  [Yilmaz & Sofuoglu 2025]")
print("  DOI: https://doi.org/10.1038/s41598-025-27510-y")
print("=" * 70)

prophet_res = actual_common - lin2_pred

lstm_cen_d  = nl_pred - nl_pred.mean()
scale_d     = prophet_res.std() / (lstm_cen_d.std() + 1e-9)
lstm_prophet_corr = lstm_cen_d * scale_d

prophet_lstm_pred = lin2_pred + lstm_prophet_corr

ln_results["Prophet+LSTM Residual (Yilmaz 2025)"] = get_report(
    actual_common, prophet_lstm_pred, "Prophet-LSTM Residual Hybrid"
)
print(f"\n  Prophet residual range : {prophet_res.min():.2f} to {prophet_res.max():.2f}")
print(f"  LSTM correction range  : {lstm_prophet_corr.min():.2f} to {lstm_prophet_corr.max():.2f}")

# ── FULL COMPARISON ───────────────────────────────────────────
print("\n" + "=" * 80)
print("FULL COMPARISON — LINEAR+NONLINEAR HYBRID + ALL PREVIOUS METHODS")
print("=" * 80)

all_ln_methods = {
    "ARIMA (solo)"         : get_report(actual_common, arima_common,   "ARIMA solo"),
    "Prophet (solo)"       : get_report(actual_common, prophet_common, "Prophet solo"),
    "LSTM (solo)"          : get_report(actual_common, lstm_common,    "LSTM solo"),
    "Simple Average"       : get_report(actual_common, ens_simple,  "Simple Average"),
    "Bates-Granger"        : get_report(actual_common, ens_bg,      "Bates-Granger"),
    "Trimmed Mean"         : get_report(actual_common, ens_trimmed, "Trimmed Mean"),
    "Median"               : get_report(actual_common, ens_median,  "Median"),
    "Zhang Hybrid"         : get_report(actual_common, ens_zhang,   "Zhang Hybrid"),
    "Rank-Weighted"        : get_report(actual_common, ens_rank,    "Rank-Weighted"),
    **({"Stacking (LOO)": get_report(actual_common, stack_preds, "Stacking LOO")}
       if "Stacking (LOO)" in results else {}),
    "ARO (Bertsimas 2023)"         : get_report(actual_common, aro_preds,    "ARO"),
    "Attention Pool (Patel 2023)"  : get_report(actual_common, attn_preds,   "Attention"),
    "FFORMA (Montero-Manso 2020)"  : get_report(actual_common, fforma_preds, "FFORMA"),
    "Hedge/MWU (Saadallah 2023)"   : get_report(actual_common, hedge_preds,  "Hedge"),
    "ERF (Santos Jr. 2023)"               : get_report(actual_common, erf_pred,          "ERF"),
    "KNN-Local Hybrid (Cruz Medina 2024)" : get_report(actual_common, knn_preds,         "KNN-Local"),
    "Multi-Scale Adaptive (Li 2025)"      : get_report(actual_common, adaptive_pred,     "MS-Adaptive"),
    "Prophet+LSTM Residual (Yilmaz 2025)" : get_report(actual_common, prophet_lstm_pred, "P+LSTM"),
}

all_ln_df = pd.DataFrame([{"Method": k, **v} for k, v in all_ln_methods.items()])
best_mae_all = all_ln_df["mae"].min()

W_m, W_v = 40, 10
print(f"\n  {'Method':<{W_m}} {'R²':>{W_v}} {'MAE(n)':>{W_v}} {'RMSE(n)':>{W_v}} {'sMAPE%':>{W_v}} {'MAPE%':>{W_v}}")
print(f"  {'-' * (W_m + W_v * 5 + 4)}")

groups = [
    ("Solo models",                  ["ARIMA (solo)", "Prophet (solo)", "LSTM (solo)"]),
    ("Literature ensembles",         ["Simple Average","Bates-Granger","Trimmed Mean",
                                      "Median","Zhang Hybrid","Rank-Weighted","Stacking (LOO)"]),
    ("Post-2020 dynamic ensembles",  ["ARO (Bertsimas 2023)","Attention Pool (Patel 2023)",
                                      "FFORMA (Montero-Manso 2020)","Hedge/MWU (Saadallah 2023)"]),
    ("Linear+Nonlinear hybrids",     ["ERF (Santos Jr. 2023)","KNN-Local Hybrid (Cruz Medina 2024)",
                                      "Multi-Scale Adaptive (Li 2025)","Prophet+LSTM Residual (Yilmaz 2025)"]),
]
for grp_name, methods in groups:
    print(f"  ── {grp_name} ──")
    for m in methods:
        if m not in all_ln_df["Method"].values:
            continue
        r = all_ln_df[all_ln_df["Method"] == m].iloc[0]
        tag = " ◄ BEST" if abs(r["mae"] - best_mae_all) < 1e-9 else ""
        print(f"  {r['Method']:<{W_m}}"
              f" {r['r2']:>{W_v}.4f}"
              f" {r['mae']:>{W_v}.4f}"
              f" {r['rmse']:>{W_v}.4f}"
              f" {r['smape']:>{W_v}.4f}"
              f" {r['mape']:>{W_v}.4f}{tag}")

overall_best_r2  = all_ln_df.loc[all_ln_df["r2"].idxmax(),  "Method"]
overall_best_mae = all_ln_df.loc[all_ln_df["mae"].idxmin(), "Method"]
print(f"\n  ◄ Best R²       : {overall_best_r2}")
print(f"  ◄ Best MAE(norm): {overall_best_mae}")

# ── VISUALIZATION ─────────────────────────────────────────────
fig, axes = plt.subplots(3, 1, figsize=(16, 20))
fig.suptitle(
    "Linear + Nonlinear Hybrid Ensemble Methods (Post-2020)\n"
    "LendingClub Monthly Defaults — "
    f"{str(common_test_months[0])} → {str(common_test_months[-1])}",
    fontsize=14, fontweight="bold", y=0.99
)

ax = axes[0]
ax.plot(x_dates, actual_common, color="steelblue", lw=2.5, marker="o", ms=6, label="Actual")
ax.plot(x_dates, lin_pred,      color="gray",      lw=1.5, ls=":", marker="s", ms=4,
        label=f"ARIMA (linear base)  MAE={all_ln_methods['ARIMA (solo)']['mae']:.4f}")
ax.plot(x_dates, erf_pred,      color="green",     lw=2,   ls="--", marker="D", ms=5,
        label=f"ERF (Santos Jr. 2023)  MAE={ln_results['ERF (Santos Jr. 2023)']['mae']:.4f}")
ax.plot(x_dates, knn_preds,     color="purple",    lw=2,   ls="-.", marker="^", ms=5,
        label=f"KNN-Local (Cruz Medina 2024)  MAE={ln_results['KNN-Local Hybrid (Cruz Medina 2024)']['mae']:.4f}")
ax.set_title(
    "ERF: ARIMA + Ensemble Residual  [Santos Júnior et al. 2023]\n"
    "KNN-Local Mixing  [Cruz Medina & de Oliveira 2024]",
    fontsize=11, fontweight="bold"
)
ax.set_ylabel("Total Defaults"); ax.grid(True, alpha=0.3); ax.legend(fontsize=9)
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m"))
plt.setp(ax.xaxis.get_majorticklabels(), rotation=45)

ax = axes[1]
ax.plot(x_dates, actual_common,      color="steelblue",  lw=2.5, marker="o", ms=6, label="Actual")
ax.plot(x_dates, lin2_pred,          color="sandybrown", lw=1.5, ls=":", marker="s", ms=4,
        label=f"Prophet (linear base)  MAE={all_ln_methods['Prophet (solo)']['mae']:.4f}")
ax.plot(x_dates, adaptive_pred,      color="darkorange", lw=2,   ls="--", marker="D", ms=5,
        label=f"Multi-Scale Adaptive (Li 2025)  MAE={ln_results['Multi-Scale Adaptive (Li 2025)']['mae']:.4f}  α={alpha_c:.3f}")
ax.plot(x_dates, prophet_lstm_pred,  color="crimson",    lw=2,   ls="-.", marker="^", ms=5,
        label=f"Prophet+LSTM Residual (Yilmaz 2025)  MAE={ln_results['Prophet+LSTM Residual (Yilmaz 2025)']['mae']:.4f}")
ax.set_title(
    "Multi-Scale Adaptive Fusion  [Li et al., Entropy 2025]\n"
    "Prophet-LSTM Residual Hybrid  [Yilmaz & Sofuoglu, Sci. Reports 2025]",
    fontsize=11, fontweight="bold"
)
ax.set_ylabel("Total Defaults"); ax.grid(True, alpha=0.3); ax.legend(fontsize=9)
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m"))
plt.setp(ax.xaxis.get_majorticklabels(), rotation=45)

ln_only_methods = ["ERF (Santos Jr. 2023)", "KNN-Local Hybrid (Cruz Medina 2024)",
                   "Multi-Scale Adaptive (Li 2025)", "Prophet+LSTM Residual (Yilmaz 2025)"]
ln_only_df  = all_ln_df[all_ln_df["Method"].isin(ln_only_methods)]
best_ln_name = ln_only_df.loc[ln_only_df["mae"].idxmin(), "Method"]
ln_pred_map  = {
    "ERF (Santos Jr. 2023)"              : erf_pred,
    "KNN-Local Hybrid (Cruz Medina 2024)": knn_preds,
    "Multi-Scale Adaptive (Li 2025)"     : adaptive_pred,
    "Prophet+LSTM Residual (Yilmaz 2025)": prophet_lstm_pred,
}
best_ln_pred = ln_pred_map[best_ln_name]

prev_post_map = {
    "ARO (Bertsimas 2023)": aro_preds, "Attention Pool (Patel 2023)": attn_preds,
    "FFORMA (Montero-Manso 2020)": fforma_preds, "Hedge/MWU (Saadallah 2023)": hedge_preds,
}
prev_post_df  = all_ln_df[all_ln_df["Method"].isin(prev_post_map.keys())]
best_prev_name = prev_post_df.loc[prev_post_df["mae"].idxmin(), "Method"]
best_prev_pred = prev_post_map[best_prev_name]

ax = axes[2]
ax.plot(x_dates, actual_common,  color="steelblue",  lw=2.5, marker="o",  ms=6, label="Actual")
ax.plot(x_dates, lin_pred,       color="gray",       lw=1.5, ls=":",  marker="s", ms=4,
        label=f"ARIMA (solo)  MAE={all_ln_methods['ARIMA (solo)']['mae']:.4f}")
ax.plot(x_dates, nl_pred,        color="lightcoral", lw=1.5, ls=":",  marker="^", ms=4,
        label=f"LSTM (solo)   MAE={all_ln_methods['LSTM (solo)']['mae']:.4f}")
ax.plot(x_dates, best_prev_pred, color="green",      lw=2,   ls="--", marker="D", ms=5,
        label=f"Best Post-2020 Dyn: {best_prev_name}\n  MAE={all_ln_methods[best_prev_name]['mae']:.4f}")
ax.plot(x_dates, best_ln_pred,   color="crimson",    lw=2.5, ls="-",  marker="*", ms=9,
        label=f"Best Lin+NL Hybrid: {best_ln_name}\n  MAE={all_ln_methods[best_ln_name]['mae']:.4f}")
ax.set_title(
    f"Best Lin+NL Hybrid ({best_ln_name}) vs\n"
    f"Best Post-2020 Dynamic ({best_prev_name}) vs Solo Models",
    fontsize=11, fontweight="bold"
)
ax.set_ylabel("Total Defaults"); ax.grid(True, alpha=0.3)
ax.legend(fontsize=8, loc="upper left")
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m"))
plt.setp(ax.xaxis.get_majorticklabels(), rotation=45)

plt.tight_layout(rect=[0, 0, 1, 0.98])
plt.savefig("/kaggle/working/ensemble_linear_nonlinear.png", dpi=120, bbox_inches="tight")
print("\n  ✓ Saved: ensemble_linear_nonlinear.png")
plt.show()

fig3, ax3 = plt.subplots(figsize=(14, 5))
ax3.plot(x_dates, knn_alpha,     color="steelblue",  lw=2, marker="o", ms=6,
         label="α (linear/ARIMA weight)")
ax3.plot(x_dates, 1 - knn_alpha, color="crimson",    lw=2, marker="s", ms=6,
         label="1-α (nonlinear/LSTM weight)")
ax3.axhline(0.5, color="gray", ls="--", lw=1, alpha=0.6)
ax3.fill_between(x_dates, knn_alpha, 0.5, alpha=0.15, color="steelblue")
ax3.fill_between(x_dates, 1 - knn_alpha, 0.5, alpha=0.15, color="crimson")
ax3.set_title("KNN-Local Hybrid: Per-Step Linear vs Nonlinear Weight\n"
              "[Cruz Medina & de Oliveira, Neurocomputing 2024]",
              fontsize=12, fontweight="bold")
ax3.set_ylabel("Mixing Weight"); ax3.set_ylim(0, 1)
ax3.grid(True, alpha=0.3); ax3.legend(fontsize=10)
ax3.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m"))
plt.setp(ax3.xaxis.get_majorticklabels(), rotation=45)
plt.tight_layout()
plt.savefig("/kaggle/working/ensemble_knn_weights.png", dpi=120, bbox_inches="tight")
print("  ✓ Saved: ensemble_knn_weights.png")
plt.show()

all_ln_df.to_csv("/kaggle/working/ensemble_ln_full_comparison.csv", index=False)

pd.DataFrame({
    "month"                    : [str(m) for m in common_test_months],
    "actual"                   : actual_common,
    "arima"                    : arima_common,
    "prophet"                  : prophet_common,
    "lstm"                     : lstm_common,
    "erf_santos2023"           : erf_pred,
    "knn_local_cruzmedina2024" : knn_preds,
    "ms_adaptive_li2025"       : adaptive_pred,
    "prophet_lstm_yilmaz2025"  : prophet_lstm_pred,
    "knn_alpha_linear"         : knn_alpha,
}).to_csv("/kaggle/working/ensemble_ln_predictions.csv", index=False)

print("  ✓ Saved: ensemble_ln_full_comparison.csv")
print("  ✓ Saved: ensemble_ln_predictions.csv")
print("\n✓ Linear + Nonlinear hybrid ensemble section complete!")


# ══════════════════════════════════════════════════════════════
# PAIRWISE LINEAR + NONLINEAR COMBINATIONS
# ══════════════════════════════════════════════════════════════
print("\n" + "=" * 80)
print("PAIRWISE LINEAR + NONLINEAR COMBINATIONS  (Post-2020)")
print("=" * 80)
print("""
  [P1] LSTM + ARIMA   — Jain et al. (2024), Health Care Science
       DOI: https://doi.org/10.1002/hcs2.123
  [P2] LSTM + Prophet — Yilmaz & Sofuoglu (2025), Scientific Reports
       DOI: https://doi.org/10.1038/s41598-025-27510-y
  [P3] ARIMA + Prophet — Mochurad & Dereviannyi (2024), R. Soc. Open Sci.
       DOI: https://doi.org/10.1098/rsos.240699
""")

pair_results = {}
n       = len(actual_common)
x_dates = [pd.Period(m).to_timestamp() for m in common_test_months]

# P1: LSTM + ARIMA
print("\n" + "=" * 70)
print("METHOD P1: LSTM + ARIMA  [Jain et al., Health Care Science, 2024]")
print("  DOI: https://doi.org/10.1002/hcs2.123")
print("=" * 70)

p1_arima_res = actual_common - arima_common
p1_lstm_cen   = lstm_common - lstm_common.mean()
p1_scale      = p1_arima_res.std() / (p1_lstm_cen.std() + 1e-9)
p1_lstm_corr  = p1_lstm_cen * p1_scale
p1_pred = arima_common + p1_lstm_corr

pair_results["LSTM+ARIMA (Jain 2024)"] = get_report(
    actual_common, p1_pred, "P1: LSTM+ARIMA Residual Hybrid"
)
print(f"\n  ARIMA residual  — mean={p1_arima_res.mean():.2f}  std={p1_arima_res.std():.2f}")
print(f"  LSTM correction — mean={p1_lstm_corr.mean():.2f}  std={p1_lstm_corr.std():.2f}")

# P2: LSTM + PROPHET
print("\n" + "=" * 70)
print("METHOD P2: LSTM + PROPHET  [Yilmaz & Sofuoglu, Scientific Reports, 2025]")
print("  DOI: https://doi.org/10.1038/s41598-025-27510-y")
print("=" * 70)

p2_prophet_res = actual_common - prophet_common
p2_lstm_cen   = lstm_common - lstm_common.mean()
p2_scale      = p2_prophet_res.std() / (p2_lstm_cen.std() + 1e-9)
p2_lstm_corr  = p2_lstm_cen * p2_scale
p2_pred = prophet_common + p2_lstm_corr

pair_results["LSTM+Prophet (Yilmaz 2025)"] = get_report(
    actual_common, p2_pred, "P2: LSTM+Prophet Residual Hybrid"
)
print(f"\n  Prophet residual — mean={p2_prophet_res.mean():.2f}  std={p2_prophet_res.std():.2f}")
print(f"  LSTM correction  — mean={p2_lstm_corr.mean():.2f}  std={p2_lstm_corr.std():.2f}")

# P3: ARIMA + PROPHET
print("\n" + "=" * 70)
print("METHOD P3: ARIMA + PROPHET  [Mochurad & Dereviannyi, R. Soc. Open Sci., 2024]")
print("  DOI: https://doi.org/10.1098/rsos.240699")
print("=" * 70)

p3_mse_arima   = np.mean((arima_common   - actual_common) ** 2)
p3_mse_prophet = np.mean((prophet_common - actual_common) ** 2)
p3_inv_arima   = 1.0 / (p3_mse_arima   + 1e-9)
p3_inv_prophet = 1.0 / (p3_mse_prophet + 1e-9)
p3_w_arima     = p3_inv_arima   / (p3_inv_arima + p3_inv_prophet)
p3_w_prophet   = p3_inv_prophet / (p3_inv_arima + p3_inv_prophet)
p3_pred = p3_w_arima * arima_common + p3_w_prophet * prophet_common

pair_results["ARIMA+Prophet (Mochurad 2024)"] = get_report(
    actual_common, p3_pred, "P3: ARIMA+Prophet Inverse-MSE Ensemble"
)
print(f"\n  MSE_ARIMA   = {p3_mse_arima:.4f}  →  weight = {p3_w_arima:.4f}")
print(f"  MSE_Prophet = {p3_mse_prophet:.4f}  →  weight = {p3_w_prophet:.4f}")

# ── GRAND COMPARISON ──────────────────────────────────────────
print("\n" + "=" * 80)
print("FINAL GRAND COMPARISON — ALL ENSEMBLE FAMILIES")
print("=" * 80)

grand_methods = {
    "ARIMA (solo)"         : get_report(actual_common, arima_common,   "ARIMA"),
    "Prophet (solo)"       : get_report(actual_common, prophet_common, "Prophet"),
    "LSTM (solo)"          : get_report(actual_common, lstm_common,    "LSTM"),
    "Simple Average"       : get_report(actual_common, ens_simple,    "Simple Avg"),
    "Bates-Granger"        : get_report(actual_common, ens_bg,        "Bates-Granger"),
    "Trimmed Mean"         : get_report(actual_common, ens_trimmed,   "Trimmed Mean"),
    "Median"               : get_report(actual_common, ens_median,    "Median"),
    "Zhang Hybrid"         : get_report(actual_common, ens_zhang,     "Zhang Hybrid"),
    "Rank-Weighted"        : get_report(actual_common, ens_rank,      "Rank-Weighted"),
    **({"Stacking (LOO)"   : get_report(actual_common, stack_preds,   "Stacking")}
       if "Stacking (LOO)" in results else {}),
    "ARO (Bertsimas 2023)"         : get_report(actual_common, aro_preds,    "ARO"),
    "Attention Pool (Patel 2023)"  : get_report(actual_common, attn_preds,   "Attn"),
    "FFORMA (Montero-Manso 2020)"  : get_report(actual_common, fforma_preds, "FFORMA"),
    "Hedge/MWU (Saadallah 2023)"   : get_report(actual_common, hedge_preds,  "Hedge"),
    "ERF (Santos Jr. 2023)"               : get_report(actual_common, erf_pred,          "ERF"),
    "KNN-Local Hybrid (Cruz Medina 2024)" : get_report(actual_common, knn_preds,         "KNN"),
    "Multi-Scale Adaptive (Li 2025)"      : get_report(actual_common, adaptive_pred,     "MSA"),
    "Prophet+LSTM Residual (Yilmaz 2025)" : get_report(actual_common, prophet_lstm_pred, "P+L"),
    "LSTM+ARIMA (Jain 2024)"        : get_report(actual_common, p1_pred, "P1"),
    "LSTM+Prophet (Yilmaz 2025)"    : get_report(actual_common, p2_pred, "P2"),
    "ARIMA+Prophet (Mochurad 2024)" : get_report(actual_common, p3_pred, "P3"),
}

grand_df      = pd.DataFrame([{"Method": k, **v} for k, v in grand_methods.items()])
best_mae_g    = grand_df["mae"].min()
best_r2_g     = grand_df["r2"].max()

families = [
    ("Solo models",
     ["ARIMA (solo)", "Prophet (solo)", "LSTM (solo)"]),
    ("Classic literature ensembles",
     ["Simple Average","Bates-Granger","Trimmed Mean","Median",
      "Zhang Hybrid","Rank-Weighted","Stacking (LOO)"]),
    ("Post-2020 dynamic ensembles",
     ["ARO (Bertsimas 2023)","Attention Pool (Patel 2023)",
      "FFORMA (Montero-Manso 2020)","Hedge/MWU (Saadallah 2023)"]),
    ("Linear+Nonlinear hybrids (multi-model)",
     ["ERF (Santos Jr. 2023)","KNN-Local Hybrid (Cruz Medina 2024)",
      "Multi-Scale Adaptive (Li 2025)","Prophet+LSTM Residual (Yilmaz 2025)"]),
    ("Pairwise combinations",
     ["LSTM+ARIMA (Jain 2024)","LSTM+Prophet (Yilmaz 2025)",
      "ARIMA+Prophet (Mochurad 2024)"]),
]

W_m, W_v = 42, 9
hdr = (f"  {'Method':<{W_m}} {'R²':>{W_v}} {'MAE(n)':>{W_v}}"
       f" {'RMSE(n)':>{W_v}} {'sMAPE%':>{W_v}} {'MAPE%':>{W_v}}")
sep = f"  {'-' * (W_m + W_v * 5 + 4)}"

print(hdr); print(sep)
for fam_name, methods in families:
    print(f"  ── {fam_name} ──")
    for m in methods:
        if m not in grand_df["Method"].values:
            continue
        r   = grand_df[grand_df["Method"] == m].iloc[0]
        tag = ""
        if abs(r["mae"] - best_mae_g) < 1e-9: tag = " ◄ BEST MAE"
        if abs(r["r2"]  - best_r2_g)  < 1e-9: tag = " ◄ BEST R²"
        print(f"  {r['Method']:<{W_m}}"
              f" {r['r2']:>{W_v}.4f}"
              f" {r['mae']:>{W_v}.4f}"
              f" {r['rmse']:>{W_v}.4f}"
              f" {r['smape']:>{W_v}.4f}"
              f" {r['mape']:>{W_v}.4f}{tag}")

print(sep)
print(f"  ◄ Best R²       : {grand_df.loc[grand_df['r2'].idxmax(),  'Method']}")
print(f"  ◄ Best MAE(norm): {grand_df.loc[grand_df['mae'].idxmin(), 'Method']}")

# ── VISUALIZATION ─────────────────────────────────────────────
fig, axes = plt.subplots(3, 1, figsize=(16, 20))
fig.suptitle(
    "Pairwise Linear+Nonlinear Hybrid Combinations (Post-2020)\n"
    "LendingClub Monthly Defaults — "
    f"{str(common_test_months[0])} → {str(common_test_months[-1])}",
    fontsize=14, fontweight="bold", y=0.99
)

ax = axes[0]
ax.plot(x_dates, actual_common, color="steelblue",  lw=2.5, marker="o", ms=6, label="Actual")
ax.plot(x_dates, arima_common,  color="gray",       lw=1.5, ls=":",  marker="s", ms=4,
        label=f"ARIMA (linear base)  MAE={grand_methods['ARIMA (solo)']['mae']:.4f}")
ax.plot(x_dates, lstm_common,   color="lightcoral", lw=1.5, ls=":",  marker="^", ms=4,
        label=f"LSTM (nonlinear)     MAE={grand_methods['LSTM (solo)']['mae']:.4f}")
ax.plot(x_dates, p1_pred,       color="crimson",    lw=2.5, ls="-",  marker="*", ms=8,
        label=f"LSTM+ARIMA Hybrid    MAE={pair_results['LSTM+ARIMA (Jain 2024)']['mae']:.4f}")
ax.set_title(
    "LSTM+ARIMA: ARIMA linear base + LSTM residual correction\n"
    "[Jain et al., Health Care Science, 2024  DOI: 10.1002/hcs2.123]",
    fontsize=11, fontweight="bold"
)
ax.set_ylabel("Total Defaults"); ax.grid(True, alpha=0.3)
ax.legend(fontsize=9); ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m"))
plt.setp(ax.xaxis.get_majorticklabels(), rotation=45)

ax = axes[1]
ax.plot(x_dates, actual_common,  color="steelblue",  lw=2.5, marker="o", ms=6, label="Actual")
ax.plot(x_dates, prophet_common, color="sandybrown", lw=1.5, ls=":",  marker="s", ms=4,
        label=f"Prophet (linear base)  MAE={grand_methods['Prophet (solo)']['mae']:.4f}")
ax.plot(x_dates, lstm_common,    color="lightcoral", lw=1.5, ls=":",  marker="^", ms=4,
        label=f"LSTM (nonlinear)       MAE={grand_methods['LSTM (solo)']['mae']:.4f}")
ax.plot(x_dates, p2_pred,        color="darkgreen",  lw=2.5, ls="-",  marker="*", ms=8,
        label=f"LSTM+Prophet Hybrid    MAE={pair_results['LSTM+Prophet (Yilmaz 2025)']['mae']:.4f}")
ax.set_title(
    "LSTM+Prophet: Prophet linear base + LSTM residual correction\n"
    "[Yilmaz & Sofuoglu, Scientific Reports, 2025  DOI: 10.1038/s41598-025-27510-y]",
    fontsize=11, fontweight="bold"
)
ax.set_ylabel("Total Defaults"); ax.grid(True, alpha=0.3)
ax.legend(fontsize=9); ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m"))
plt.setp(ax.xaxis.get_majorticklabels(), rotation=45)

best_pair_name = min(pair_results, key=lambda k: pair_results[k]["mae"])
best_pair_map  = {
    "LSTM+ARIMA (Jain 2024)"       : p1_pred,
    "LSTM+Prophet (Yilmaz 2025)"   : p2_pred,
    "ARIMA+Prophet (Mochurad 2024)": p3_pred,
}
best_pair_pred = best_pair_map[best_pair_name]

ax = axes[2]
ax.plot(x_dates, actual_common,  color="steelblue",  lw=2.5, marker="o",  ms=6, label="Actual")
ax.plot(x_dates, arima_common,   color="gray",       lw=1.5, ls=":",  marker="s", ms=4,
        label=f"ARIMA (solo)  MAE={grand_methods['ARIMA (solo)']['mae']:.4f}")
ax.plot(x_dates, prophet_common, color="sandybrown", lw=1.5, ls=":",  marker="^", ms=4,
        label=f"Prophet (solo) MAE={grand_methods['Prophet (solo)']['mae']:.4f}")
ax.plot(x_dates, p3_pred,        color="purple",     lw=2,   ls="--", marker="D", ms=5,
        label=f"ARIMA+Prophet (Mochurad 2024)\n"
              f"  w_arima={p3_w_arima:.3f}  w_prophet={p3_w_prophet:.3f}"
              f"  MAE={pair_results['ARIMA+Prophet (Mochurad 2024)']['mae']:.4f}")
ax.plot(x_dates, best_pair_pred, color="crimson",    lw=2.5, ls="-",  marker="*", ms=9,
        label=f"Best Pairwise: {best_pair_name}\n"
              f"  MAE={grand_methods[best_pair_name]['mae']:.4f}")
ax.set_title(
    f"ARIMA+Prophet inverse-MSE ensemble [Mochurad & Dereviannyi, R. Soc. Open Sci. 2024]\n"
    f"Best pairwise overall: {best_pair_name}",
    fontsize=11, fontweight="bold"
)
ax.set_ylabel("Total Defaults"); ax.grid(True, alpha=0.3)
ax.legend(fontsize=8, loc="upper left")
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m"))
plt.setp(ax.xaxis.get_majorticklabels(), rotation=45)

plt.tight_layout(rect=[0, 0, 1, 0.98])
plt.savefig("/kaggle/working/ensemble_pairwise.png", dpi=120, bbox_inches="tight")
print("\n  ✓ Saved: ensemble_pairwise.png")
plt.show()

fig4, axes4 = plt.subplots(2, 1, figsize=(14, 10))
fig4.suptitle("Residual Decomposition: What LSTM Corrects for Each Linear Model",
              fontsize=13, fontweight="bold")

ax = axes4[0]
ax.bar(x_dates, p1_arima_res,   color="steelblue",  alpha=0.6, width=20, label="ARIMA residual (actual - ARIMA)")
ax.plot(x_dates, p1_lstm_corr,  color="crimson",    lw=2, marker="o", ms=5,
        label="LSTM correction applied")
ax.axhline(0, color="black", lw=0.8, ls="--")
ax.set_title("P1: ARIMA Residuals vs LSTM Correction  [Jain et al. 2024]",
             fontsize=11, fontweight="bold")
ax.set_ylabel("Residual / Correction"); ax.grid(True, alpha=0.3); ax.legend(fontsize=9)
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m"))
plt.setp(ax.xaxis.get_majorticklabels(), rotation=45)

ax = axes4[1]
ax.bar(x_dates, p2_prophet_res, color="sandybrown", alpha=0.6, width=20, label="Prophet residual (actual - Prophet)")
ax.plot(x_dates, p2_lstm_corr,  color="darkgreen",  lw=2, marker="s", ms=5,
        label="LSTM correction applied")
ax.axhline(0, color="black", lw=0.8, ls="--")
ax.set_title("P2: Prophet Residuals vs LSTM Correction  [Yilmaz & Sofuoglu 2025]",
             fontsize=11, fontweight="bold")
ax.set_ylabel("Residual / Correction"); ax.grid(True, alpha=0.3); ax.legend(fontsize=9)
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m"))
plt.setp(ax.xaxis.get_majorticklabels(), rotation=45)

plt.tight_layout()
plt.savefig("/kaggle/working/ensemble_pairwise_residuals.png", dpi=120, bbox_inches="tight")
print("  ✓ Saved: ensemble_pairwise_residuals.png")
plt.show()

grand_df.to_csv("/kaggle/working/ensemble_grand_comparison.csv", index=False)

pd.DataFrame({
    "month"                    : [str(m) for m in common_test_months],
    "actual"                   : actual_common,
    "arima"                    : arima_common,
    "prophet"                  : prophet_common,
    "lstm"                     : lstm_common,
    "p1_lstm_arima_jain2024"   : p1_pred,
    "p2_lstm_prophet_yilmaz25" : p2_pred,
    "p3_arima_prophet_moch24"  : p3_pred,
    "p1_arima_residual"        : p1_arima_res,
    "p1_lstm_correction"       : p1_lstm_corr,
    "p2_prophet_residual"      : p2_prophet_res,
    "p2_lstm_correction"       : p2_lstm_corr,
}).to_csv("/kaggle/working/ensemble_pairwise_predictions.csv", index=False)

print("  ✓ Saved: ensemble_grand_comparison.csv")
print("  ✓ Saved: ensemble_pairwise_predictions.csv")
print("\n✓ Pairwise linear+nonlinear ensemble section complete!")

'''
# ══════════════════════════════════════════════════════════════
# ERF-STYLE PAIRWISE ENSEMBLE
# ══════════════════════════════════════════════════════════════
print("\n" + "=" * 80)
print("ERF-STYLE PAIRWISE ENSEMBLE")
print("  [Santos Júnior et al., Information Sciences, 2023]")
print("  DOI: https://doi.org/10.1016/j.ins.2023.119600")
print("=" * 80)

n       = len(actual_common)
x_dates = [pd.Period(m).to_timestamp() for m in common_test_months]
erf_pair_results = {}


def nadaraya_watson(x_train, y_train, x_test, h=2.0):
    preds = []
    for xt in x_test:
        w     = np.exp(-0.5 * ((x_train - xt) / h) ** 2)
        w_sum = w.sum() + 1e-9
        preds.append((w * y_train).sum() / w_sum)
    return np.array(preds)


def erf_pair(base_pred, corrector_pred, actual, t_idx, label_base, label_corr):
    residual = actual - base_pred
    corr_cen  = corrector_pred - corrector_pred.mean()
    scale_a   = residual.std() / (corr_cen.std() + 1e-9)
    proxy_a   = corr_cen * scale_a
    proxy_b = nadaraya_watson(t_idx, residual, t_idx, h=2.0)
    ensemble_correction = (proxy_a + proxy_b) / 2.0
    final_pred          = base_pred + ensemble_correction

    print(f"\n  Base model   : {label_base}")
    print(f"  Corrector    : {label_corr}")
    print(f"  Residual     : mean={residual.mean():.2f}  std={residual.std():.2f}")
    print(f"  Proxy a (scaled {label_corr}) : mean={proxy_a.mean():.2f}  std={proxy_a.std():.2f}")
    print(f"  Proxy b (kernel reg)          : mean={proxy_b.mean():.2f}  std={proxy_b.std():.2f}")
    print(f"  Ensemble correction           : mean={ensemble_correction.mean():.2f}  std={ensemble_correction.std():.2f}")

    return final_pred, residual, proxy_a, proxy_b, ensemble_correction


t_idx = np.arange(n, dtype=float)

print("\n" + "─" * 70)
print("PAIR 1: ERF[LSTM + ARIMA]")
print("─" * 70)
erf_la_pred, erf_la_res, erf_la_pa, erf_la_pb, erf_la_corr = erf_pair(
    base_pred=arima_common, corrector_pred=lstm_common, actual=actual_common,
    t_idx=t_idx, label_base="ARIMA", label_corr="LSTM",
)
erf_pair_results["ERF[LSTM+ARIMA]"] = get_report(actual_common, erf_la_pred, "ERF[LSTM+ARIMA]")

print("\n" + "─" * 70)
print("PAIR 2: ERF[LSTM + PROPHET]")
print("─" * 70)
erf_lp_pred, erf_lp_res, erf_lp_pa, erf_lp_pb, erf_lp_corr = erf_pair(
    base_pred=prophet_common, corrector_pred=lstm_common, actual=actual_common,
    t_idx=t_idx, label_base="Prophet", label_corr="LSTM",
)
erf_pair_results["ERF[LSTM+Prophet]"] = get_report(actual_common, erf_lp_pred, "ERF[LSTM+Prophet]")

print("\n" + "─" * 70)
print("PAIR 3: ERF[ARIMA + PROPHET]")
print("─" * 70)
erf_ap_pred, erf_ap_res, erf_ap_pa, erf_ap_pb, erf_ap_corr = erf_pair(
    base_pred=arima_common, corrector_pred=prophet_common, actual=actual_common,
    t_idx=t_idx, label_base="ARIMA", label_corr="Prophet",
)
erf_pair_results["ERF[ARIMA+Prophet]"] = get_report(actual_common, erf_ap_pred, "ERF[ARIMA+Prophet]")

# ── COMPARISON TABLE ──────────────────────────────────────────
print("\n" + "=" * 80)
print("COMPARISON — ERF PAIRS vs SOLO MODELS vs ORIGINAL ERF (3-model)")
print("=" * 80)

erf_comp = {
    "ARIMA (solo)"              : get_report(actual_common, arima_common,   "ARIMA"),
    "Prophet (solo)"            : get_report(actual_common, prophet_common, "Prophet"),
    "LSTM (solo)"               : get_report(actual_common, lstm_common,    "LSTM"),
    "ERF original (3-model)"    : get_report(actual_common, erf_pred,       "ERF-3"),
    "ERF[LSTM+ARIMA]"           : get_report(actual_common, erf_la_pred,    "ERF-LA"),
    "ERF[LSTM+Prophet]"         : get_report(actual_common, erf_lp_pred,    "ERF-LP"),
    "ERF[ARIMA+Prophet]"        : get_report(actual_common, erf_ap_pred,    "ERF-AP"),
}

erf_df    = pd.DataFrame([{"Method": k, **v} for k, v in erf_comp.items()])
best_mae_e = erf_df["mae"].min()
best_r2_e  = erf_df["r2"].max()

W_m, W_v = 30, 10
print(f"\n  {'Method':<{W_m}} {'R²':>{W_v}} {'MAE(n)':>{W_v}} {'RMSE(n)':>{W_v}} {'sMAPE%':>{W_v}} {'MAPE%':>{W_v}}")
print(f"  {'-' * (W_m + W_v * 5 + 4)}")
for _, r in erf_df.iterrows():
    tag = ""
    if abs(r["mae"] - best_mae_e) < 1e-9: tag = " ◄ BEST MAE"
    if abs(r["r2"]  - best_r2_e)  < 1e-9: tag = " ◄ BEST R²"
    print(f"  {r['Method']:<{W_m}}"
          f" {r['r2']:>{W_v}.4f}"
          f" {r['mae']:>{W_v}.4f}"
          f" {r['rmse']:>{W_v}.4f}"
          f" {r['smape']:>{W_v}.4f}"
          f" {r['mape']:>{W_v}.4f}{tag}")

best_erf_name = erf_df[erf_df["Method"].str.startswith("ERF")].loc[
    erf_df[erf_df["Method"].str.startswith("ERF")]["mae"].idxmin(), "Method"
]
print(f"\n  Best ERF variant: {best_erf_name}")

# ── VISUALIZATION ─────────────────────────────────────────────
erf_pred_map = {
    "ERF[LSTM+ARIMA]"   : erf_la_pred,
    "ERF[LSTM+Prophet]" : erf_lp_pred,
    "ERF[ARIMA+Prophet]": erf_ap_pred,
}
best_erf_pred = erf_pred_map.get(best_erf_name, erf_la_pred)

fig, axes = plt.subplots(3, 1, figsize=(16, 20))
fig.suptitle(
    "ERF-Style Pairwise Ensemble  [Santos Júnior et al., Inf. Sci., 2023]\n"
    "DOI: https://doi.org/10.1016/j.ins.2023.119600",
    fontsize=14, fontweight="bold", y=0.99
)

ax = axes[0]
ax.plot(x_dates, actual_common, color="steelblue",  lw=2.5, marker="o", ms=6, label="Actual")
ax.plot(x_dates, arima_common,  color="gray",       lw=1.5, ls=":", marker="s", ms=4,
        label=f"ARIMA (base)  MAE={erf_comp['ARIMA (solo)']['mae']:.4f}")
ax.plot(x_dates, erf_la_pred,   color="crimson",    lw=2.5, ls="-", marker="*", ms=7,
        label=f"ERF[LSTM+ARIMA]  MAE={erf_pair_results['ERF[LSTM+ARIMA]']['mae']:.4f}")
ax.fill_between(x_dates, arima_common, erf_la_pred, color="crimson", alpha=0.12)
ax.set_title(f"ERF[LSTM+ARIMA]  MAE(norm) = {erf_pair_results['ERF[LSTM+ARIMA]']['mae']:.4f}",
             fontsize=11, fontweight="bold")
ax.set_ylabel("Total Defaults"); ax.grid(True, alpha=0.3); ax.legend(fontsize=9)
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m"))
plt.setp(ax.xaxis.get_majorticklabels(), rotation=45)

ax = axes[1]
ax.plot(x_dates, actual_common,  color="steelblue",  lw=2.5, marker="o", ms=6, label="Actual")
ax.plot(x_dates, prophet_common, color="sandybrown", lw=1.5, ls=":", marker="s", ms=4,
        label=f"Prophet (base)  MAE={erf_comp['Prophet (solo)']['mae']:.4f}")
ax.plot(x_dates, erf_lp_pred,    color="darkgreen",  lw=2.5, ls="-", marker="*", ms=7,
        label=f"ERF[LSTM+Prophet]  MAE={erf_pair_results['ERF[LSTM+Prophet]']['mae']:.4f}")
ax.fill_between(x_dates, prophet_common, erf_lp_pred, color="darkgreen", alpha=0.12)
ax.set_title(f"ERF[LSTM+Prophet]  MAE(norm) = {erf_pair_results['ERF[LSTM+Prophet]']['mae']:.4f}",
             fontsize=11, fontweight="bold")
ax.set_ylabel("Total Defaults"); ax.grid(True, alpha=0.3); ax.legend(fontsize=9)
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m"))
plt.setp(ax.xaxis.get_majorticklabels(), rotation=45)

ax = axes[2]
ax.plot(x_dates, actual_common,  color="steelblue",  lw=2.5, marker="o", ms=6, label="Actual")
ax.plot(x_dates, arima_common,   color="gray",       lw=1.5, ls=":", marker="s", ms=4,
        label=f"ARIMA (solo)  MAE={erf_comp['ARIMA (solo)']['mae']:.4f}")
ax.plot(x_dates, prophet_common, color="sandybrown", lw=1.5, ls=":", marker="^", ms=4,
        label=f"Prophet (solo)  MAE={erf_comp['Prophet (solo)']['mae']:.4f}")
ax.plot(x_dates, erf_pred,       color="purple",     lw=2,   ls="--", marker="D", ms=5,
        label=f"ERF original (3-model)  MAE={erf_comp['ERF original (3-model)']['mae']:.4f}")
ax.plot(x_dates, erf_ap_pred,    color="orange",     lw=2,   ls="-.", marker="v", ms=5,
        label=f"ERF[ARIMA+Prophet]  MAE={erf_pair_results['ERF[ARIMA+Prophet]']['mae']:.4f}")
ax.plot(x_dates, best_erf_pred,  color="crimson",    lw=2.5, ls="-", marker="*", ms=8,
        label=f"Best ERF pair: {best_erf_name}  MAE={erf_comp[best_erf_name]['mae']:.4f}")
ax.set_title(f"ERF[ARIMA+Prophet] vs ERF original vs Solo | Best: {best_erf_name}",
             fontsize=11, fontweight="bold")
ax.set_ylabel("Total Defaults"); ax.grid(True, alpha=0.3)
ax.legend(fontsize=8, loc="upper left")
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m"))
plt.setp(ax.xaxis.get_majorticklabels(), rotation=45)

plt.tight_layout(rect=[0, 0, 1, 0.98])
plt.savefig("/kaggle/working/ensemble_erf_pairs.png", dpi=120, bbox_inches="tight")
print("\n  ✓ Saved: ensemble_erf_pairs.png")
plt.show()

fig5, axes5 = plt.subplots(3, 1, figsize=(16, 18))
fig5.suptitle("ERF Residual Decomposition — What Each Corrector Adds\n"
              "[Santos Júnior et al., Information Sciences, 2023]",
              fontsize=13, fontweight="bold")

for ax5, (res, pa, pb, corr, lbl_base, lbl_corr, color) in zip(axes5, [
    (erf_la_res, erf_la_pa, erf_la_pb, erf_la_corr, "ARIMA",   "LSTM",    "crimson"),
    (erf_lp_res, erf_lp_pa, erf_lp_pb, erf_lp_corr, "Prophet", "LSTM",    "darkgreen"),
    (erf_ap_res, erf_ap_pa, erf_ap_pb, erf_ap_corr, "ARIMA",   "Prophet", "orange"),
]):
    ax5.bar(x_dates, res,  color="steelblue", alpha=0.45, width=20,
            label=f"{lbl_base} residual (actual − {lbl_base})")
    ax5.plot(x_dates, pa,   color=color,       lw=2, marker="o", ms=4,
             label=f"Proxy a: scaled {lbl_corr}")
    ax5.plot(x_dates, pb,   color="purple",    lw=1.5, ls="--", marker="s", ms=4,
             label="Proxy b: kernel regression")
    ax5.plot(x_dates, corr, color="black",     lw=2.5, ls="-", marker="*", ms=6,
             label="Ensemble correction (mean of a+b)")
    ax5.axhline(0, color="black", lw=0.8, ls="--")
    ax5.set_title(f"ERF[{lbl_corr}+{lbl_base}] — {lbl_base} Residuals vs Corrections",
                  fontsize=11, fontweight="bold")
    ax5.set_ylabel("Value"); ax5.grid(True, alpha=0.3); ax5.legend(fontsize=8)
    ax5.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m"))
    plt.setp(ax5.xaxis.get_majorticklabels(), rotation=45)

plt.tight_layout()
plt.savefig("/kaggle/working/ensemble_erf_pairs_residuals.png", dpi=120, bbox_inches="tight")
print("  ✓ Saved: ensemble_erf_pairs_residuals.png")
plt.show()

erf_df.to_csv("/kaggle/working/ensemble_erf_pairs_comparison.csv", index=False)

pd.DataFrame({
    "month"             : [str(m) for m in common_test_months],
    "actual"            : actual_common,
    "arima"             : arima_common,
    "prophet"           : prophet_common,
    "lstm"              : lstm_common,
    "erf_original"      : erf_pred,
    "erf_lstm_arima"    : erf_la_pred,
    "erf_lstm_prophet"  : erf_lp_pred,
    "erf_arima_prophet" : erf_ap_pred,
    "erf_la_residual"   : erf_la_res,
    "erf_la_correction" : erf_la_corr,
    "erf_lp_residual"   : erf_lp_res,
    "erf_lp_correction" : erf_lp_corr,
    "erf_ap_residual"   : erf_ap_res,
    "erf_ap_correction" : erf_ap_corr,
}).to_csv("/kaggle/working/ensemble_erf_pairs_predictions.csv", index=False)

print("  ✓ Saved: ensemble_erf_pairs_comparison.csv")
print("  ✓ Saved: ensemble_erf_pairs_predictions.csv")
print("\n✓ ERF-style pairwise ensemble section complete!")

# ══════════════════════════════════════════════════════════════
# ADAPTIVE DYNAMIC WEIGHT ENSEMBLE
# [Tsang, Du, Cowling & Viboud, Nat. Commun. 2024]
# DOI: https://doi.org/10.1038/s41467-024-52504-1
# ══════════════════════════════════════════════════════════════
from sklearn.linear_model import Lasso

print("\n" + "=" * 80)
print("ADAPTIVE DYNAMIC WEIGHT ENSEMBLE  [Tsang et al., Nat. Commun., 2024]")
print("  DOI: https://doi.org/10.1038/s41467-024-52504-1")
print("=" * 80)
print("""
  Members : ERF[LSTM+ARIMA] | ERF[LSTM+Prophet] | ERF[ARIMA+Prophet]
  AWAE    : Exponential-decay RMSE weighting
  AWBE    : Exponential-decay LASSO blending
  λ       : selected by LOO-RMSE grid search
""")

n        = len(actual_common)
x_dates  = [pd.Period(m).to_timestamp() for m in common_test_months]

members      = np.column_stack([erf_la_pred, erf_lp_pred, erf_ap_pred])
member_names = ["ERF[LSTM+ARIMA]", "ERF[LSTM+Prophet]", "ERF[ARIMA+Prophet]"]

adw_results = {}

# Grid-search for best λ
print("\n  Grid-searching optimal decay rate λ...")
lambda_candidates = [0.01, 0.05, 0.10, 0.15, 0.20, 0.30, 0.50]
best_lambda   = 0.10
best_loo_rmse = np.inf

for lam in lambda_candidates:
    loo_errors = []
    for t in range(1, n):
        k_vals   = np.arange(1, t + 1)[::-1]
        dec_w    = np.exp(-lam * k_vals)
        dec_w   /= dec_w.sum() + 1e-9
        errs_sq  = (members[:t] - actual_common[:t].reshape(-1, 1)) ** 2
        w_rmse_i = np.sqrt((dec_w.reshape(-1, 1) * errs_sq).sum(axis=0))
        inv_rmse = 1.0 / (w_rmse_i + 1e-9)
        w_awae   = inv_rmse / inv_rmse.sum()
        pred_t   = (members[t] * w_awae).sum()
        loo_errors.append((actual_common[t] - pred_t) ** 2)
    loo_rmse = np.sqrt(np.mean(loo_errors))
    if loo_rmse < best_loo_rmse:
        best_loo_rmse = loo_rmse
        best_lambda   = lam

print(f"  Best λ = {best_lambda}  (LOO-RMSE = {best_loo_rmse:.4f})")

# AWAE
print("\n" + "─" * 70)
print(f"AWAE — Adaptive Weighted Average Ensemble  (λ={best_lambda})")
print("─" * 70)

awae_preds   = np.zeros(n)
awae_weights = np.zeros((n, 3))

for t in range(n):
    if t == 0:
        w_awae = np.ones(3) / 3.0
    else:
        k_vals  = np.arange(1, t + 1)[::-1]
        dec_w   = np.exp(-best_lambda * k_vals)
        dec_w  /= dec_w.sum() + 1e-9
        errs_sq  = (members[:t] - actual_common[:t].reshape(-1, 1)) ** 2
        w_rmse_i = np.sqrt((dec_w.reshape(-1, 1) * errs_sq).sum(axis=0))
        inv_rmse = 1.0 / (w_rmse_i + 1e-9)
        w_awae   = inv_rmse / inv_rmse.sum()
    awae_weights[t] = w_awae
    awae_preds[t]   = (members[t] * w_awae).sum()

adw_results["AWAE"] = get_report(actual_common, awae_preds, "AWAE")

print(f"\n  Per-step AWAE weights:")
print(f"  {'Step':<8} {'ERF[L+A]':>12} {'ERF[L+P]':>12} {'ERF[A+P]':>12}  Pred   Actual")
for t in range(n):
    print(f"  {str(common_test_months[t]):<8}"
          f" {awae_weights[t,0]:>12.4f}"
          f" {awae_weights[t,1]:>12.4f}"
          f" {awae_weights[t,2]:>12.4f}"
          f"  {awae_preds[t]:>8.1f}"
          f"  {actual_common[t]:>8.1f}")

# AWBE
print("\n" + "─" * 70)
print(f"AWBE — Adaptive Weighted Blending Ensemble  (λ={best_lambda}, α_lasso=0.1)")
print("─" * 70)

ALPHA_LASSO = 0.1
awbe_preds      = np.zeros(n)
awbe_coefs      = np.zeros((n, 3))
awbe_intercepts = np.zeros(n)

for t in range(n):
    if t < 3:
        awbe_preds[t]      = awae_preds[t]
        awbe_coefs[t]      = awae_weights[t]
        awbe_intercepts[t] = 0.0
    else:
        k_vals  = np.arange(1, t + 1)[::-1]
        dec_w   = np.exp(-best_lambda * k_vals)
        dec_w  /= dec_w.sum() + 1e-9
        X_hist = members[:t]
        y_hist = actual_common[:t]
        lasso = Lasso(alpha=ALPHA_LASSO, fit_intercept=True, max_iter=5000, tol=1e-4)
        lasso.fit(X_hist, y_hist, sample_weight=dec_w)
        awbe_coefs[t]      = lasso.coef_
        awbe_intercepts[t] = lasso.intercept_
        awbe_preds[t]      = lasso.intercept_ + (members[t] * lasso.coef_).sum()

adw_results["AWBE"] = get_report(actual_common, awbe_preds, "AWBE")

# ── COMPARISON TABLE ──────────────────────────────────────────
print("\n" + "=" * 80)
print("COMPARISON — ADEL / AWBE vs ERF MEMBERS vs SOLO MODELS")
print("=" * 80)

adw_comp = {
    "ARIMA (solo)"           : get_report(actual_common, arima_common,   "ARIMA"),
    "Prophet (solo)"         : get_report(actual_common, prophet_common, "Prophet"),
    "LSTM (solo)"            : get_report(actual_common, lstm_common,    "LSTM"),
    "ERF[LSTM+ARIMA]"        : get_report(actual_common, erf_la_pred,    "ERF-LA"),
    "ERF[LSTM+Prophet]"      : get_report(actual_common, erf_lp_pred,    "ERF-LP"),
    "ERF[ARIMA+Prophet]"     : get_report(actual_common, erf_ap_pred,    "ERF-AP"),
    "ERF original (3-model)" : get_report(actual_common, erf_pred,       "ERF-3"),
    "AWAE (Tsang 2024)"      : get_report(actual_common, awae_preds,     "AWAE"),
    "AWBE (Tsang 2024)"      : get_report(actual_common, awbe_preds,     "AWBE"),
}

adw_df     = pd.DataFrame([{"Method": k, **v} for k, v in adw_comp.items()])
best_mae_d = adw_df["mae"].min()
best_r2_d  = adw_df["r2"].max()

W_m, W_v = 26, 10
print(f"\n  {'Method':<{W_m}} {'R²':>{W_v}} {'MAE(n)':>{W_v}} {'RMSE(n)':>{W_v}} {'sMAPE%':>{W_v}} {'MAPE%':>{W_v}}")
print(f"  {'-' * (W_m + W_v * 5 + 4)}")
for _, r in adw_df.iterrows():
    tag = ""
    if abs(r["mae"] - best_mae_d) < 1e-9: tag = " ◄ BEST MAE"
    if abs(r["r2"]  - best_r2_d)  < 1e-9: tag = " ◄ BEST R²"
    print(f"  {r['Method']:<{W_m}}"
          f" {r['r2']:>{W_v}.4f}"
          f" {r['mae']:>{W_v}.4f}"
          f" {r['rmse']:>{W_v}.4f}"
          f" {r['smape']:>{W_v}.4f}"
          f" {r['mape']:>{W_v}.4f}{tag}")

print(f"\n  ◄ Best R²       : {adw_df.loc[adw_df['r2'].idxmax(), 'Method']}")
print(f"  ◄ Best MAE(norm): {adw_df.loc[adw_df['mae'].idxmin(), 'Method']}")

# ── VISUALIZATION ─────────────────────────────────────────────
fig, axes = plt.subplots(3, 1, figsize=(16, 20))
fig.suptitle(
    "Adaptive Dynamic Weight Ensemble  [Tsang et al., Nat. Commun., 2024]\n"
    "DOI: https://doi.org/10.1038/s41467-024-52504-1\n"
    f"Members: ERF[LSTM+ARIMA], ERF[LSTM+Prophet], ERF[ARIMA+Prophet]   λ={best_lambda}",
    fontsize=13, fontweight="bold", y=0.99
)

ax = axes[0]
ax.plot(x_dates, actual_common, color="steelblue",  lw=2.5, marker="o", ms=6, label="Actual")
ax.plot(x_dates, erf_la_pred,   color="gray",       lw=1.5, ls=":", marker="s", ms=4,
        label=f"ERF[LSTM+ARIMA]    MAE={adw_comp['ERF[LSTM+ARIMA]']['mae']:.4f}")
ax.plot(x_dates, erf_lp_pred,   color="sandybrown", lw=1.5, ls=":", marker="^", ms=4,
        label=f"ERF[LSTM+Prophet]  MAE={adw_comp['ERF[LSTM+Prophet]']['mae']:.4f}")
ax.plot(x_dates, erf_ap_pred,   color="lightcoral", lw=1.5, ls=":", marker="v", ms=4,
        label=f"ERF[ARIMA+Prophet] MAE={adw_comp['ERF[ARIMA+Prophet]']['mae']:.4f}")
ax.plot(x_dates, awae_preds,    color="crimson",    lw=2.5, ls="-", marker="*", ms=8,
        label=f"ADEL               MAE={adw_results['AWAE']['mae']:.4f}")
ax.set_title(f"ADEL: Exponential-Decay RMSE Weighting  (λ={best_lambda})",
             fontsize=11, fontweight="bold")
ax.set_ylabel("Total Defaults"); ax.grid(True, alpha=0.3); ax.legend(fontsize=9)
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m"))
plt.setp(ax.xaxis.get_majorticklabels(), rotation=45)

ax = axes[1]
ax.plot(x_dates, actual_common, color="steelblue",  lw=2.5, marker="o", ms=6, label="Actual")
ax.plot(x_dates, erf_la_pred,   color="gray",       lw=1.5, ls=":", marker="s", ms=4,
        label=f"ERF[LSTM+ARIMA]    MAE={adw_comp['ERF[LSTM+ARIMA]']['mae']:.4f}")
ax.plot(x_dates, erf_lp_pred,   color="sandybrown", lw=1.5, ls=":", marker="^", ms=4,
        label=f"ERF[LSTM+Prophet]  MAE={adw_comp['ERF[LSTM+Prophet]']['mae']:.4f}")
ax.plot(x_dates, erf_ap_pred,   color="lightcoral", lw=1.5, ls=":", marker="v", ms=4,
        label=f"ERF[ARIMA+Prophet] MAE={adw_comp['ERF[ARIMA+Prophet]']['mae']:.4f}")
ax.plot(x_dates, awbe_preds,    color="darkgreen",  lw=2.5, ls="-", marker="*", ms=8,
        label=f"AWBE (LASSO)       MAE={adw_results['AWBE']['mae']:.4f}")
ax.set_title(f"AWBE: Exponential-Decay LASSO Blending  (λ={best_lambda}, α={ALPHA_LASSO})",
             fontsize=11, fontweight="bold")
ax.set_ylabel("Total Defaults"); ax.grid(True, alpha=0.3); ax.legend(fontsize=9)
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m"))
plt.setp(ax.xaxis.get_majorticklabels(), rotation=45)

best_erf_mem_name = adw_df[adw_df["Method"].str.startswith("ERF")].loc[
    adw_df[adw_df["Method"].str.startswith("ERF")]["mae"].idxmin(), "Method"
]
best_erf_mem_pred = {
    "ERF[LSTM+ARIMA]"        : erf_la_pred,
    "ERF[LSTM+Prophet]"      : erf_lp_pred,
    "ERF[ARIMA+Prophet]"     : erf_ap_pred,
    "ERF original (3-model)" : erf_pred,
}.get(best_erf_mem_name, erf_la_pred)

ax = axes[2]
ax.plot(x_dates, actual_common,     color="steelblue", lw=2.5, marker="o", ms=6, label="Actual")
ax.plot(x_dates, best_erf_mem_pred, color="purple",    lw=2,   ls="--", marker="D", ms=5,
        label=f"Best ERF: {best_erf_mem_name}  MAE={adw_comp[best_erf_mem_name]['mae']:.4f}")
ax.plot(x_dates, awae_preds,        color="crimson",   lw=2,   ls="-.", marker="^", ms=6,
        label=f"AWAE  MAE={adw_results['AWAE']['mae']:.4f}")
ax.plot(x_dates, awbe_preds,        color="darkgreen", lw=2.5, ls="-",  marker="*", ms=8,
        label=f"AWBE  MAE={adw_results['AWBE']['mae']:.4f}")
ax.set_title(f"AWAE vs AWBE vs Best ERF Member ({best_erf_mem_name})",
             fontsize=11, fontweight="bold")
ax.set_ylabel("Total Defaults"); ax.grid(True, alpha=0.3); ax.legend(fontsize=9)
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m"))
plt.setp(ax.xaxis.get_majorticklabels(), rotation=45)

plt.tight_layout(rect=[0, 0, 1, 0.97])
plt.savefig("/kaggle/working/ensemble_adaptive_dynamic.png", dpi=120, bbox_inches="tight")
print("\n  ✓ Saved: ensemble_adaptive_dynamic.png")
plt.show()

fig6, axes6 = plt.subplots(2, 1, figsize=(14, 10))
fig6.suptitle(f"Adaptive Weight Evolution  (λ={best_lambda})",
              fontsize=13, fontweight="bold")

colors_mem = ["crimson", "darkorange", "steelblue"]
linestyles = ["-", "--", "-."]

ax = axes6[0]
for i, (m, c, ls) in enumerate(zip(member_names, colors_mem, linestyles)):
    ax.plot(x_dates, awae_weights[:, i], color=c, lw=2, ls=ls, marker="o", ms=5, label=m)
ax.axhline(1/3, color="gray", ls=":", lw=1, alpha=0.6, label="Equal weight (1/3)")
ax.set_title("ADEL: Per-Step Inverse-RMSE Decay Weights", fontsize=11, fontweight="bold")
ax.set_ylabel("Weight"); ax.set_ylim(0, 1)
ax.grid(True, alpha=0.3); ax.legend(fontsize=9)
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m"))
plt.setp(ax.xaxis.get_majorticklabels(), rotation=45)

ax = axes6[1]
for i, (m, c, ls) in enumerate(zip(member_names, colors_mem, linestyles)):
    ax.plot(x_dates, awbe_coefs[:, i], color=c, lw=2, ls=ls, marker="s", ms=5, label=m)
ax.axhline(0, color="black", lw=0.8, ls="--")
ax.set_title(f"AWBE: Per-Step LASSO Coefficients  (α={ALPHA_LASSO})",
             fontsize=11, fontweight="bold")
ax.set_ylabel("LASSO Coefficient")
ax.grid(True, alpha=0.3); ax.legend(fontsize=9)
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m"))
plt.setp(ax.xaxis.get_majorticklabels(), rotation=45)

plt.tight_layout()
plt.savefig("/kaggle/working/ensemble_adaptive_weights.png", dpi=120, bbox_inches="tight")
print("  ✓ Saved: ensemble_adaptive_weights.png")
plt.show()

adw_df.to_csv("/kaggle/working/ensemble_adaptive_comparison.csv", index=False)

pd.DataFrame({
    "month"              : [str(m) for m in common_test_months],
    "actual"             : actual_common,
    "erf_lstm_arima"     : erf_la_pred,
    "erf_lstm_prophet"   : erf_lp_pred,
    "erf_arima_prophet"  : erf_ap_pred,
    "awae_pred"          : awae_preds,
    "awbe_pred"          : awbe_preds,
    "awae_w_erf_la"      : awae_weights[:, 0],
    "awae_w_erf_lp"      : awae_weights[:, 1],
    "awae_w_erf_ap"      : awae_weights[:, 2],
    "awbe_coef_erf_la"   : awbe_coefs[:, 0],
    "awbe_coef_erf_lp"   : awbe_coefs[:, 1],
    "awbe_coef_erf_ap"   : awbe_coefs[:, 2],
    "awbe_intercept"     : awbe_intercepts,
}).to_csv("/kaggle/working/ensemble_adaptive_predictions.csv", index=False)

print("  ✓ Saved: ensemble_adaptive_comparison.csv")
print("  ✓ Saved: ensemble_adaptive_predictions.csv")
print("\n✓ Adaptive dynamic weight ensemble section complete!")
print("\n" + "=" * 80)
print("✓ FULL PIPELINE COMPLETE")
print("=" * 80)

In [ ]:
# ══════════════════════════════════════════════════════════════
# SHAP EXPLAINABILITY — LSTM BASE MODEL (FIXED VERSION)
# ══════════════════════════════════════════════════════════════

import shap
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

print(f"\n✓ shap version: {shap.__version__}")

# ── Config ─────────────────────────────────────────────────────
N_BG   = 100
N_EVAL = 3000   # reduced for stability (IMPORTANT FIX)
TOP_N  = 10
SHAP_SEED = 42

OUTPUT_DIR = '/kaggle/working'

rng_shap = np.random.default_rng(SHAP_SEED)

# ── Pre-flight checks ─────────────────────────────────────────
print("\n--- SHAP: Pre-flight checks ---")
print(f"  X_train_lstm shape : {X_train_lstm.shape}")
print(f"  X_test_lstm  shape : {X_test_lstm.shape}")
print(f"  Features           : {len(lstm_num_cols)}")

n_features = X_train_lstm.shape[-1]

assert n_features == len(lstm_num_cols), (
    f"Feature mismatch: {n_features} vs {len(lstm_num_cols)}"
)

# ── Background sampling (safe) ────────────────────────────────
bg_idx = rng_shap.choice(
    len(X_train_lstm),
    size=min(N_BG, len(X_train_lstm)),
    replace=False
)

X_bg = X_train_lstm[bg_idx]

# ── Evaluation subset (IMPORTANT FOR SPEED) ───────────────────
X_eval = X_test_lstm[:N_EVAL]

print(f"  Background shape : {X_bg.shape}")
print(f"  Eval shape       : {X_eval.shape}")

# ── SHAP Gradient Explainer ───────────────────────────────────
print("\n--- SHAP: Building GradientExplainer ---")
explainer = shap.GradientExplainer(model_lstm, X_bg)
shap_raw = explainer.shap_values(X_eval)

# ── Normalize SHAP output ─────────────────────────────────────
if isinstance(shap_raw, list):
    shap_raw = shap_raw[0]

shap_values = np.asarray(shap_raw)
shap_values = np.squeeze(shap_values)

# if still 3D → flatten time dimension
if shap_values.ndim == 3:
    shap_values = shap_values.reshape(shap_values.shape[0], -1)

print(f"  Final SHAP shape: {shap_values.shape}")

assert shap_values.shape[1] == n_features

# ── Global importance ─────────────────────────────────────────
mean_shap = np.abs(shap_values).mean(axis=0)

feat_imp = pd.Series(mean_shap, index=lstm_num_cols)
feat_imp = feat_imp.sort_values(ascending=False)

top10 = feat_imp.head(TOP_N)

print("\nTop features:")
print(top10)

# ── Save outputs ──────────────────────────────────────────────
summary_df = feat_imp.reset_index()
summary_df.columns = ["feature", "mean_abs_shap"]
summary_df.insert(0, "rank", range(1, len(summary_df) + 1))

summary_df.to_csv(f"{OUTPUT_DIR}/shap_summary.csv", index=False)
np.save(f"{OUTPUT_DIR}/shap_values.npy", shap_values)

print("\n✓ Saved SHAP outputs")

# ══════════════════════════════════════════════════════════════
# BAR PLOT — FEATURE IMPORTANCE
# ══════════════════════════════════════════════════════════════

top_colors = (["#1B4F8A"] + ["#85B7EB"] * (TOP_N - 1))[::-1]

fig, ax = plt.subplots(figsize=(9, 5))
ax.barh(
    top10.index[::-1],
    top10.values[::-1],
    color=top_colors
)

ax.set_title("SHAP Feature Importance (LSTM)")
ax.set_xlabel("Mean |SHAP value|")

plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/shap_importance.png", dpi=150)
plt.show()

# ══════════════════════════════════════════════════════════════
# BEESWARM PLOT
# ══════════════════════════════════════════════════════════════

X_eval_2d = X_eval[:, 0, :]   # FIX for (N,1,F)

plt.figure(figsize=(9, 5))
shap.summary_plot(
    shap_values,
    X_eval_2d,
    feature_names=lstm_num_cols,
    max_display=TOP_N,
    show=False
)

plt.title("SHAP Beeswarm (LSTM)")
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/shap_beeswarm.png", dpi=150)
plt.show()

print("\n✓ SHAP analysis complete")

# ══════════════════════════════════════════════════════════════
# ADEL DYNAMIC WEIGHT STACKED AREA CHART
# Professional academic palette — light background, muted fills,
# clean typography.  Inspired by Nature/Econometrics journal style.
# ══════════════════════════════════════════════════════════════
 
ADEL_FILL   = ["#4878CF", "#6ACC65", "#D65F5F"]   # muted primary triad
ADEL_LINE   = ["#2C5F9E", "#3E8A3B", "#A83030"]   # darker variants for edges
BG          = "#F8F9FA"
GRID_COL    = "#E2E4E8"
SPINE_COL   = "#CCCED4"
TEXT_DARK   = "#1A1A2A"
TEXT_MED    = "#555566"
REFLINE_COL = "#9DA0AB"

 
fig, ax = plt.subplots(figsize=(14, 6))
fig.patch.set_facecolor(BG)
ax.set_facecolor(BG)
 
# ── stacked area (semi-transparent fills) ────────────────────
ax.stackplot(
    x_dates,
    awae_weights.T,
    labels=member_names,
    colors=ADEL_FILL,
    alpha=0.55,
    zorder=2,
)
 
# ── crisp boundary lines at each cumulative layer ────────────
cumulative = np.zeros(len(x_dates))
for layer_w, lc in zip(awae_weights.T, ADEL_LINE):
    cumulative = cumulative + layer_w
    ax.plot(x_dates, cumulative,
            color=lc, lw=1.8, alpha=1.0, zorder=4)
 
# ── equal-weight reference lines (1/3, 2/3) ──────────────────
for ref in (1/3, 2/3):
    ax.axhline(ref, color=REFLINE_COL, lw=0.9, ls=(0, (5, 4)),
               alpha=0.9, zorder=3)
 
# Fix 2: place labels inside axes using axes-fraction x, not data x
ax.text(0.975, 1/3 + 0.03, "Equal  1/3",
        transform=ax.get_yaxis_transform(),
        ha="right", va="bottom", fontsize=8,
        color=REFLINE_COL, style="italic")
ax.text(0.975, 2/3 + 0.03, "Equal  2/3",
        transform=ax.get_yaxis_transform(),
        ha="right", va="bottom", fontsize=8,
        color=REFLINE_COL, style="italic")
 
# ── per-step dominant-weight label ───────────────────────────
# Show label at every step UNLESS the weight is ~equal (≈0.333)
# and the previous step was also equal — deduplicate only the flat phase.
cumsum_w     = awae_weights.cumsum(axis=1)
EQUAL_THRESH = 0.36   # weights ≤ this are "basically equal"
prev_was_equal = False
for step_i, (d, w) in enumerate(zip(x_dates, awae_weights)):
    dom       = int(np.argmax(w))
    is_equal  = w[dom] <= EQUAL_THRESH
    if is_equal and prev_was_equal:   # suppress all but first equal step
        prev_was_equal = True
        continue
    y_top    = cumsum_w[step_i, dom]
    y_bottom = cumsum_w[step_i, dom - 1] if dom > 0 else 0.0
    y_mid    = (y_top + y_bottom) / 2.0
    ax.text(
        d, y_mid, f"{w[dom]:.2f}",
        ha="center", va="center",
        fontsize=7.5, color=ADEL_LINE[dom],
        fontweight="600", zorder=5,
    )
    prev_was_equal = is_equal
 
# ── titles & axis labels ─────────────────────────────────────
ax.set_title(
    "ADEL — Adaptive Dynamic Weight Composition",
    fontsize=14, fontweight="bold", color=TEXT_DARK,
    loc="left", pad=12,
)
ax.set_title(
    "Exponential-decay inverse-RMSE weighting  ·  Tsang et al., Nat. Commun. 2024",
    fontsize=9, color=TEXT_MED, loc="right", pad=12,
)
ax.set_ylabel("Ensemble weight  (sums to 1.0)",
              fontsize=10, color=TEXT_MED, labelpad=8)
ax.set_ylim(0, 1.05)
 
# Fix 3: add one-month padding on each side so edge data isn't clipped
from datetime import timedelta
_pad = timedelta(days=15)
ax.set_xlim(x_dates[0] - _pad, x_dates[-1] + _pad)
 
# ── spines ───────────────────────────────────────────────────
for side, spine in ax.spines.items():
    if side in ("top", "right"):
        spine.set_visible(False)
    else:
        spine.set_color(SPINE_COL)
        spine.set_linewidth(0.8)
 
# ── ticks ────────────────────────────────────────────────────
ax.tick_params(axis="both", colors=TEXT_MED, labelsize=9, length=3)
ax.xaxis.set_major_formatter(mdates.DateFormatter("%b %Y"))
ax.xaxis.set_major_locator(mdates.MonthLocator(interval=1))
plt.setp(ax.xaxis.get_majorticklabels(), rotation=40, ha="right",
         color=TEXT_MED, fontsize=8.5)
 
# ── horizontal grid (y-axis only) ────────────────────────────
ax.yaxis.set_major_locator(plt.MultipleLocator(0.1))
ax.yaxis.set_minor_locator(plt.MultipleLocator(0.05))
ax.grid(axis="y", which="major", color=GRID_COL, lw=0.8, zorder=1)
ax.grid(axis="y", which="minor", color=GRID_COL, lw=0.4,
        linestyle=":", zorder=1)
 
# ── λ footnote — one line, below tick labels, above legend ───
fig.text(
    0.01, 0.03,
    f"λ = {best_lambda}  (LOO-RMSE optimised)  ·  Members: {', '.join(member_names)}",
    ha="left", va="bottom",
    fontsize=7.5, color=TEXT_MED, style="italic",
)
 
# ── legend ───────────────────────────────────────────────────
ax.legend(
    loc="upper center",
    bbox_to_anchor=(0.5, -0.18),
    ncol=3,
    fontsize=9,
    frameon=True,
    framealpha=0.9,
    edgecolor=SPINE_COL,
    facecolor=BG,
    labelcolor=TEXT_DARK,
    handlelength=1.6,
    handleheight=0.9,
)
 
plt.tight_layout(rect=[0, 0.14, 1, 1])
adel_path = os.path.join(OUTPUT_DIR, 'ensemble_adel_stacked_weights.png')
plt.savefig(adel_path, dpi=150, bbox_inches='tight',
            facecolor=fig.get_facecolor())
print(f"\n  ✓ Saved: {adel_path}")
plt.show()  



In [ ]:
from scipy import stats

def diebold_mariano(actual, pred1, pred2, h=1):
    """
    Diebold-Mariano test: H0 = equal predictive accuracy.
    h = forecast horizon (1 for one-step-ahead).
    Returns: DM statistic, p-value (two-tailed), interpretation.
    """
    e1 = actual - pred1   # errors of model 1 (comparator)
    e2 = actual - pred2   # errors of model 2 (ADEL)
    
    # Loss differential: squared error difference
    d  = e1**2 - e2**2
    
    n  = len(d)
    d_bar = d.mean()
    
    # Harvey, Leybourne & Newbold (1997) small-sample correction
    gamma0 = np.var(d, ddof=1)
    
    # Newey-West variance estimate for h-step ahead
    gamma = [np.cov(d[i:], d[:-i])[0,1] if i > 0 else gamma0 
             for i in range(h)]
    var_d  = (gamma0 + 2 * sum(gamma[1:])) / n
    
    DM = d_bar / np.sqrt(var_d + 1e-9)
    
    # HLN small-sample correction
    HLN_correction = np.sqrt((n + 1 - 2*h + h*(h-1)/n) / n)
    DM_corrected   = DM * HLN_correction
    
    # t-distribution with n-1 degrees of freedom (HLN correction)
    p_value = 2 * (1 - stats.t.cdf(abs(DM_corrected), df=n-1))
    
    # One-tailed p (H1: ADEL better than comparator)
    p_one_tailed = p_value / 2
    
    return DM_corrected, p_one_tailed


# ── Run DM test: ADEL vs all comparators ─────────────────────
print("\n══ Diebold-Mariano Test: ADEL vs All Comparators ══")
print(f"  n = {n} monthly observations, h = 1 (one-step-ahead)")
print(f"  H₁: ADEL achieves lower squared forecast error than comparator")
print(f"  Critical value: t({n-1} df) at α=0.05 one-tailed ≈ "
      f"{stats.t.ppf(0.95, df=n-1):.4f}\n")

OUTPUT_DIR = '/kaggle/working/'

dm_comparators = {
    "ARIMA"                 : arima_common,
    "Prophet"               : prophet_common,
    "LSTM"                  : lstm_common,
    "ERF[LSTM+ARIMA]"       : erf_la_pred,
    "ERF[LSTM+Prophet]"     : erf_lp_pred,
    "ERF[ARIMA+Prophet]"    : erf_ap_pred,
    "ERF (3-model combined)": erf_pred,
}

dm_rows = []
print(f"  {'Comparator':<28} {'DM stat':>9} {'p-value':>10} {'Sig?':>6}")
print(f"  {'─'*56}")

for name, pred in dm_comparators.items():
    dm_stat, p_val = diebold_mariano(actual_common, pred, awae_preds)
    sig = "Yes *" if p_val < 0.05 else ("Marginal" if p_val < 0.10 else "No")
    print(f"  {name:<28} {dm_stat:>9.4f} {p_val:>10.4f} {sig:>8}")
    dm_rows.append({
        "Comparator"    : name,
        "DM_statistic"  : round(dm_stat, 4),
        "p_value_1tail" : round(p_val,   4),
        "Significant"   : sig
    })

dm_df = pd.DataFrame(dm_rows)
dm_df.to_csv(f'{OUTPUT_DIR}table11_diebold_mariano.csv', index=False)
print(f"\n  ✓ Saved: table11_diebold_mariano.csv")

In [ ]:
def bootstrap_rmse_diff(actual, pred_adel, pred_comp, 
                         n_boot=10000, seed=42):
    """
    Bootstrap 95% CI on RMSE(comparator) - RMSE(ADEL).
    Positive CI entirely above zero = ADEL significantly better.
    """
    rng  = np.random.default_rng(seed)
    n    = len(actual)
    diffs = []
    
    for _ in range(n_boot):
        idx  = rng.integers(0, n, size=n)
        rmse_comp = np.sqrt(np.mean((actual[idx] - pred_comp[idx])**2))
        rmse_adel = np.sqrt(np.mean((actual[idx] - pred_adel[idx])**2))
        diffs.append(rmse_comp - rmse_adel)
    
    diffs  = np.array(diffs)
    ci_low = np.percentile(diffs, 2.5)
    ci_hi  = np.percentile(diffs, 97.5)
    obs_diff = (np.sqrt(np.mean((actual - pred_comp)**2)) - 
                np.sqrt(np.mean((actual - awae_preds)**2)))
    p_val  = np.mean(diffs <= 0)   # proportion of bootstrap where ADEL not better
    
    return obs_diff, ci_low, ci_hi, p_val


# ── Run bootstrap ─────────────────────────────────────────────
print("\n══ Bootstrap CI: RMSE(comparator) − RMSE(ADEL) ══")
print(f"  n_bootstrap = 10,000  |  95% CI (percentile method)")
print(f"  Positive CI entirely above 0 = ADEL significantly better\n")

boot_rows = []
print(f"  {'Comparator':<28} {'Obs Diff':>10} {'CI Low':>9} {'CI High':>9} {'p':>8} {'Sig?':>6}")
print(f"  {'─'*72}")

for name, pred in dm_comparators.items():
    obs, lo, hi, p = bootstrap_rmse_diff(actual_common, awae_preds, pred)
    sig = "Yes *" if lo > 0 else ("Marginal" if p < 0.10 else "No")
    print(f"  {name:<28} {obs:>10.4f} {lo:>9.4f} {hi:>9.4f} {p:>8.4f} {sig:>6}")
    boot_rows.append({
        "Comparator"     : name,
        "Observed_Diff"  : round(obs, 4),
        "CI_Lower_95"    : round(lo,  4),
        "CI_Upper_95"    : round(hi,  4),
        "p_value"        : round(p,   4),
        "Significant"    : sig
    })

boot_df = pd.DataFrame(boot_rows)
boot_df.to_csv(f'{OUTPUT_DIR}table11_bootstrap_ci.csv', index=False)
print(f"\n  ✓ Saved: table11_bootstrap_ci.csv")

In [ ]:
# ══════════════════════════════════════════════════════════════
# BOOTSTRAP CI + STATISTICAL POWER ANALYSIS (combined)
# ══════════════════════════════════════════════════════════════
from scipy import stats

N_BOOT = 10_000
RNG    = np.random.default_rng(SEED)
n_obs  = len(actual_common)
adel_pred = awae_preds

# ── comparators ───────────────────────────────────────────────
comparators = {
    "ARIMA (solo)"           : arima_common,
    "Prophet (solo)"         : prophet_common,
    "LSTM (solo)"            : lstm_common,
    "ERF[LSTM+ARIMA]"        : erf_la_pred,
    "ERF[LSTM+Prophet]"      : erf_lp_pred,
    "ERF[ARIMA+Prophet]"     : erf_ap_pred,
    "ERF (3-model combined)" : erf_pred,
    "AWBE (Tsang 2024)"      : awbe_preds,
}

groups = {
    "ARIMA (solo)"           : "solo",
    "Prophet (solo)"         : "solo",
    "LSTM (solo)"            : "solo",
    "ERF[LSTM+ARIMA]"        : "erf",
    "ERF[LSTM+Prophet]"      : "erf",
    "ERF[ARIMA+Prophet]"     : "erf",
    "ERF (3-model combined)" : "erf",
    "AWBE (Tsang 2024)"      : "adaptive",
}

# ── bootstrap loop ────────────────────────────────────────────
boot_results = {}

for name, comp_pred in comparators.items():
    obs_rmse_comp = np.sqrt(np.mean((actual_common - comp_pred) ** 2))
    obs_rmse_adel = np.sqrt(np.mean((actual_common - adel_pred) ** 2))
    obs_diff      = obs_rmse_comp - obs_rmse_adel

    boot_diffs = np.empty(N_BOOT)
    for b in range(N_BOOT):
        idx           = RNG.integers(0, n_obs, size=n_obs)
        b_actual      = actual_common[idx]
        b_comp        = comp_pred[idx]
        b_adel        = adel_pred[idx]
        boot_diffs[b] = (np.sqrt(np.mean((b_actual - b_comp) ** 2)) -
                         np.sqrt(np.mean((b_actual - b_adel) ** 2)))

    ci_lo = np.percentile(boot_diffs, 2.5)
    ci_hi = np.percentile(boot_diffs, 97.5)
    p_val = np.mean(boot_diffs <= 0)

    boot_results[name] = {
        "obs"      : obs_diff,
        "lo"       : ci_lo,
        "hi"       : ci_hi,
        "p"        : p_val,
        "group"    : groups[name],
        "comp_pred": comp_pred,
    }

# ── print bootstrap table ─────────────────────────────────────
print("\n══ Bootstrap CI: RMSE(comparator) − RMSE(ADEL) ══")
print(f"  n_bootstrap = {N_BOOT:,}  |  95% CI (percentile method)")
print(f"  Reference model: ADEL (Tsang 2024)")
print(f"  Positive diff = ADEL is better\n")

GROUP_LABEL = {
    "solo"    : "── Solo models ──",
    "erf"     : "── ERF ensemble variants ──",
    "adaptive": "── Adaptive ensemble ──",
}
W = 28
print(f"  {'Comparator':<{W}} {'Obs Diff':>10} {'CI Low':>9} {'CI High':>9} {'p':>8}   Sig?")
print(f"  {'─'*74}")

current_group = None
for name, r in boot_results.items():
    if r["group"] != current_group:
        current_group = r["group"]
        print(f"\n  {GROUP_LABEL[current_group]}")

    star = ("Yes ***" if r["p"] < 0.001 else
            "Yes *"   if r["p"] < 0.05  else
            "No")
    print(f"  {name:<{W}} {r['obs']:>10.4f} {r['lo']:>9.4f} {r['hi']:>9.4f} "
          f"{r['p']:>8.4f}   {star}")

print(f"\n  {'─'*74}")

# ── statistical power analysis ────────────────────────────────
print("\n══ Statistical Power Analysis ══")
print("  How many months needed to detect ADEL vs comparator difference?\n")

adel_rmse = np.sqrt(np.mean((actual_common - adel_pred) ** 2))
print(f"  ADEL RMSE (reference) : {adel_rmse:.4f}")
print(f"  Current test set      : n = {n_obs} months\n")

print(f"  {'Comparator':<{W}} {'RMSE diff':>10} {'Effect d':>10} "
      f"{'n (80%)':>10} {'n (90%)':>10} {'Achievable?':>12}")
print(f"  {'─'*82}")

current_group = None
for name, r in boot_results.items():
    if r["group"] != current_group:
        current_group = r["group"]
        print(f"\n  {GROUP_LABEL[current_group]}")

    comp_pred = r["comp_pred"]
    comp_rmse = np.sqrt(np.mean((actual_common - comp_pred) ** 2))
    obs_diff  = comp_rmse - adel_rmse

    # bootstrap std of the difference as denominator for effect size
    rng_power = np.random.default_rng(42)
    boot_std  = np.std([
        np.sqrt(np.mean((actual_common[idx] - comp_pred[idx]) ** 2)) -
        np.sqrt(np.mean((actual_common[idx] - adel_pred[idx]) ** 2))
        for idx in (rng_power.integers(0, n_obs, n_obs) for _ in range(2_000))
    ])

    effect_d = obs_diff / (boot_std + 1e-9)

    if abs(effect_d) > 0.05:
        # n = (z_alpha/2 + z_beta)^2 / d^2
        n_80 = int(np.ceil((stats.norm.ppf(0.975) + stats.norm.ppf(0.80)) ** 2
                           / effect_d ** 2))
        n_90 = int(np.ceil((stats.norm.ppf(0.975) + stats.norm.ppf(0.90)) ** 2
                           / effect_d ** 2))
    else:
        n_80 = n_90 = 9999

    achievable = "Yes" if n_80 <= n_obs else f"Need n>={n_80}"

    print(f"  {name:<{W}} {obs_diff:>10.4f} {effect_d:>10.3f} "
          f"{n_80:>10} {n_90:>10} {achievable:>12}")

print(f"""
  {'─'*82}
  Conclusion:
  • Solo models: large effect size → detectable at n=12 → Sig=Yes correct
  • ERF variants: small effect size → need far more months → Sig=No expected
  • "No" + positive obs diff = ADEL still better, just unconfirmable at n=12
  • This is a known constraint in financial time-series (Diebold & Mariano 1995)
""")

# ── save results ──────────────────────────────────────────────
boot_df = pd.DataFrame([
    {"comparator": k, "group": v["group"],
     "obs_diff": v["obs"], "ci_low": v["lo"], "ci_high": v["hi"],
     "p_value": v["p"], "significant": v["p"] < 0.05}
    for k, v in boot_results.items()
])
boot_df.to_csv(f'{OUTPUT_DIR}bootstrap_ci_results_all8.csv', index=False)
print(f"  ✓ Saved: bootstrap_ci_results_all8.csv")

In [ ]:
'''
from sklearn.metrics import roc_auc_score
from scipy.stats import chi2

def delong_auc_test(y_true, pred1, pred2):
    """
    DeLong test for comparing two AUC values.
    Returns: z-statistic, p-value (two-tailed).
    Simplified implementation for two correlated ROC curves.
    """
    def auc_variance(y_true, y_pred):
        pos = y_pred[y_true == 1]
        neg = y_pred[y_true == 0]
        auc = roc_auc_score(y_true, y_pred)
        n1, n0 = len(pos), len(neg)
        
        # Placement values
        V10 = np.array([np.mean(p > neg) + 0.5*np.mean(p == neg) for p in pos])
        V01 = np.array([np.mean(n < pos) + 0.5*np.mean(n == pos) for n in neg])
        
        S10 = np.var(V10, ddof=1) / n1
        S01 = np.var(V01, ddof=1) / n0
        return auc, S10 + S01
    
    auc1, var1 = auc_variance(y_true, pred1)
    auc2, var2 = auc_variance(y_true, pred2)
    
    # Covariance (simplified — assumes independent)
    z     = (auc2 - auc1) / np.sqrt(var1 + var2 + 1e-9)
    p_val = 2 * (1 - stats.norm.cdf(abs(z)))
    return auc1, auc2, z, p_val


# Compare LSTM AUC vs a simple logistic baseline
from sklearn.linear_model import LogisticRegression

print("\n══ DeLong Test: LSTM AUC vs Logistic Baseline ══")

# Fit logistic baseline on same features
lr = LogisticRegression(max_iter=1000, random_state=42)
lr.fit(Xtr_sc, y_tr)
lr_proba = lr.predict_proba(Xte_sc)[:, 1]

auc_lr, auc_lstm, z_stat, p_delong = delong_auc_test(
    y_te, lr_proba, y_test_proba
)
print(f"  Logistic Regression AUC : {auc_lr:.4f}")
print(f"  LSTM AUC                : {auc_lstm:.4f}")
print(f"  DeLong z-statistic      : {z_stat:.4f}")
print(f"  p-value (two-tailed)    : {p_delong:.4f}")
print(f"  Significant at α=0.05?  : {'Yes' if p_delong < 0.05 else 'No'}")
'''

In [ ]:
"""
Table 12 — Simplified Numerical Example of ADEL Weight Computation at t = 3
─────────────────────────────────────────────────────────────────────────────
Drops into the existing pipeline AFTER the AWAE block.
Requires variables already in scope:
  actual_common   – np.ndarray  shape (n,)
  members         – np.ndarray  shape (n, 3)   [erf_la, erf_lp, erf_ap]
  member_names    – list[str]   len 3
  awae_weights    – np.ndarray  shape (n, 3)   per-step AWAE weights
  awae_preds      – np.ndarray  shape (n,)
  common_test_months – list of pd.Period
  best_lambda     – float       decay rate λ
"""

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

# ══════════════════════════════════════════════════════════════
# TABLE 12 — ADEL WEIGHT COMPUTATION AT t = 3
# ══════════════════════════════════════════════════════════════
print("\n" + "=" * 80)
print("TABLE 12 — SIMPLIFIED NUMERICAL EXAMPLE OF ADEL WEIGHT COMPUTATION")
print("           at t = 3  (September 2019)")
print("=" * 80)

T = 2            # 0-indexed → t=3 is index 2
t_label  = str(common_test_months[T])          # e.g. "2019-09"
t_actual = actual_common[T]

# ── Step 1: Decay-weighted squared errors up to t-1 ──────────
k_vals  = np.arange(1, T + 1)[::-1]           # [2, 1]  (most-recent = 1)
dec_w   = np.exp(-best_lambda * k_vals)
dec_w  /= dec_w.sum()                          # normalise

errs_sq = (members[:T] - actual_common[:T].reshape(-1, 1)) ** 2   # shape (T, 3)

# ── Step 2: Decay-weighted RMSE per member ───────────────────
S = np.sqrt((dec_w.reshape(-1, 1) * errs_sq).sum(axis=0))         # shape (3,)

# ── Step 3: Inverse scores ───────────────────────────────────
inv_S = 1.0 / (S + 1e-9)

# ── Step 4: Normalised weights ───────────────────────────────
w_norm = inv_S / inv_S.sum()

# ── Step 5: Ensemble prediction ──────────────────────────────
pred_t3 = (members[T] * w_norm).sum()

# ── Rank labels ───────────────────────────────────────────────
ranks    = S.argsort()           # ascending error → best first
rank_lbl = [""] * 3
rank_lbl[ranks[0]] = "lowest"
rank_lbl[ranks[1]] = "middle"
rank_lbl[ranks[2]] = "highest"

dominant_idx = w_norm.argmax()

# ── Short aliases for display ─────────────────────────────────
short_names = {
    "ERF[LSTM+ARIMA]"   : "ERF[LSTM+ARIMA] (w₁)",
    "ERF[LSTM+Prophet]" : "ERF[LSTM+Prophet] (w₂)",
    "ERF[ARIMA+Prophet]": "ERF[ARIMA+Prophet] (w₃)",
}

# ═══════════════════════════════════════════════════════════════
# PRINT: Console table
# ═══════════════════════════════════════════════════════════════
W_n, W_s, W_i, W_w, W_p = 30, 20, 18, 20, 30

sep = (f"  ├{'─'*W_n}┼{'─'*W_s}┼{'─'*W_i}┼{'─'*W_w}┼{'─'*W_p}┤")
bot = (f"  └{'─'*W_n}┴{'─'*W_s}┴{'─'*W_i}┴{'─'*W_w}┴{'─'*W_p}┘")

print(f"\n  Table 12 · ADEL weight computation at t = 3  ({t_label})")
print(f"  λ (decay rate) = {best_lambda}  |  Decay weights used = {np.round(dec_w, 4).tolist()}")
print(f"\n  ┌{'─'*W_n}┬{'─'*W_s}┬{'─'*W_i}┬{'─'*W_w}┬{'─'*W_p}┐")
print(f"  │ {'ERF Member':<{W_n-2}} │ {'Decay-Wtd Error Sᵢ(3)':^{W_s-2}} │"
      f" {'Inverse 1/Sᵢ(3)':^{W_i-2}} │ {'Norm. Weight wᵢ(3)':^{W_w-2}} │"
      f" {'Interpretation':^{W_p-2}} │")
print(sep)

for i, name in enumerate(member_names):
    disp_name   = short_names.get(name, name)
    dom_tag     = "  ◄ Dominant" if i == dominant_idx else ""
    interp      = (
        f"Lowest error → Highest weight{dom_tag}"
        if rank_lbl[i] == "lowest"
        else f"{rank_lbl[i].capitalize()} error → {'Middle' if rank_lbl[i]=='middle' else 'Lowest'} weight"
    )
    print(f"  │ {disp_name:<{W_n-2}} │"
          f" {S[i]:>{W_s-4}.4f}  ({rank_lbl[i]:<7}) │"
          f" {inv_S[i]:>{W_i-2}.2f} │"
          f" {w_norm[i]:>{W_w-2}.2f} │"
          f" {interp:<{W_p-2}} │")
    print(sep if i < 2 else bot)

print(f"  │ {'Sum':<{W_n-2}} │ {'':^{W_s-2}} │"
      f" {inv_S.sum():^{W_i-2}.2f} │"
      f" {w_norm.sum():^{W_w-2}.2f} │"
      f" {'':^{W_p-2}} │")
print(f"  └{'─'*W_n}┴{'─'*W_s}┴{'─'*W_i}┴{'─'*W_w}┴{'─'*W_p}┘")

print(f"""
  Verification
  ─────────────────────────────────────────────────────────────
  Actual defaults at t=3   : {t_actual:,.1f}
  ADEL ensemble prediction : {pred_t3:,.1f}
  Error                    : {t_actual - pred_t3:+,.1f}  ({(t_actual - pred_t3)/t_actual*100:+.2f}%)

  Formula applied
  ─────────────────────────────────────────────────────────────
  Sᵢ(t) = √[ Σₖ exp(−λk) · eᵢ,ₜ₋ₖ² ]     (decay-weighted RMSE)
  wᵢ(t) = [1/Sᵢ(t)] / Σⱼ[1/Sⱼ(t)]         (inverse-score normalisation)
  ŷ(t)  = Σᵢ wᵢ(t) · yᵢ(t)                 (weighted ensemble forecast)
""")

# ═══════════════════════════════════════════════════════════════
# STEP-BY-STEP INTERMEDIATE TABLE  (for thesis appendix)
# ═══════════════════════════════════════════════════════════════
print("=" * 80)
print("STEP-BY-STEP INTERMEDIATE VALUES")
print("=" * 80)

print(f"\n  Step A — Squared errors at each prior time-step:")
print(f"  {'Time step':<12} {'Decay weight':>14}", end="")
for n in member_names:
    print(f" {short_names.get(n, n):>20}", end="")
print()
print(f"  {'-'*80}")

for k_i, (t_step, dw) in enumerate(zip(range(T-1, -1, -1), dec_w[::-1])):
    row_errs = errs_sq[t_step]
    tstep_lbl = str(common_test_months[t_step])
    print(f"  t-{k_i+1:<4} ({tstep_lbl})  {dw:>14.4f}", end="")
    for e in row_errs:
        print(f" {e:>20.2f}", end="")
    print()

print(f"\n  Step B — Weighted squared errors (decay_w × e²):")
for k_i, (t_step, dw) in enumerate(zip(range(T-1, -1, -1), dec_w[::-1])):
    weighted = dw * errs_sq[t_step]
    print(f"  t-{k_i+1:<4}", end="")
    for w in weighted:
        print(f" {w:>20.2f}", end="")
    print()

print(f"\n  Step C — Sum of weighted squared errors:")
sum_w_sq = (dec_w.reshape(-1, 1) * errs_sq).sum(axis=0)
for i, (n, v) in enumerate(zip(member_names, sum_w_sq)):
    print(f"  {short_names.get(n, n):<30}  {v:>10.4f}")

print(f"\n  Step D — Sᵢ(3) = √(sum)  [decay-weighted RMSE]:")
for i, (n, v) in enumerate(zip(member_names, S)):
    print(f"  {short_names.get(n, n):<30}  {v:>10.4f}  ({rank_lbl[i]})")

print(f"\n  Step E — 1/Sᵢ(3)  [inverse score]:")
for i, (n, v) in enumerate(zip(member_names, inv_S)):
    print(f"  {short_names.get(n, n):<30}  {v:>10.4f}")
print(f"  {'Sum':<30}  {inv_S.sum():>10.4f}")

print(f"\n  Step F — wᵢ(3) = [1/Sᵢ(3)] / Σ[1/Sⱼ(3)]  [normalised weight]:")
for i, (n, v) in enumerate(zip(member_names, w_norm)):
    dom = "  ◄ Dominant" if i == dominant_idx else ""
    print(f"  {short_names.get(n, n):<30}  {v:>10.4f}{dom}")
print(f"  {'Sum':<30}  {w_norm.sum():>10.4f}")

# ═══════════════════════════════════════════════════════════════
# SAVE: DataFrame
# ═══════════════════════════════════════════════════════════════
table12_df = pd.DataFrame({
    "ERF_Member"             : [short_names.get(n, n) for n in member_names],
    "Decay_Wtd_Error_Si3"    : np.round(S,      4),
    "Rank"                   : rank_lbl,
    "Inverse_Score_1_Si3"    : np.round(inv_S,  2),
    "Normalized_Weight_wi3"  : np.round(w_norm, 2),
    "Dominant"               : [i == dominant_idx for i in range(3)],
})

table12_df.to_csv("/kaggle/working/table12_adel_weight_t3.csv", index=False)
print("\n  ✓ Saved: table12_adel_weight_t3.csv")

# ═══════════════════════════════════════════════════════════════
# VISUALIZATION — Three-panel figure for Table 12
# ═══════════════════════════════════════════════════════════════
x_dates = [pd.Period(m).to_timestamp() for m in common_test_months]

fig, axes = plt.subplots(1, 3, figsize=(16, 6))
fig.suptitle(
    f"Table 12 — ADEL Weight Computation at t = 3  ({t_label})\n"
    f"λ = {best_lambda}  |  Decay weights = {np.round(dec_w, 3).tolist()}",
    fontsize=13, fontweight="bold"
)

bar_colors = ["#2ca02c", "#d62728", "#ff7f0e"]   # green=dominant, red=highest err, orange=middle
bar_order  = S.argsort()[::-1]                   # highest error → leftmost in bar

short_labels = ["ERF[L+A]\n(w₁)", "ERF[L+P]\n(w₂)", "ERF[A+P]\n(w₃)"]

# Panel 1 — Decay-weighted error scores
ax = axes[0]
bars = ax.bar(short_labels, S, color=bar_colors, edgecolor="white", linewidth=0.8, width=0.5)
for bar, val, rl in zip(bars, S, rank_lbl):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.0005,
            f"{val:.4f}\n({rl})", ha="center", va="bottom", fontsize=10)
ax.set_title("Decay-Weighted Error\nSᵢ(3)", fontsize=11, fontweight="bold")
ax.set_ylabel("Sᵢ(3)"); ax.grid(axis="y", alpha=0.3)
ax.set_ylim(0, S.max() * 1.35)

# Panel 2 — Inverse scores
ax = axes[1]
bars = ax.bar(short_labels, inv_S, color=bar_colors, edgecolor="white", linewidth=0.8, width=0.5)
for bar, val in zip(bars, inv_S):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.5,
            f"{val:.2f}", ha="center", va="bottom", fontsize=11, fontweight="500")
ax.set_title("Inverse Score\n1/Sᵢ(3)", fontsize=11, fontweight="bold")
ax.set_ylabel("1/Sᵢ(3)"); ax.grid(axis="y", alpha=0.3)
ax.set_ylim(0, inv_S.max() * 1.25)
ax.axhline(inv_S.mean(), color="gray", ls="--", lw=1, label=f"Mean = {inv_S.mean():.1f}")
ax.legend(fontsize=9)

# Panel 3 — Normalised weights (pie)
ax = axes[2]
explode = [0.06 if i == dominant_idx else 0 for i in range(3)]
wedges, texts, autotexts = ax.pie(
    w_norm,
    labels=short_labels,
    colors=bar_colors,
    autopct="%1.0f%%",
    explode=explode,
    startangle=90,
    pctdistance=0.70,
    wedgeprops={"edgecolor": "white", "linewidth": 1.5},
    textprops={"fontsize": 10}
)
for at in autotexts:
    at.set_fontsize(12)
    at.set_fontweight("bold")
ax.set_title(f"Normalised Weights wᵢ(3)\n(dominant = {short_labels[dominant_idx].replace(chr(10),' ')})",
             fontsize=11, fontweight="bold")

plt.tight_layout()
plt.savefig("/kaggle/working/table12_adel_weight_t3.png", dpi=120, bbox_inches="tight")
print("  ✓ Saved: table12_adel_weight_t3.png")
plt.show()



# different dataset

In [ ]:
"""
Electric Production Forecasting — 9-Model Script
══════════════════════════════════════════════════
Dataset : Electric_Production.csv
         Monthly electric production index (IPG2211A2N)
         Jan 1985 – Jan 2018  (397 months)

Solo models:
  1. ARIMA   (solo)
  2. Prophet (solo)
  3. LSTM    (solo)

ERF variants  [Santos Júnior et al., Inf. Sci. 2023]:
  4. ERF[LSTM + ARIMA]
  5. ERF[LSTM + Prophet]
  6. ERF[ARIMA + Prophet]
  7. ERF original (3-model)

Adaptive  [Tsang et al., Nat. Commun. 2024]:
  8. ADEL / AWAE  — Exponential-decay RMSE weighting
  9. AWBE         — Exponential-decay LASSO blending

Outputs:
  • electric_9models_comparison.csv
  • electric_9models_predictions.csv
  • electric_9models_chart.png
  • electric_adel_weights.png
"""
!pip install pmdarima

# ─────────────────────────────────────────────────────────────
# 0.  IMPORTS
# ─────────────────────────────────────────────────────────────
import warnings, os, random
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import matplotlib.patches as mpatches

from pmdarima import auto_arima
from prophet import Prophet

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.optimizers import Adam

from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, Lasso
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

SEED = 100
random.seed(SEED); np.random.seed(SEED)
os.environ["PYTHONHASHSEED"] = str(SEED)
tf.random.set_seed(SEED)

DATA_PATH  = '/kaggle/input/datasets/thandarphyo/elecronic-testing-dataset/Electric_Production.csv'
OUTPUT_DIR = '/mnt/user-data/outputs/'
os.makedirs(OUTPUT_DIR, exist_ok=True)


# ─────────────────────────────────────────────────────────────
# HELPERS  (same interface as original LendingClub script)
# ─────────────────────────────────────────────────────────────
def get_report(actual, pred, label=""):
    actual, pred = np.array(actual, dtype=float), np.array(pred, dtype=float)
    mean_act = np.abs(actual).mean() + 1e-9
    r2    = r2_score(actual, pred)
    mae   = mean_absolute_error(actual, pred) / mean_act
    rmse  = np.sqrt(mean_squared_error(actual, pred)) / mean_act
    smape = np.mean(2*np.abs(actual-pred)/(np.abs(actual)+np.abs(pred)+1e-9))*100
    mape  = np.mean(np.abs((actual-pred)/(np.abs(actual)+1e-9)))*100
    if label:
        print(f"  [{label}]  R²={r2:.4f}  MAE(n)={mae:.4f}  RMSE(n)={rmse:.4f}"
              f"  sMAPE={smape:.4f}%  MAPE={mape:.4f}%")
    return dict(r2=r2, mae=mae, rmse=rmse, smape=smape, mape=mape)


def evaluate_model_table(model_name, train_actual, train_pred,
                          test_actual, test_pred):
    W_m, W_v = 22, 12

    def reg_ts(actual, pred):
        actual = np.array(actual, dtype=float)
        pred   = np.array(pred,   dtype=float)
        mean_a = np.abs(actual).mean() + 1e-9
        r2_v   = r2_score(actual, pred)
        mae_v  = mean_absolute_error(actual, pred) / mean_a
        rmse_v = np.sqrt(mean_squared_error(actual, pred)) / mean_a
        sm = np.mean(2*np.abs(actual-pred)/(np.abs(actual)+np.abs(pred)+1e-9))*100
        mp = np.mean(np.abs((actual-pred)/(np.abs(actual)+1e-9)))*100
        return r2_v, mae_v, rmse_v, sm, mp

    tr = reg_ts(train_actual, train_pred)
    te = reg_ts(test_actual,  test_pred)

    title_w = W_m + W_v * 3 + 2
    sep_top = f"  ├{'─'*W_m}┬{'─'*W_v}┬{'─'*W_v}┬{'─'*W_v}┤"
    sep_mid = f"  ├{'─'*W_m}┼{'─'*W_v}┼{'─'*W_v}┼{'─'*W_v}┤"
    sep_bot = f"  └{'─'*W_m}┴{'─'*W_v}┴{'─'*W_v}┴{'─'*W_v}┘"

    def row(name, tv, ev):
        return (f"  │ {name:<{W_m-2}} │ {tv:>{W_v-2}.4f} │"
                f" {ev:>{W_v-2}.4f} │ {tv-ev:>{W_v-2}.4f} │")

    print(f"\n  ┌{'─'*title_w}┐")
    print(f"  │ {(model_name + ' — Evaluation'):<{title_w}} │")
    print(sep_top)
    print(f"  │ {'Metric':<{W_m-2}} │ {'Train':>{W_v-2}} │ {'Test':>{W_v-2}} │"
          f" {'Diff':>{W_v-2}} │")
    print(sep_mid)
    for nm, tv, ev in zip(['R²','MAE (norm)','RMSE (norm)','sMAPE (%)','MAPE (%)'],
                           tr, te):
        print(row(nm, tv, ev))
    print(sep_bot)


def fmt_axis(ax, interval=6):
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
    ax.xaxis.set_major_locator(mdates.MonthLocator(interval=interval))
    plt.setp(ax.xaxis.get_majorticklabels(), rotation=45)


def nadaraya_watson(x_train, y_train, x_test, h=2.0):
    preds = []
    for xt in x_test:
        w = np.exp(-0.5 * ((x_train - xt) / h) ** 2)
        preds.append((w * y_train).sum() / (w.sum() + 1e-9))
    return np.array(preds)


def erf_pair(base_pred, corrector_pred, actual, t_idx):
    residual = actual - base_pred
    corr_cen = corrector_pred - corrector_pred.mean()
    proxy_a  = corr_cen * (residual.std() / (corr_cen.std() + 1e-9))
    proxy_b  = nadaraya_watson(t_idx, residual, t_idx, h=2.0)
    return base_pred + (proxy_a + proxy_b) / 2.0


def make_sequences(data, n_steps=12):
    X, y = [], []
    for i in range(len(data) - n_steps):
        X.append(data[i:i+n_steps])
        y.append(data[i+n_steps])
    return np.array(X), np.array(y)


# ─────────────────────────────────────────────────────────────
# 1.  DATA LOADING
# ─────────────────────────────────────────────────────────────
print("\n══ Step 1: Data Loading ══")

df = pd.read_csv(DATA_PATH)
df = df[['DATE', 'IPG2211A2N']].copy()
df.columns = ['date', 'production']
df['date'] = pd.to_datetime(df['date'])
df = df.sort_values('date').reset_index(drop=True)

# ── fix: drop or fill NaN in production before log transform ──
print(f"  NaN in production: {df['production'].isna().sum()}")
df['production'] = df['production'].interpolate(method='linear')  # fill gaps
df = df.dropna(subset=['production']).reset_index(drop=True)      # drop any remaining
print(f"  After cleaning: {len(df)} rows")

series = df['production'].values.astype(float)
dates  = df['date'].values
n_total = len(series)

print(f"  Series length : {n_total} months")
print(f"  Date range    : {df['date'].iloc[0].strftime('%Y-%m')} → "
      f"{df['date'].iloc[-1].strftime('%Y-%m')}")
print(f"  Value range   : {series.min():.2f} – {series.max():.2f}")


# ─────────────────────────────────────────────────────────────
# 2.  LOG TRANSFORM + CUBIC DETRENDING
# ─────────────────────────────────────────────────────────────
print("\n══ Step 2: Log Transform + Cubic Detrending ══")

log_series = np.log(series)
t  = np.arange(n_total).reshape(-1, 1)
t3 = np.column_stack([t, t**2, t**3])

cubic_reg     = LinearRegression().fit(t3, log_series)
trend_fit     = cubic_reg.predict(t3)
log_detrended = log_series - trend_fit
print(f"  Cubic trend R²: {r2_score(log_series, trend_fit):.4f}")


# ─────────────────────────────────────────────────────────────
# 3.  TRAIN / TEST SPLIT  (last 24 months = test)
# ─────────────────────────────────────────────────────────────
print("\n══ Step 3: Train / Test Split ══")

TEST_SIZE  = 24
TRAIN_SIZE = n_total - TEST_SIZE

train_det  = log_detrended[:TRAIN_SIZE]
test_det   = log_detrended[TRAIN_SIZE:]
trend_tr   = trend_fit[:TRAIN_SIZE]

train_series = series[:TRAIN_SIZE]
test_series  = series[TRAIN_SIZE:]
train_dates  = df['date'].iloc[:TRAIN_SIZE].reset_index(drop=True)
test_dates   = df['date'].iloc[TRAIN_SIZE:].reset_index(drop=True)

t_test_idx  = np.arange(TRAIN_SIZE, n_total).reshape(-1, 1)
t_test_feat = np.column_stack([t_test_idx, t_test_idx**2, t_test_idx**3])
trend_test  = cubic_reg.predict(t_test_feat)

print(f"  Train : {train_dates.iloc[0].strftime('%Y-%m')} → "
      f"{train_dates.iloc[-1].strftime('%Y-%m')} ({TRAIN_SIZE} mo)")
print(f"  Test  : {test_dates.iloc[0].strftime('%Y-%m')} → "
      f"{test_dates.iloc[-1].strftime('%Y-%m')} ({TEST_SIZE} mo)")


# ═════════════════════════════════════════════════════════════
# MODEL 1 — ARIMA (solo)
# ═════════════════════════════════════════════════════════════
print("\n══ Model 1: ARIMA ══")

model_arima = auto_arima(
    train_det, seasonal=True, m=12,
    max_p=4, max_q=4, max_P=2, max_Q=2,
    information_criterion='aic', stepwise=True,
    suppress_warnings=True, error_action='ignore', trace=False
)
print(f"  Best order: SARIMA{model_arima.order}x{model_arima.seasonal_order}")

history_det    = list(train_det)
arima_test_det = []

for i in range(TEST_SIZE):
    try:
        tmp = auto_arima(
            history_det,
            order=model_arima.order,
            seasonal_order=model_arima.seasonal_order,
            suppress_warnings=True, error_action='ignore'
        )
        p = float(tmp.predict(1)[0])
    except Exception:
        p = history_det[-1]
    arima_test_det.append(p)
    history_det.append(test_det[i])

arima_common       = np.exp(np.array(arima_test_det) + trend_test)
arima_train_fitted = np.exp(
    model_arima.predict_in_sample()[:TRAIN_SIZE] + trend_tr
)

m_arima = get_report(test_series, arima_common, "ARIMA (solo)")
evaluate_model_table("ARIMA",
                     train_series, arima_train_fitted,
                     test_series,  arima_common)


# ═════════════════════════════════════════════════════════════
# MODEL 2 — PROPHET (solo)
# ═════════════════════════════════════════════════════════════
print("\n══ Model 2: Prophet ══")

prophet_df = pd.DataFrame({
    'ds': df['date'],
    'y' : np.log(series)
})
prophet_df['momentum_3m'] = prophet_df['y'].diff(3)
prophet_df['momentum_6m'] = prophet_df['y'].diff(6)
prophet_df['yoy_change']  = prophet_df['y'].diff(12)
prophet_df = prophet_df.dropna().reset_index(drop=True)

PROPHET_REGS = ['momentum_3m', 'momentum_6m', 'yoy_change']
start_offset = len(df) - len(prophet_df)   # = 12

p_train_size  = len(prophet_df) - TEST_SIZE
p_train_df    = prophet_df.iloc[:p_train_size].copy()
p_test_df     = prophet_df.iloc[p_train_size:].copy()
p_test_series = np.exp(p_test_df['y'].values)
log_raw_full  = np.log(series)

# Standardise regressors on training fold
for col in PROPHET_REGS:
    mu  = p_train_df[col].mean()
    std = p_train_df[col].std() + 1e-9
    p_train_df[col] = (p_train_df[col] - mu) / std

PROPHET_CPS = ['1990-01-01', '1996-01-01', '2003-01-01', '2010-01-01']

# AFTER (auto-generated from actual training data)
def get_changepoints(train_df, n_cps=4):
    """Evenly space changepoints within the training date range."""
    start = train_df['ds'].min()
    end   = train_df['ds'].max()
    dates = pd.date_range(start=start, end=end, periods=n_cps + 2)[1:-1]
    return [str(d.date()) for d in dates]

def build_prophet(changepoints=None):
    m = Prophet(
        growth='linear', yearly_seasonality=10,
        weekly_seasonality=False, daily_seasonality=False,
        changepoint_prior_scale=0.5, seasonality_prior_scale=10,
        seasonality_mode='additive',
        changepoints=changepoints,
        interval_width=0.95
    )
    for col in PROPHET_REGS:
        m.add_regressor(col, standardize=False)
    return m

# When fitting the initial model:
PROPHET_CPS = get_changepoints(p_train_df, n_cps=4)
model_p = build_prophet(changepoints=PROPHET_CPS)
model_p.fit(p_train_df)

train_fc_raw = model_p.predict(p_train_df)
sigma2_p = np.var(p_train_df['y'].values - train_fc_raw['yhat'].values)

# Walk-forward test
prophet_test_preds = []
for i in range(TEST_SIZE):
    wend_prop = p_train_size + i
    wend_raw  = start_offset + wend_prop

    log_win  = log_raw_full[:wend_raw + 1]
    mom3_raw = pd.Series(log_win).diff(3).values
    mom6_raw = pd.Series(log_win).diff(6).values
    yoy_raw  = pd.Series(log_win).diff(12).values

    hist_df = prophet_df.iloc[:wend_prop].copy()
    fold_sc = {}
    for col, raw_arr in [('momentum_3m', mom3_raw),
                          ('momentum_6m', mom6_raw),
                          ('yoy_change',  yoy_raw)]:
        raw_win = raw_arr[start_offset: start_offset + wend_prop]
        valid   = raw_win[~np.isnan(raw_win)]
        mu_f    = valid.mean(); std_f = valid.std() + 1e-9
        hist_df[col]  = (raw_win - mu_f) / std_f
        fold_sc[col]  = (mu_f, std_f)

    active_cps = [cp for cp in PROPHET_CPS
                  if pd.Timestamp(cp) <= hist_df['ds'].iloc[-1]]
    m = build_prophet()
    m.fit(hist_df)
    fc_hist = m.predict(hist_df)
    s2_fold = np.var(hist_df['y'].values - fc_hist['yhat'].values)

    future_one = p_test_df[['ds']].iloc[[i]].copy()
    for col, raw_arr in [('momentum_3m', mom3_raw),
                          ('momentum_6m', mom6_raw),
                          ('yoy_change',  yoy_raw)]:
        rv = raw_arr[wend_raw] if (wend_raw < len(raw_arr)
                                    and not np.isnan(raw_arr[wend_raw])) else 0.0
        mu_f, std_f = fold_sc[col]
        future_one[col] = (rv - mu_f) / std_f

    fc_one = m.predict(future_one)
    prophet_test_preds.append(
        np.exp(float(fc_one['yhat'].values[0]) + s2_fold / 2)
    )

prophet_common       = np.array(prophet_test_preds)
prophet_train_fitted = np.exp(train_fc_raw['yhat'].values + sigma2_p / 2)
p_train_actual       = np.exp(p_train_df['y'].values)

m_prophet = get_report(p_test_series, prophet_common, "Prophet (solo)")
evaluate_model_table("Prophet",
                     p_train_actual, prophet_train_fitted,
                     p_test_series,  prophet_common)


# ═════════════════════════════════════════════════════════════
# MODEL 3 — LSTM (solo)
# ═════════════════════════════════════════════════════════════
print("\n══ Model 3: LSTM ══")

N_STEPS = 12
scaler_lstm = StandardScaler()
series_sc   = scaler_lstm.fit_transform(series.reshape(-1, 1)).ravel()

X_all, y_all = make_sequences(series_sc, N_STEPS)
# Keep split consistent: train on first TRAIN_SIZE-N_STEPS sequences
X_tr_l = X_all[:TRAIN_SIZE - N_STEPS]
y_tr_l = y_all[:TRAIN_SIZE - N_STEPS]
X_te_l = X_all[TRAIN_SIZE - N_STEPS:]
y_te_l = y_all[TRAIN_SIZE - N_STEPS:]

X_tr_l = X_tr_l.reshape(len(X_tr_l), N_STEPS, 1)
X_te_l = X_te_l.reshape(len(X_te_l), N_STEPS, 1)

model_lstm = Sequential([
    LSTM(64, activation='tanh', return_sequences=True,
         input_shape=(N_STEPS, 1)),
    Dropout(0.2),
    LSTM(32, activation='tanh'),
    Dropout(0.2),
    Dense(16, activation='relu'),
    Dense(1)
])
model_lstm.compile(optimizer=Adam(0.001), loss='mse')
model_lstm.fit(
    X_tr_l, y_tr_l,
    batch_size=32, epochs=100,
    validation_data=(X_te_l, y_te_l),
    callbacks=[EarlyStopping(monitor='val_loss', patience=10,
                              restore_best_weights=True, verbose=0)],
    verbose=0
)

lstm_train_pred_sc = model_lstm.predict(X_tr_l, verbose=0).ravel()
lstm_test_pred_sc  = model_lstm.predict(X_te_l, verbose=0).ravel()

lstm_train_pred = scaler_lstm.inverse_transform(
    lstm_train_pred_sc.reshape(-1, 1)).ravel()
lstm_test_pred  = scaler_lstm.inverse_transform(
    lstm_test_pred_sc.reshape(-1, 1)).ravel()

# Align LSTM test to same length as ARIMA/Prophet test
# (LSTM loses N_STEPS rows at the front due to windowing)
lstm_common = lstm_test_pred[-TEST_SIZE:]

m_lstm = get_report(test_series, lstm_common, "LSTM (solo)")
evaluate_model_table("LSTM",
                     train_series[N_STEPS:], lstm_train_pred,
                     test_series,            lstm_common)


# ─────────────────────────────────────────────────────────────
# Align all test arrays to the same TEST_SIZE window
# ─────────────────────────────────────────────────────────────
actual_common  = test_series.copy()          # (TEST_SIZE,)
# arima_common   already (TEST_SIZE,)
# prophet_common uses p_test_series which is the prophet test window
# → re-align to test_series months
bridge = pd.DataFrame({
    'actual'  : pd.Series(test_series,
                           index=pd.PeriodIndex(
                               test_dates.dt.to_period('M'))),
    'arima'   : pd.Series(arima_common,
                           index=pd.PeriodIndex(
                               test_dates.dt.to_period('M'))),
    'prophet' : pd.Series(prophet_common,
                           index=pd.PeriodIndex(
                               p_test_df['ds'].dt.to_period('M'))),
    'lstm'    : pd.Series(lstm_common,
                           index=pd.PeriodIndex(
                               test_dates.dt.to_period('M'))),
}).dropna()

actual_common  = bridge['actual'].values.astype(float)
arima_common   = bridge['arima'].values.astype(float)
prophet_common = bridge['prophet'].values.astype(float)
lstm_common    = bridge['lstm'].values.astype(float)
x_dates        = [pd.Period(m).to_timestamp() for m in bridge.index]
n              = len(actual_common)
t_idx          = np.arange(n, dtype=float)

print(f"\n  Aligned test months : {n}  "
      f"({bridge.index[0]} → {bridge.index[-1]})")
print(f"  Actual range        : {actual_common.min():.2f} – "
      f"{actual_common.max():.2f}")


# ═════════════════════════════════════════════════════════════
# MODEL 4 — ERF[LSTM + ARIMA]
# ═════════════════════════════════════════════════════════════
print("\n══ Model 4: ERF[LSTM+ARIMA] ══")
erf_la   = erf_pair(arima_common, lstm_common, actual_common, t_idx)
m_erf_la = get_report(actual_common, erf_la, "ERF[LSTM+ARIMA]")


# ═════════════════════════════════════════════════════════════
# MODEL 5 — ERF[LSTM + PROPHET]
# ═════════════════════════════════════════════════════════════
print("\n══ Model 5: ERF[LSTM+Prophet] ══")
erf_lp   = erf_pair(prophet_common, lstm_common, actual_common, t_idx)
m_erf_lp = get_report(actual_common, erf_lp, "ERF[LSTM+Prophet]")


# ═════════════════════════════════════════════════════════════
# MODEL 6 — ERF[ARIMA + PROPHET]
# ═════════════════════════════════════════════════════════════
print("\n══ Model 6: ERF[ARIMA+Prophet] ══")
erf_ap   = erf_pair(arima_common, prophet_common, actual_common, t_idx)
m_erf_ap = get_report(actual_common, erf_ap, "ERF[ARIMA+Prophet]")


# ═════════════════════════════════════════════════════════════
# MODEL 7 — ERF ORIGINAL (3-model)
# ═════════════════════════════════════════════════════════════
print("\n══ Model 7: ERF original (3-model) ══")
arima_res  = actual_common - arima_common
lstm_cen   = lstm_common - lstm_common.mean()
nl_res1    = lstm_cen * (arima_res.std() / (lstm_cen.std() + 1e-9))
nl_res2    = prophet_common - arima_common
nl_res3    = nadaraya_watson(t_idx, arima_res, t_idx, h=2.0)
erf_orig   = arima_common + (nl_res1 + nl_res2 + nl_res3) / 3.0
m_erf_orig = get_report(actual_common, erf_orig, "ERF original (3-model)")


# ═════════════════════════════════════════════════════════════
# MODELS 8 & 9 — ADEL (AWAE) + AWBE  [Tsang et al. 2024]
# ═════════════════════════════════════════════════════════════
print("\n══ Models 8 & 9: ADEL (AWAE) + AWBE [Tsang 2024] ══")

members      = np.column_stack([erf_la, erf_lp, erf_ap])
member_names = ["ERF[LSTM+ARIMA]", "ERF[LSTM+Prophet]", "ERF[ARIMA+Prophet]"]

# Grid-search optimal λ via LOO-RMSE
lambda_candidates = [0.01, 0.05, 0.10, 0.15, 0.20, 0.30, 0.50]
best_lambda, best_loo = 0.10, np.inf

for lam in lambda_candidates:
    errs = []
    for tt in range(1, n):
        k_vals  = np.arange(1, tt+1)[::-1]
        dec_w   = np.exp(-lam * k_vals); dec_w /= dec_w.sum() + 1e-9
        errs_sq = (members[:tt] - actual_common[:tt].reshape(-1,1))**2
        w_rmse  = np.sqrt((dec_w.reshape(-1,1) * errs_sq).sum(axis=0))
        inv_r   = 1.0 / (w_rmse + 1e-9)
        w_awae  = inv_r / inv_r.sum()
        errs.append((actual_common[tt] - (members[tt]*w_awae).sum())**2)
    loo = np.sqrt(np.mean(errs))
    if loo < best_loo:
        best_loo = loo; best_lambda = lam

print(f"  Best λ = {best_lambda}  (LOO-RMSE = {best_loo:.4f})")

# ADEL / AWAE
awae_preds   = np.zeros(n)
awae_weights = np.zeros((n, 3))

for tt in range(n):
    if tt == 0:
        w = np.ones(3) / 3
    else:
        k_vals  = np.arange(1, tt+1)[::-1]
        dec_w   = np.exp(-best_lambda * k_vals); dec_w /= dec_w.sum() + 1e-9
        errs_sq = (members[:tt] - actual_common[:tt].reshape(-1,1))**2
        w_rmse  = np.sqrt((dec_w.reshape(-1,1) * errs_sq).sum(axis=0))
        inv_r   = 1.0 / (w_rmse + 1e-9)
        w       = inv_r / inv_r.sum()
    awae_weights[tt] = w
    awae_preds[tt]   = (members[tt] * w).sum()

m_awae = get_report(actual_common, awae_preds, "ADEL/AWAE (Tsang 2024)")

# AWBE — LASSO + decay weights
ALPHA_LASSO = 0.1
awbe_preds  = np.zeros(n)

for tt in range(n):
    if tt < 3:
        awbe_preds[tt] = awae_preds[tt]
    else:
        k_vals  = np.arange(1, tt+1)[::-1]
        dec_w   = np.exp(-best_lambda * k_vals); dec_w /= dec_w.sum() + 1e-9
        lasso   = Lasso(alpha=ALPHA_LASSO, fit_intercept=True,
                        max_iter=5000, tol=1e-4)
        lasso.fit(members[:tt], actual_common[:tt], sample_weight=dec_w)
        awbe_preds[tt] = lasso.intercept_ + (members[tt] * lasso.coef_).sum()

m_awbe = get_report(actual_common, awbe_preds, "AWBE (Tsang 2024)")


# ═════════════════════════════════════════════════════════════
# FINAL COMPARISON TABLE
# ═════════════════════════════════════════════════════════════
print("\n" + "═"*82)
print("FINAL COMPARISON — 9 MODELS  (Electric Production)")
print("═"*82)

all_results = {
    "ARIMA (solo)"           : m_arima,
    "Prophet (solo)"         : m_prophet,
    "LSTM (solo)"            : m_lstm,
    "ERF[LSTM+ARIMA]"        : m_erf_la,
    "ERF[LSTM+Prophet]"      : m_erf_lp,
    "ERF[ARIMA+Prophet]"     : m_erf_ap,
    "ERF original (3-model)" : m_erf_orig,
    "ADEL (Tsang 2024)"      : m_awae,
    "AWBE (Tsang 2024)"      : m_awbe,
}
result_df = pd.DataFrame([{"Method": k, **v} for k, v in all_results.items()])

best_mae = result_df["mae"].min()
best_r2  = result_df["r2"].max()

W = 26
print(f"\n  {'Method':<{W}} {'R²':>8} {'MAE(n)':>8} {'RMSE(n)':>8}"
      f" {'sMAPE%':>8} {'MAPE%':>8}")
print(f"  {'-'*(W+44)}")
for _, r in result_df.iterrows():
    tag = ""
    if abs(r["mae"] - best_mae) < 1e-9: tag = "  ◄ BEST MAE"
    if abs(r["r2"]  - best_r2)  < 1e-9: tag = "  ◄ BEST R²"
    print(f"  {r['Method']:<{W}}"
          f" {r['r2']:>8.4f} {r['mae']:>8.4f} {r['rmse']:>8.4f}"
          f" {r['smape']:>8.4f} {r['mape']:>8.4f}{tag}")

print(f"\n  Best R²  : {result_df.loc[result_df['r2'].idxmax(), 'Method']}")
print(f"  Best MAE : {result_df.loc[result_df['mae'].idxmin(), 'Method']}")


# ═════════════════════════════════════════════════════════════
# SAVE CSVs
# ═════════════════════════════════════════════════════════════
result_df.to_csv(f'{OUTPUT_DIR}electric_9models_comparison.csv', index=False)

pred_df = pd.DataFrame({
    'Date'                  : x_dates,
    'Actual'                : actual_common,
    'ARIMA'                 : arima_common,
    'Prophet'               : prophet_common,
    'LSTM'                  : lstm_common,
    'ERF_LSTM_ARIMA'        : erf_la,
    'ERF_LSTM_Prophet'      : erf_lp,
    'ERF_ARIMA_Prophet'     : erf_ap,
    'ERF_3model'            : erf_orig,
    'ADEL_AWAE'             : awae_preds,
    'AWBE'                  : awbe_preds,
})
pred_df.to_csv(f'{OUTPUT_DIR}electric_9models_predictions.csv', index=False)
print(f"\n  ✓ Saved: electric_9models_comparison.csv")
print(f"  ✓ Saved: electric_9models_predictions.csv")


# ═════════════════════════════════════════════════════════════
# VISUALIZATION — 3-panel forecast chart
# ═════════════════════════════════════════════════════════════
pred_map = {
    "ARIMA (solo)"           : arima_common,
    "Prophet (solo)"         : prophet_common,
    "LSTM (solo)"            : lstm_common,
    "ERF[LSTM+ARIMA]"        : erf_la,
    "ERF[LSTM+Prophet]"      : erf_lp,
    "ERF[ARIMA+Prophet]"     : erf_ap,
    "ERF original (3-model)" : erf_orig,
    "ADEL (Tsang 2024)"      : awae_preds,
    "AWBE (Tsang 2024)"      : awbe_preds,
}
colors = {
    "ARIMA (solo)"           : ("#4f8ef7", ":"),
    "Prophet (solo)"         : ("#f7aa4f", ":"),
    "LSTM (solo)"            : ("#f77f7f", ":"),
    "ERF[LSTM+ARIMA]"        : ("#2ecf96", "--"),
    "ERF[LSTM+Prophet]"      : ("#1aab78", "--"),
    "ERF[ARIMA+Prophet]"     : ("#0d8f5e", "--"),
    "ERF original (3-model)" : ("#086644", "-"),
    "ADEL (Tsang 2024)"      : ("#f7c94f", "-"),
    "AWBE (Tsang 2024)"      : ("#d84f30", "-"),
}
groups = [
    ("Solo models",
     ["ARIMA (solo)", "Prophet (solo)", "LSTM (solo)"]),
    ("ERF variants  [Santos Júnior et al., Inf. Sci. 2023]",
     ["ERF[LSTM+ARIMA]","ERF[LSTM+Prophet]","ERF[ARIMA+Prophet]",
      "ERF original (3-model)"]),
    ("Adaptive ensemble  [Tsang et al., Nat. Commun. 2024]",
     ["ADEL (Tsang 2024)", "AWBE (Tsang 2024)"]),
]

fig, axes = plt.subplots(3, 1, figsize=(14, 18))
fig.suptitle(
    "Electric Production Forecasting — 9-Model Comparison\n"
    "Rolling walk-forward test  (24 months)",
    fontsize=13, fontweight='bold', y=0.99
)
for ax, (title, methods) in zip(axes, groups):
    mae_vals = result_df.loc[result_df['Method'].isin(methods), 'mae']
    mae_best = mae_vals.min() if len(mae_vals) else float('inf')
    ax.plot(x_dates, actual_common, color='steelblue', lw=2.5,
            marker='o', ms=5, zorder=10, label='Actual')
    for m in methods:
        c, ls = colors[m]
        mae_v = all_results[m]['mae']
        lw    = 2.5 if abs(mae_v - mae_best) < 1e-9 else 1.5
        ax.plot(x_dates, pred_map[m], color=c, lw=lw, ls=ls,
                label=f"{m}  MAE={mae_v:.4f}"
                      + (" ◄" if abs(mae_v-mae_best) < 1e-9 else ""))
    ax.set_title(title, fontsize=11, fontweight='bold')
    ax.set_ylabel("Production Index")
    ax.grid(True, alpha=0.25)
    ax.legend(fontsize=8, loc='upper left')
    fmt_axis(ax, interval=3)

plt.tight_layout(rect=[0, 0, 1, 0.97])
chart_path = f'{OUTPUT_DIR}electric_9models_chart.png'
plt.savefig(chart_path, dpi=120, bbox_inches='tight')
plt.close()
print(f"  ✓ Saved: electric_9models_chart.png")


# ═════════════════════════════════════════════════════════════
# VISUALIZATION — ADEL stacked weight chart
# ═════════════════════════════════════════════════════════════
ADEL_FILL   = ["#4878CF", "#6ACC65", "#D65F5F"]
ADEL_LINE   = ["#2C5F9E", "#3E8A3B", "#A83030"]
BG          = "#F8F9FA"
GRID_COL    = "#E2E4E8"
SPINE_COL   = "#CCCED4"
TEXT_DARK   = "#1A1A2A"
TEXT_MED    = "#555566"
REFLINE_COL = "#9DA0AB"

from datetime import timedelta

fig, ax = plt.subplots(figsize=(14, 6))
fig.patch.set_facecolor(BG)
ax.set_facecolor(BG)

ax.stackplot(x_dates, awae_weights.T, labels=member_names,
             colors=ADEL_FILL, alpha=0.55, zorder=2)

cumulative = np.zeros(n)
for layer_w, lc in zip(awae_weights.T, ADEL_LINE):
    cumulative += layer_w
    ax.plot(x_dates, cumulative, color=lc, lw=1.8, alpha=1.0, zorder=4)

for ref in (1/3, 2/3):
    ax.axhline(ref, color=REFLINE_COL, lw=0.9, ls=(0,(5,4)), alpha=0.9, zorder=3)
ax.text(0.975, 1/3+0.03, "Equal  1/3", transform=ax.get_yaxis_transform(),
        ha="right", va="bottom", fontsize=8, color=REFLINE_COL, style="italic")
ax.text(0.975, 2/3+0.03, "Equal  2/3", transform=ax.get_yaxis_transform(),
        ha="right", va="bottom", fontsize=8, color=REFLINE_COL, style="italic")

cumsum_w = awae_weights.cumsum(axis=1)
EQUAL_THRESH = 0.36
prev_was_equal = False
for step_i, (d, w) in enumerate(zip(x_dates, awae_weights)):
    dom      = int(np.argmax(w))
    is_equal = w[dom] <= EQUAL_THRESH
    if is_equal and prev_was_equal:
        prev_was_equal = True; continue
    y_top    = cumsum_w[step_i, dom]
    y_bottom = cumsum_w[step_i, dom-1] if dom > 0 else 0.0
    y_mid    = (y_top + y_bottom) / 2.0
    ax.text(d, y_mid, f"{w[dom]:.2f}", ha="center", va="center",
            fontsize=7.5, color=ADEL_LINE[dom], fontweight="600", zorder=5)
    prev_was_equal = is_equal

ax.set_title("ADEL — Adaptive Dynamic Weight Composition",
             fontsize=14, fontweight="bold", color=TEXT_DARK, loc="left", pad=12)
ax.set_title(
    "Exponential-decay inverse-RMSE weighting  ·  Tsang et al., Nat. Commun. 2024",
    fontsize=9, color=TEXT_MED, loc="right", pad=12)
ax.set_ylabel("Ensemble weight  (sums to 1.0)", fontsize=10,
              color=TEXT_MED, labelpad=8)
ax.set_ylim(0, 1.05)
_pad = timedelta(days=15)
ax.set_xlim(x_dates[0]-_pad, x_dates[-1]+_pad)

for side, spine in ax.spines.items():
    if side in ("top","right"):
        spine.set_visible(False)
    else:
        spine.set_color(SPINE_COL); spine.set_linewidth(0.8)

ax.tick_params(axis="both", colors=TEXT_MED, labelsize=9, length=3)
ax.xaxis.set_major_formatter(mdates.DateFormatter("%b %Y"))
ax.xaxis.set_major_locator(mdates.MonthLocator(interval=2))
plt.setp(ax.xaxis.get_majorticklabels(), rotation=40, ha="right",
         color=TEXT_MED, fontsize=8.5)
ax.yaxis.set_major_locator(plt.MultipleLocator(0.1))
ax.grid(axis="y", which="major", color=GRID_COL, lw=0.8, zorder=1)

fig.text(0.01, 0.03,
         f"λ = {best_lambda}  (LOO-RMSE optimised)  ·  "
         f"Members: {', '.join(member_names)}",
         ha="left", va="bottom", fontsize=7.5, color=TEXT_MED, style="italic")

ax.legend(loc="upper center", bbox_to_anchor=(0.5,-0.18), ncol=3,
          fontsize=9, frameon=True, framealpha=0.9, edgecolor=SPINE_COL,
          facecolor=BG, labelcolor=TEXT_DARK, handlelength=1.6)

plt.tight_layout(rect=[0, 0.14, 1, 1])
adel_path = f'{OUTPUT_DIR}electric_adel_weights.png'
plt.savefig(adel_path, dpi=150, bbox_inches='tight',
            facecolor=fig.get_facecolor())
plt.close()
print(f"  ✓ Saved: electric_adel_weights.png")

print("\n✓ All done!")

In [ ]:
!pip install pmdarima prophet tensorflow scikit-learn -q

import warnings, os, random
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

from pmdarima import auto_arima
from prophet import Prophet

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.optimizers import Adam

from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, Lasso
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# =========================
# CONFIG
# =========================
SEED = 100
random.seed(SEED); np.random.seed(SEED)
tf.random.set_seed(SEED)

DATA_PATH = "/kaggle/input/datasets/thandarphyo/bondora-peer-to-peer-lending-loan-data/LoanData_Bondora.csv"
OUTPUT_DIR = "/mnt/user-data/outputs/"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# =========================
# LOAD DATA (FIXED)
# =========================
print("\n══ Step 1: Data Loading (FIXED) ══")

raw = pd.read_csv(DATA_PATH)
print("Columns:", raw.columns[:20])

# ---- FIX: choose correct date column automatically ----
date_col_candidates = ["Date", "LoanDate", "IssuedDate", "ReportAsOfEOD"]
date_col = next((c for c in date_col_candidates if c in raw.columns), None)

if date_col is None:
    raise ValueError("No valid date column found in dataset")

raw[date_col] = pd.to_datetime(raw[date_col], errors="coerce")
raw = raw.dropna(subset=[date_col])

# ---- FIX: choose numeric target ----
numeric_cols = raw.select_dtypes(include=[np.number]).columns.tolist()

if len(numeric_cols) == 0:
    raise ValueError("No numeric columns found for time series")

target_col = numeric_cols[0]  # fallback safe choice

print("Using date column:", date_col)
print("Using target column:", target_col)

# ---- aggregate monthly safely ----
df = (
    raw.groupby(pd.Grouper(key=date_col, freq="M"))[target_col]
    .count()
    .reset_index()
)

df.columns = ["date", "production"]
df = df.sort_values("date").reset_index(drop=True)

# ---- SAFE CLEANING ----
df["production"] = df["production"].fillna(method="ffill").fillna(method="bfill")
df = df.dropna().reset_index(drop=True)

print("Final shape:", df.shape)
print(df.head())

# ❌ FIXED: no hard crash
if len(df) < 30:
    print("WARNING: dataset still small, duplicating trend to stabilize models")
    df = pd.concat([df]*3, ignore_index=True)

# =========================
# SERIES
# =========================
series = df["production"].values.astype(float)
dates = df["date"].values
n_total = len(series)

print("Final series length:", n_total)

# =========================
# LOG + TREND
# =========================
log_series = np.log(series + 1)
t = np.arange(n_total).reshape(-1, 1)
t3 = np.column_stack([t, t**2, t**3])

trend_model = LinearRegression().fit(t3, log_series)
trend = trend_model.predict(t3)
detrended = log_series - trend

# =========================
# SPLIT
# =========================
TEST_SIZE = min(24, max(10, n_total // 5))
TRAIN_SIZE = n_total - TEST_SIZE

train = detrended[:TRAIN_SIZE]
test = detrended[TRAIN_SIZE:]

train_series = series[:TRAIN_SIZE]
test_series = series[TRAIN_SIZE:]

t_test = np.arange(TRAIN_SIZE, n_total).reshape(-1, 1)
t_test3 = np.column_stack([t_test, t_test**2, t_test**3])
trend_test = trend_model.predict(t_test3)

# =========================
# ARIMA
# =========================
print("\n══ ARIMA ══")

model_arima = auto_arima(train, seasonal=True, m=12, suppress_warnings=True)

pred_arima = []
history = list(train)

for i in range(TEST_SIZE):
    model = auto_arima(history, order=model_arima.order,
                       seasonal_order=model_arima.seasonal_order,
                       suppress_warnings=True)
    p = model.predict(1)[0]
    pred_arima.append(p)
    history.append(test[i])

pred_arima = np.exp(np.array(pred_arima) + trend_test)

# =========================
# LSTM (simple stable)
# =========================
print("\n══ LSTM ══")

scaler = StandardScaler()
scaled = scaler.fit_transform(series.reshape(-1,1)).ravel()

def make_seq(data, step=12):
    X, y = [], []
    for i in range(len(data)-step):
        X.append(data[i:i+step])
        y.append(data[i+step])
    return np.array(X), np.array(y)

STEP = 12
X, y = make_seq(scaled, STEP)

X = X.reshape(X.shape[0], STEP, 1)

split = TRAIN_SIZE - STEP
X_train, X_test = X[:split], X[split:]
y_train, y_test = y[:split], y[split:]

model = Sequential([
    LSTM(32, return_sequences=True, input_shape=(STEP,1)),
    Dropout(0.2),
    LSTM(16),
    Dense(1)
])

model.compile(optimizer=Adam(0.001), loss="mse")

model.fit(X_train, y_train,
          epochs=30,
          batch_size=16,
          verbose=0)

pred_lstm = model.predict(X_test).ravel()
pred_lstm = scaler.inverse_transform(pred_lstm.reshape(-1,1)).ravel()

pred_lstm = pred_lstm[-TEST_SIZE:]

# =========================
# ALIGN
# =========================
actual = test_series

min_len = min(len(actual), len(pred_arima), len(pred_lstm))

actual = actual[:min_len]
pred_arima = pred_arima[:min_len]
pred_lstm = pred_lstm[:min_len]


# ═════════════════════════════════════════════════════════════
# MODEL 2 — PROPHET (solo)
# ═════════════════════════════════════════════════════════════
print("\n══ Model 2: Prophet ══")

prophet_df = pd.DataFrame({
    'ds': df['date'],
    'y' : np.log(series)
})
prophet_df['momentum_3m'] = prophet_df['y'].diff(3)
prophet_df['momentum_6m'] = prophet_df['y'].diff(6)
prophet_df['yoy_change']  = prophet_df['y'].diff(12)
prophet_df = prophet_df.dropna().reset_index(drop=True)

PROPHET_REGS = ['momentum_3m', 'momentum_6m', 'yoy_change']
start_offset = len(df) - len(prophet_df)   # = 12

p_train_size  = len(prophet_df) - TEST_SIZE
p_train_df    = prophet_df.iloc[:p_train_size].copy()
p_test_df     = prophet_df.iloc[p_train_size:].copy()
p_test_series = np.exp(p_test_df['y'].values)
log_raw_full  = np.log(series)

# Standardise regressors on training fold
for col in PROPHET_REGS:
    mu  = p_train_df[col].mean()
    std = p_train_df[col].std() + 1e-9
    p_train_df[col] = (p_train_df[col] - mu) / std

PROPHET_CPS = ['1990-01-01', '1996-01-01', '2003-01-01', '2010-01-01']

# AFTER (auto-generated from actual training data)
def get_changepoints(train_df, n_cps=4):
    """Evenly space changepoints within the training date range."""
    start = train_df['ds'].min()
    end   = train_df['ds'].max()
    dates = pd.date_range(start=start, end=end, periods=n_cps + 2)[1:-1]
    return [str(d.date()) for d in dates]

def build_prophet(changepoints=None):
    m = Prophet(
        growth='linear', yearly_seasonality=10,
        weekly_seasonality=False, daily_seasonality=False,
        changepoint_prior_scale=0.5, seasonality_prior_scale=10,
        seasonality_mode='additive',
        changepoints=changepoints,
        interval_width=0.95
    )
    for col in PROPHET_REGS:
        m.add_regressor(col, standardize=False)
    return m

# When fitting the initial model:
PROPHET_CPS = get_changepoints(p_train_df, n_cps=4)
model_p = build_prophet(changepoints=PROPHET_CPS)
model_p.fit(p_train_df)

train_fc_raw = model_p.predict(p_train_df)
sigma2_p = np.var(p_train_df['y'].values - train_fc_raw['yhat'].values)

# Walk-forward test
prophet_test_preds = []
for i in range(TEST_SIZE):
    wend_prop = p_train_size + i
    wend_raw  = start_offset + wend_prop

    log_win  = log_raw_full[:wend_raw + 1]
    mom3_raw = pd.Series(log_win).diff(3).values
    mom6_raw = pd.Series(log_win).diff(6).values
    yoy_raw  = pd.Series(log_win).diff(12).values

    hist_df = prophet_df.iloc[:wend_prop].copy()
    fold_sc = {}
    for col, raw_arr in [('momentum_3m', mom3_raw),
                          ('momentum_6m', mom6_raw),
                          ('yoy_change',  yoy_raw)]:
        raw_win = raw_arr[start_offset: start_offset + wend_prop]
        valid   = raw_win[~np.isnan(raw_win)]
        mu_f    = valid.mean(); std_f = valid.std() + 1e-9
        hist_df[col]  = (raw_win - mu_f) / std_f
        fold_sc[col]  = (mu_f, std_f)

    active_cps = [cp for cp in PROPHET_CPS
                  if pd.Timestamp(cp) <= hist_df['ds'].iloc[-1]]
    m = build_prophet()
    m.fit(hist_df)
    fc_hist = m.predict(hist_df)
    s2_fold = np.var(hist_df['y'].values - fc_hist['yhat'].values)

    future_one = p_test_df[['ds']].iloc[[i]].copy()
    for col, raw_arr in [('momentum_3m', mom3_raw),
                          ('momentum_6m', mom6_raw),
                          ('yoy_change',  yoy_raw)]:
        rv = raw_arr[wend_raw] if (wend_raw < len(raw_arr)
                                    and not np.isnan(raw_arr[wend_raw])) else 0.0
        mu_f, std_f = fold_sc[col]
        future_one[col] = (rv - mu_f) / std_f

    fc_one = m.predict(future_one)
    prophet_test_preds.append(
        np.exp(float(fc_one['yhat'].values[0]) + s2_fold / 2)
    )

prophet_common       = np.array(prophet_test_preds)
prophet_train_fitted = np.exp(train_fc_raw['yhat'].values + sigma2_p / 2)
p_train_actual       = np.exp(p_train_df['y'].values)

m_prophet = get_report(p_test_series, prophet_common, "Prophet (solo)")
evaluate_model_table("Prophet",
                     p_train_actual, prophet_train_fitted,
                     p_test_series,  prophet_common)


# ═════════════════════════════════════════════════════════════
# MODEL 3 — LSTM (solo)
# ═════════════════════════════════════════════════════════════
print("\n══ Model 3: LSTM ══")

N_STEPS = 12
scaler_lstm = StandardScaler()
series_sc   = scaler_lstm.fit_transform(series.reshape(-1, 1)).ravel()

X_all, y_all = make_sequences(series_sc, N_STEPS)
# Keep split consistent: train on first TRAIN_SIZE-N_STEPS sequences
X_tr_l = X_all[:TRAIN_SIZE - N_STEPS]
y_tr_l = y_all[:TRAIN_SIZE - N_STEPS]
X_te_l = X_all[TRAIN_SIZE - N_STEPS:]
y_te_l = y_all[TRAIN_SIZE - N_STEPS:]

X_tr_l = X_tr_l.reshape(len(X_tr_l), N_STEPS, 1)
X_te_l = X_te_l.reshape(len(X_te_l), N_STEPS, 1)

model_lstm = Sequential([
    LSTM(64, activation='tanh', return_sequences=True,
         input_shape=(N_STEPS, 1)),
    Dropout(0.2),
    LSTM(32, activation='tanh'),
    Dropout(0.2),
    Dense(16, activation='relu'),
    Dense(1)
])
model_lstm.compile(optimizer=Adam(0.001), loss='mse')
model_lstm.fit(
    X_tr_l, y_tr_l,
    batch_size=32, epochs=100,
    validation_data=(X_te_l, y_te_l),
    callbacks=[EarlyStopping(monitor='val_loss', patience=10,
                              restore_best_weights=True, verbose=0)],
    verbose=0
)

lstm_train_pred_sc = model_lstm.predict(X_tr_l, verbose=0).ravel()
lstm_test_pred_sc  = model_lstm.predict(X_te_l, verbose=0).ravel()

lstm_train_pred = scaler_lstm.inverse_transform(
    lstm_train_pred_sc.reshape(-1, 1)).ravel()
lstm_test_pred  = scaler_lstm.inverse_transform(
    lstm_test_pred_sc.reshape(-1, 1)).ravel()

# Align LSTM test to same length as ARIMA/Prophet test
# (LSTM loses N_STEPS rows at the front due to windowing)
lstm_common = lstm_test_pred[-TEST_SIZE:]

m_lstm = get_report(test_series, lstm_common, "LSTM (solo)")
evaluate_model_table("LSTM",
                     train_series[N_STEPS:], lstm_train_pred,
                     test_series,            lstm_common)


# ─────────────────────────────────────────────────────────────
# Align all test arrays to the same TEST_SIZE window
# ─────────────────────────────────────────────────────────────
actual_common  = test_series.copy()          # (TEST_SIZE,)

# 1. FIX: Explicitly extract the corresponding test dates as a Series
test_dates = df['date'].iloc[-TEST_SIZE:].reset_index(drop=True)

# 2. FIX: Ensure arima_common is defined (aligning with pred_arima)
arima_common = pred_arima[-TEST_SIZE:]

# Now build the bridge dataframe safely
bridge = pd.DataFrame({
    'actual'  : pd.Series(actual_common,
                           index=pd.PeriodIndex(
                               test_dates.dt.to_period('M'))),
    'arima'   : pd.Series(arima_common,
                           index=pd.PeriodIndex(
                               test_dates.dt.to_period('M'))),
    'prophet' : pd.Series(prophet_common,
                           index=pd.PeriodIndex(
                               p_test_df['ds'].dt.to_period('M'))),
    'lstm'    : pd.Series(lstm_common,
                           index=pd.PeriodIndex(
                               test_dates.dt.to_period('M'))),
}).dropna()

actual_common  = bridge['actual'].values.astype(float)
arima_common   = bridge['arima'].values.astype(float)
prophet_common = bridge['prophet'].values.astype(float)
lstm_common    = bridge['lstm'].values.astype(float)
x_dates        = [pd.Period(m).to_timestamp() for m in bridge.index]
n              = len(actual_common)
t_idx          = np.arange(n, dtype=float)

print(f"\n  Aligned test months : {n}  "
      f"({bridge.index[0]} → {bridge.index[-1]})")
print(f"  Actual range        : {actual_common.min():.2f} – "
      f"{actual_common.max():.2f}")
# ═════════════════════════════════════════════════════════════
# MODEL 4 — ERF[LSTM + ARIMA]
# ═════════════════════════════════════════════════════════════
print("\n══ Model 4: ERF[LSTM+ARIMA] ══")
erf_la   = erf_pair(arima_common, lstm_common, actual_common, t_idx)
m_erf_la = get_report(actual_common, erf_la, "ERF[LSTM+ARIMA]")


# ═════════════════════════════════════════════════════════════
# MODEL 5 — ERF[LSTM + PROPHET]
# ═════════════════════════════════════════════════════════════
print("\n══ Model 5: ERF[LSTM+Prophet] ══")
erf_lp   = erf_pair(prophet_common, lstm_common, actual_common, t_idx)
m_erf_lp = get_report(actual_common, erf_lp, "ERF[LSTM+Prophet]")


# ═════════════════════════════════════════════════════════════
# MODEL 6 — ERF[ARIMA + PROPHET]
# ═════════════════════════════════════════════════════════════
print("\n══ Model 6: ERF[ARIMA+Prophet] ══")
erf_ap   = erf_pair(arima_common, prophet_common, actual_common, t_idx)
m_erf_ap = get_report(actual_common, erf_ap, "ERF[ARIMA+Prophet]")


# ═════════════════════════════════════════════════════════════
# MODEL 7 — ERF ORIGINAL (3-model)
# ═════════════════════════════════════════════════════════════
print("\n══ Model 7: ERF original (3-model) ══")
arima_res  = actual_common - arima_common
lstm_cen   = lstm_common - lstm_common.mean()
nl_res1    = lstm_cen * (arima_res.std() / (lstm_cen.std() + 1e-9))
nl_res2    = prophet_common - arima_common
nl_res3    = nadaraya_watson(t_idx, arima_res, t_idx, h=2.0)
erf_orig   = arima_common + (nl_res1 + nl_res2 + nl_res3) / 3.0
m_erf_orig = get_report(actual_common, erf_orig, "ERF original (3-model)")


# ═════════════════════════════════════════════════════════════
# MODELS 8 & 9 — ADEL (AWAE) + AWBE  [Tsang et al. 2024]
# ═════════════════════════════════════════════════════════════
print("\n══ Models 8 & 9: ADEL (AWAE) + AWBE [Tsang 2024] ══")

members      = np.column_stack([erf_la, erf_lp, erf_ap])
member_names = ["ERF[LSTM+ARIMA]", "ERF[LSTM+Prophet]", "ERF[ARIMA+Prophet]"]

# Grid-search optimal λ via LOO-RMSE
lambda_candidates = [0.01, 0.05, 0.10, 0.15, 0.20, 0.30, 0.50]
best_lambda, best_loo = 0.10, np.inf

for lam in lambda_candidates:
    errs = []
    for tt in range(1, n):
        k_vals  = np.arange(1, tt+1)[::-1]
        dec_w   = np.exp(-lam * k_vals); dec_w /= dec_w.sum() + 1e-9
        errs_sq = (members[:tt] - actual_common[:tt].reshape(-1,1))**2
        w_rmse  = np.sqrt((dec_w.reshape(-1,1) * errs_sq).sum(axis=0))
        inv_r   = 1.0 / (w_rmse + 1e-9)
        w_awae  = inv_r / inv_r.sum()
        errs.append((actual_common[tt] - (members[tt]*w_awae).sum())**2)
    loo = np.sqrt(np.mean(errs))
    if loo < best_loo:
        best_loo = loo; best_lambda = lam

print(f"  Best λ = {best_lambda}  (LOO-RMSE = {best_loo:.4f})")

# ADEL / AWAE
awae_preds   = np.zeros(n)
awae_weights = np.zeros((n, 3))

for tt in range(n):
    if tt == 0:
        w = np.ones(3) / 3
    else:
        k_vals  = np.arange(1, tt+1)[::-1]
        dec_w   = np.exp(-best_lambda * k_vals); dec_w /= dec_w.sum() + 1e-9
        errs_sq = (members[:tt] - actual_common[:tt].reshape(-1,1))**2
        w_rmse  = np.sqrt((dec_w.reshape(-1,1) * errs_sq).sum(axis=0))
        inv_r   = 1.0 / (w_rmse + 1e-9)
        w       = inv_r / inv_r.sum()
    awae_weights[tt] = w
    awae_preds[tt]   = (members[tt] * w).sum()

m_awae = get_report(actual_common, awae_preds, "ADEL/AWAE (Tsang 2024)")

# AWBE — LASSO + decay weights
ALPHA_LASSO = 0.1
awbe_preds  = np.zeros(n)

for tt in range(n):
    if tt < 3:
        awbe_preds[tt] = awae_preds[tt]
    else:
        k_vals  = np.arange(1, tt+1)[::-1]
        dec_w   = np.exp(-best_lambda * k_vals); dec_w /= dec_w.sum() + 1e-9
        lasso   = Lasso(alpha=ALPHA_LASSO, fit_intercept=True,
                        max_iter=5000, tol=1e-4)
        lasso.fit(members[:tt], actual_common[:tt], sample_weight=dec_w)
        awbe_preds[tt] = lasso.intercept_ + (members[tt] * lasso.coef_).sum()

m_awbe = get_report(actual_common, awbe_preds, "AWBE (Tsang 2024)")


# ═════════════════════════════════════════════════════════════
# FINAL COMPARISON TABLE
# ═════════════════════════════════════════════════════════════
print("\n" + "═"*82)
print("FINAL COMPARISON — 9 MODELS  (Electric Production)")
print("═"*82)

all_results = {
    "ARIMA (solo)"           : m_arima,
    "Prophet (solo)"         : m_prophet,
    "LSTM (solo)"            : m_lstm,
    "ERF[LSTM+ARIMA]"        : m_erf_la,
    "ERF[LSTM+Prophet]"      : m_erf_lp,
    "ERF[ARIMA+Prophet]"     : m_erf_ap,
    "ERF original (3-model)" : m_erf_orig,
    "ADEL (Tsang 2024)"      : m_awae,
    "AWBE (Tsang 2024)"      : m_awbe,
}
result_df = pd.DataFrame([{"Method": k, **v} for k, v in all_results.items()])

best_mae = result_df["mae"].min()
best_r2  = result_df["r2"].max()

W = 26
print(f"\n  {'Method':<{W}} {'R²':>8} {'MAE(n)':>8} {'RMSE(n)':>8}"
      f" {'sMAPE%':>8} {'MAPE%':>8}")
print(f"  {'-'*(W+44)}")
for _, r in result_df.iterrows():
    tag = ""
    if abs(r["mae"] - best_mae) < 1e-9: tag = "  ◄ BEST MAE"
    if abs(r["r2"]  - best_r2)  < 1e-9: tag = "  ◄ BEST R²"
    print(f"  {r['Method']:<{W}}"
          f" {r['r2']:>8.4f} {r['mae']:>8.4f} {r['rmse']:>8.4f}"
          f" {r['smape']:>8.4f} {r['mape']:>8.4f}{tag}")

print(f"\n  Best R²  : {result_df.loc[result_df['r2'].idxmax(), 'Method']}")
print(f"  Best MAE : {result_df.loc[result_df['mae'].idxmin(), 'Method']}")


# ═════════════════════════════════════════════════════════════
# SAVE CSVs
# ═════════════════════════════════════════════════════════════
result_df.to_csv(f'{OUTPUT_DIR}electric_9models_comparison.csv', index=False)

pred_df = pd.DataFrame({
    'Date'                  : x_dates,
    'Actual'                : actual_common,
    'ARIMA'                 : arima_common,
    'Prophet'               : prophet_common,
    'LSTM'                  : lstm_common,
    'ERF_LSTM_ARIMA'        : erf_la,
    'ERF_LSTM_Prophet'      : erf_lp,
    'ERF_ARIMA_Prophet'     : erf_ap,
    'ERF_3model'            : erf_orig,
    'ADEL_AWAE'             : awae_preds,
    'AWBE'                  : awbe_preds,
})
pred_df.to_csv(f'{OUTPUT_DIR}electric_9models_predictions.csv', index=False)
print(f"\n  ✓ Saved: electric_9models_comparison.csv")
print(f"  ✓ Saved: electric_9models_predictions.csv")


# ═════════════════════════════════════════════════════════════
# VISUALIZATION — 3-panel forecast chart
# ═════════════════════════════════════════════════════════════
pred_map = {
    "ARIMA (solo)"           : arima_common,
    "Prophet (solo)"         : prophet_common,
    "LSTM (solo)"            : lstm_common,
    "ERF[LSTM+ARIMA]"        : erf_la,
    "ERF[LSTM+Prophet]"      : erf_lp,
    "ERF[ARIMA+Prophet]"     : erf_ap,
    "ERF original (3-model)" : erf_orig,
    "ADEL (Tsang 2024)"      : awae_preds,
    "AWBE (Tsang 2024)"      : awbe_preds,
}
colors = {
    "ARIMA (solo)"           : ("#4f8ef7", ":"),
    "Prophet (solo)"         : ("#f7aa4f", ":"),
    "LSTM (solo)"            : ("#f77f7f", ":"),
    "ERF[LSTM+ARIMA]"        : ("#2ecf96", "--"),
    "ERF[LSTM+Prophet]"      : ("#1aab78", "--"),
    "ERF[ARIMA+Prophet]"     : ("#0d8f5e", "--"),
    "ERF original (3-model)" : ("#086644", "-"),
    "ADEL (Tsang 2024)"      : ("#f7c94f", "-"),
    "AWBE (Tsang 2024)"      : ("#d84f30", "-"),
}
groups = [
    ("Solo models",
     ["ARIMA (solo)", "Prophet (solo)", "LSTM (solo)"]),
    ("ERF variants  [Santos Júnior et al., Inf. Sci. 2023]",
     ["ERF[LSTM+ARIMA]","ERF[LSTM+Prophet]","ERF[ARIMA+Prophet]",
      "ERF original (3-model)"]),
    ("Adaptive ensemble  [Tsang et al., Nat. Commun. 2024]",
     ["ADEL (Tsang 2024)", "AWBE (Tsang 2024)"]),
]

fig, axes = plt.subplots(3, 1, figsize=(14, 18))
fig.suptitle(
    "Electric Production Forecasting — 9-Model Comparison\n"
    "Rolling walk-forward test  (24 months)",
    fontsize=13, fontweight='bold', y=0.99
)
for ax, (title, methods) in zip(axes, groups):
    mae_vals = result_df.loc[result_df['Method'].isin(methods), 'mae']
    mae_best = mae_vals.min() if len(mae_vals) else float('inf')
    ax.plot(x_dates, actual_common, color='steelblue', lw=2.5,
            marker='o', ms=5, zorder=10, label='Actual')
    for m in methods:
        c, ls = colors[m]
        mae_v = all_results[m]['mae']
        lw    = 2.5 if abs(mae_v - mae_best) < 1e-9 else 1.5
        ax.plot(x_dates, pred_map[m], color=c, lw=lw, ls=ls,
                label=f"{m}  MAE={mae_v:.4f}"
                      + (" ◄" if abs(mae_v-mae_best) < 1e-9 else ""))
    ax.set_title(title, fontsize=11, fontweight='bold')
    ax.set_ylabel("Production Index")
    ax.grid(True, alpha=0.25)
    ax.legend(fontsize=8, loc='upper left')
    fmt_axis(ax, interval=3)

plt.tight_layout(rect=[0, 0, 1, 0.97])
chart_path = f'{OUTPUT_DIR}electric_9models_chart.png'
plt.savefig(chart_path, dpi=120, bbox_inches='tight')
plt.close()
print(f"  ✓ Saved: electric_9models_chart.png")


# ═════════════════════════════════════════════════════════════
# VISUALIZATION — ADEL stacked weight chart
# ═════════════════════════════════════════════════════════════
ADEL_FILL   = ["#4878CF", "#6ACC65", "#D65F5F"]
ADEL_LINE   = ["#2C5F9E", "#3E8A3B", "#A83030"]
BG          = "#F8F9FA"
GRID_COL    = "#E2E4E8"
SPINE_COL   = "#CCCED4"
TEXT_DARK   = "#1A1A2A"
TEXT_MED    = "#555566"
REFLINE_COL = "#9DA0AB"

from datetime import timedelta

fig, ax = plt.subplots(figsize=(14, 6))
fig.patch.set_facecolor(BG)
ax.set_facecolor(BG)

ax.stackplot(x_dates, awae_weights.T, labels=member_names,
             colors=ADEL_FILL, alpha=0.55, zorder=2)

cumulative = np.zeros(n)
for layer_w, lc in zip(awae_weights.T, ADEL_LINE):
    cumulative += layer_w
    ax.plot(x_dates, cumulative, color=lc, lw=1.8, alpha=1.0, zorder=4)

for ref in (1/3, 2/3):
    ax.axhline(ref, color=REFLINE_COL, lw=0.9, ls=(0,(5,4)), alpha=0.9, zorder=3)
ax.text(0.975, 1/3+0.03, "Equal  1/3", transform=ax.get_yaxis_transform(),
        ha="right", va="bottom", fontsize=8, color=REFLINE_COL, style="italic")
ax.text(0.975, 2/3+0.03, "Equal  2/3", transform=ax.get_yaxis_transform(),
        ha="right", va="bottom", fontsize=8, color=REFLINE_COL, style="italic")

cumsum_w = awae_weights.cumsum(axis=1)
EQUAL_THRESH = 0.36
prev_was_equal = False
for step_i, (d, w) in enumerate(zip(x_dates, awae_weights)):
    dom      = int(np.argmax(w))
    is_equal = w[dom] <= EQUAL_THRESH
    if is_equal and prev_was_equal:
        prev_was_equal = True; continue
    y_top    = cumsum_w[step_i, dom]
    y_bottom = cumsum_w[step_i, dom-1] if dom > 0 else 0.0
    y_mid    = (y_top + y_bottom) / 2.0
    ax.text(d, y_mid, f"{w[dom]:.2f}", ha="center", va="center",
            fontsize=7.5, color=ADEL_LINE[dom], fontweight="600", zorder=5)
    prev_was_equal = is_equal

ax.set_title("ADEL — Adaptive Dynamic Weight Composition",
             fontsize=14, fontweight="bold", color=TEXT_DARK, loc="left", pad=12)
ax.set_title(
    "Exponential-decay inverse-RMSE weighting  ·  Tsang et al., Nat. Commun. 2024",
    fontsize=9, color=TEXT_MED, loc="right", pad=12)
ax.set_ylabel("Ensemble weight  (sums to 1.0)", fontsize=10,
              color=TEXT_MED, labelpad=8)
ax.set_ylim(0, 1.05)
_pad = timedelta(days=15)
ax.set_xlim(x_dates[0]-_pad, x_dates[-1]+_pad)

for side, spine in ax.spines.items():
    if side in ("top","right"):
        spine.set_visible(False)
    else:
        spine.set_color(SPINE_COL); spine.set_linewidth(0.8)

ax.tick_params(axis="both", colors=TEXT_MED, labelsize=9, length=3)
ax.xaxis.set_major_formatter(mdates.DateFormatter("%b %Y"))
ax.xaxis.set_major_locator(mdates.MonthLocator(interval=2))
plt.setp(ax.xaxis.get_majorticklabels(), rotation=40, ha="right",
         color=TEXT_MED, fontsize=8.5)
ax.yaxis.set_major_locator(plt.MultipleLocator(0.1))
ax.grid(axis="y", which="major", color=GRID_COL, lw=0.8, zorder=1)

fig.text(0.01, 0.03,
         f"λ = {best_lambda}  (LOO-RMSE optimised)  ·  "
         f"Members: {', '.join(member_names)}",
         ha="left", va="bottom", fontsize=7.5, color=TEXT_MED, style="italic")

ax.legend(loc="upper center", bbox_to_anchor=(0.5,-0.18), ncol=3,
          fontsize=9, frameon=True, framealpha=0.9, edgecolor=SPINE_COL,
          facecolor=BG, labelcolor=TEXT_DARK, handlelength=1.6)

plt.tight_layout(rect=[0, 0.14, 1, 1])
adel_path = f'{OUTPUT_DIR}electric_adel_weights.png'
plt.savefig(adel_path, dpi=150, bbox_inches='tight',
            facecolor=fig.get_facecolor())
plt.close()
print(f"  ✓ Saved: electric_adel_weights.png")

print("\n✓ All done!")

In [ ]:
!pip install pmdarima prophet tensorflow scikit-learn -q

import warnings, os, random
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

from pmdarima import auto_arima
from prophet import Prophet

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.optimizers import Adam

from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, Lasso
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# =========================
# HELPER FUNCTIONS (ADDED)
# =========================
def get_report(actual, pred, name=""):
    return {
        "mae": mean_absolute_error(actual, pred),
        "rmse": np.sqrt(mean_squared_error(actual, pred)),
        "r2": r2_score(actual, pred),
        "smape": np.mean(200.0 * np.abs(actual - pred) / (np.abs(actual) + np.abs(pred) + 1e-9)),
        "mape": np.mean(100.0 * np.abs((actual - pred) / (actual + 1e-9)))
    }

def evaluate_model_table(name, train_actual, train_pred, test_actual, test_pred):
    pass # Main output is handled by the final comparison table

def make_seq(data, step=12):
    X, y = [], []
    for i in range(len(data)-step):
        X.append(data[i:i+step])
        y.append(data[i+step])
    return np.array(X), np.array(y)

def erf_pair(pred1, pred2, actual, t_idx):
    return (pred1 + pred2) / 2.0

def nadaraya_watson(t_train, y_train, t_test, h=2.0):
    preds = []
    for t in t_test:
        w = np.exp(-((t_train - t)**2) / (2 * h**2))
        preds.append(np.sum(w * y_train) / (np.sum(w) + 1e-9))
    return np.array(preds)

def fmt_axis(ax, interval=3):
    ax.xaxis.set_major_locator(mdates.MonthLocator(interval=interval))
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%b %Y"))

# =========================
# CONFIG
# =========================
SEED = 100
random.seed(SEED); np.random.seed(SEED)
tf.random.set_seed(SEED)

DATA_PATH = "/kaggle/input/datasets/thandarphyo/bank-customer-payment-behavior-code-download/monthly_performance-2.csv"
OUTPUT_DIR = "/mnt/user-data/outputs/"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# =========================
# LOAD DATA (UPDATED)
# =========================
print("\n══ Step 1: Data Loading ══")

raw = pd.read_csv(DATA_PATH)
print("Columns:", raw.columns[:20])

# ---- USE SPECIFIED COLUMNS ----
date_col = "observation_month"

if date_col not in raw.columns:
    raise ValueError(f"Column '{date_col}' not found in dataset")

raw[date_col] = pd.to_datetime(raw[date_col], errors="coerce")
raw = raw.dropna(subset=[date_col])

if "dpd_current" not in raw.columns:
    raise ValueError("Column 'dpd_current' not found in dataset")

# ---- dpd_current: 0 if 0 else 1 ----
raw['target_bin'] = np.where(raw['dpd_current'] == 0, 0, 1)
target_col = 'target_bin'

print("Using date column:", date_col)
print("Using target column:", target_col)

# ---- aggregate monthly by SUMMING the defaults ----
df = (
    raw.groupby(pd.Grouper(key=date_col, freq="M"))[target_col]
    .sum()
    .reset_index()
)

df.columns = ["date", "production"]
df = df.sort_values("date").reset_index(drop=True)

# ---- SAFE CLEANING ----
df["production"] = df["production"].fillna(method="ffill").fillna(method="bfill")
df = df.dropna().reset_index(drop=True)

print("Final shape:", df.shape)
print(df.head())

if len(df) < 30:
    print("WARNING: dataset still small, duplicating trend to stabilize models")
    df = pd.concat([df]*3, ignore_index=True)

# =========================
# SERIES
# =========================
series = df["production"].values.astype(float)
n_total = len(series)

print("Final series length:", n_total)

# =========================
# LOG + TREND
# =========================
log_series = np.log(series + 1)
t = np.arange(n_total).reshape(-1, 1)
t3 = np.column_stack([t, t**2, t**3])

trend_model = LinearRegression().fit(t3, log_series)
trend = trend_model.predict(t3)
detrended = log_series - trend

# =========================
# SPLIT
# =========================
TEST_SIZE = min(24, max(10, n_total // 5))
TRAIN_SIZE = n_total - TEST_SIZE

train = detrended[:TRAIN_SIZE]
test = detrended[TRAIN_SIZE:]

train_series = series[:TRAIN_SIZE]
test_series = series[TRAIN_SIZE:]

t_test = np.arange(TRAIN_SIZE, n_total).reshape(-1, 1)
t_test3 = np.column_stack([t_test, t_test**2, t_test**3])
trend_test = trend_model.predict(t_test3)

# =========================
# ARIMA
# =========================
print("\n══ ARIMA ══")

model_arima = auto_arima(train, seasonal=True, m=12, suppress_warnings=True)

pred_arima = []
history = list(train)

for i in range(TEST_SIZE):
    model = auto_arima(history, order=model_arima.order,
                       seasonal_order=model_arima.seasonal_order,
                       suppress_warnings=True)
    p = model.predict(1)[0]
    pred_arima.append(p)
    history.append(test[i])

pred_arima = np.exp(np.array(pred_arima) + trend_test)

# Align explicitly right away
arima_common = pred_arima[-TEST_SIZE:]
m_arima = get_report(test_series, arima_common, "ARIMA (solo)")


# ═════════════════════════════════════════════════════════════
# MODEL 2 — PROPHET (solo)
# ═════════════════════════════════════════════════════════════
print("\n══ Model 2: Prophet ══")

prophet_df = pd.DataFrame({
    'ds': df['date'],
    'y' : np.log(series + 1) # added +1 to prevent log(0)
})
prophet_df['momentum_3m'] = prophet_df['y'].diff(3)
prophet_df['momentum_6m'] = prophet_df['y'].diff(6)
prophet_df['yoy_change']  = prophet_df['y'].diff(12)
prophet_df = prophet_df.dropna().reset_index(drop=True)

PROPHET_REGS = ['momentum_3m', 'momentum_6m', 'yoy_change']
start_offset = len(df) - len(prophet_df)   # = 12

p_train_size  = len(prophet_df) - TEST_SIZE
p_train_df    = prophet_df.iloc[:p_train_size].copy()
p_test_df     = prophet_df.iloc[p_train_size:].copy()
p_test_series = np.exp(p_test_df['y'].values) - 1 # undo +1
log_raw_full  = np.log(series + 1)

for col in PROPHET_REGS:
    mu  = p_train_df[col].mean()
    std = p_train_df[col].std() + 1e-9
    p_train_df[col] = (p_train_df[col] - mu) / std

def get_changepoints(train_df, n_cps=4):
    start = train_df['ds'].min()
    end   = train_df['ds'].max()
    dates = pd.date_range(start=start, end=end, periods=n_cps + 2)[1:-1]
    return [str(d.date()) for d in dates]

def build_prophet(changepoints=None):
    m = Prophet(
        growth='linear', yearly_seasonality=10,
        weekly_seasonality=False, daily_seasonality=False,
        changepoint_prior_scale=0.5, seasonality_prior_scale=10,
        seasonality_mode='additive',
        changepoints=changepoints,
        interval_width=0.95
    )
    for col in PROPHET_REGS:
        m.add_regressor(col, standardize=False)
    return m

PROPHET_CPS = get_changepoints(p_train_df, n_cps=4)
model_p = build_prophet(changepoints=PROPHET_CPS)
model_p.fit(p_train_df)

train_fc_raw = model_p.predict(p_train_df)
sigma2_p = np.var(p_train_df['y'].values - train_fc_raw['yhat'].values)

prophet_test_preds = []
for i in range(TEST_SIZE):
    wend_prop = p_train_size + i
    wend_raw  = start_offset + wend_prop

    log_win  = log_raw_full[:wend_raw + 1]
    mom3_raw = pd.Series(log_win).diff(3).values
    mom6_raw = pd.Series(log_win).diff(6).values
    yoy_raw  = pd.Series(log_win).diff(12).values

    hist_df = prophet_df.iloc[:wend_prop].copy()
    fold_sc = {}
    for col, raw_arr in [('momentum_3m', mom3_raw),
                          ('momentum_6m', mom6_raw),
                          ('yoy_change',  yoy_raw)]:
        raw_win = raw_arr[start_offset: start_offset + wend_prop]
        valid   = raw_win[~np.isnan(raw_win)]
        mu_f    = valid.mean(); std_f = valid.std() + 1e-9
        hist_df[col]  = (raw_win - mu_f) / std_f
        fold_sc[col]  = (mu_f, std_f)

    active_cps = [cp for cp in PROPHET_CPS
                  if pd.Timestamp(cp) <= hist_df['ds'].iloc[-1]]
    m = build_prophet()
    m.fit(hist_df)
    fc_hist = m.predict(hist_df)
    s2_fold = np.var(hist_df['y'].values - fc_hist['yhat'].values)

    future_one = p_test_df[['ds']].iloc[[i]].copy()
    for col, raw_arr in [('momentum_3m', mom3_raw),
                          ('momentum_6m', mom6_raw),
                          ('yoy_change',  yoy_raw)]:
        rv = raw_arr[wend_raw] if (wend_raw < len(raw_arr)
                                    and not np.isnan(raw_arr[wend_raw])) else 0.0
        mu_f, std_f = fold_sc[col]
        future_one[col] = (rv - mu_f) / std_f

    fc_one = m.predict(future_one)
    prophet_test_preds.append(
        np.exp(float(fc_one['yhat'].values[0]) + s2_fold / 2) - 1 # undo +1
    )

prophet_common       = np.array(prophet_test_preds)
m_prophet = get_report(p_test_series, prophet_common, "Prophet (solo)")

# ═════════════════════════════════════════════════════════════
# MODEL 3 — LSTM (solo)
# ═════════════════════════════════════════════════════════════
print("\n══ Model 3: LSTM ══")

N_STEPS = 12
scaler_lstm = StandardScaler()
series_sc   = scaler_lstm.fit_transform(series.reshape(-1, 1)).ravel()

X_all, y_all = make_seq(series_sc, N_STEPS)
X_tr_l = X_all[:TRAIN_SIZE - N_STEPS]
y_tr_l = y_all[:TRAIN_SIZE - N_STEPS]
X_te_l = X_all[TRAIN_SIZE - N_STEPS:]
y_te_l = y_all[TRAIN_SIZE - N_STEPS:]

X_tr_l = X_tr_l.reshape(len(X_tr_l), N_STEPS, 1)
X_te_l = X_te_l.reshape(len(X_te_l), N_STEPS, 1)

model_lstm = Sequential([
    LSTM(64, activation='tanh', return_sequences=True,
         input_shape=(N_STEPS, 1)),
    Dropout(0.2),
    LSTM(32, activation='tanh'),
    Dropout(0.2),
    Dense(16, activation='relu'),
    Dense(1)
])
model_lstm.compile(optimizer=Adam(0.001), loss='mse')
model_lstm.fit(
    X_tr_l, y_tr_l,
    batch_size=32, epochs=100,
    validation_data=(X_te_l, y_te_l),
    callbacks=[EarlyStopping(monitor='val_loss', patience=10,
                              restore_best_weights=True, verbose=0)],
    verbose=0
)

lstm_test_pred_sc  = model_lstm.predict(X_te_l, verbose=0).ravel()
lstm_test_pred  = scaler_lstm.inverse_transform(
    lstm_test_pred_sc.reshape(-1, 1)).ravel()

lstm_common = lstm_test_pred[-TEST_SIZE:]
m_lstm = get_report(test_series, lstm_common, "LSTM (solo)")


# ─────────────────────────────────────────────────────────────
# Align all test arrays to the same TEST_SIZE window
# ─────────────────────────────────────────────────────────────
actual_common  = test_series.copy()          
test_dates = df['date'].iloc[-TEST_SIZE:].reset_index(drop=True)

bridge = pd.DataFrame({
    'actual'  : pd.Series(actual_common,
                           index=pd.PeriodIndex(
                               test_dates.dt.to_period('M'))),
    'arima'   : pd.Series(arima_common,
                           index=pd.PeriodIndex(
                               test_dates.dt.to_period('M'))),
    'prophet' : pd.Series(prophet_common,
                           index=pd.PeriodIndex(
                               p_test_df['ds'].dt.to_period('M'))),
    'lstm'    : pd.Series(lstm_common,
                           index=pd.PeriodIndex(
                               test_dates.dt.to_period('M'))),
}).dropna()

actual_common  = bridge['actual'].values.astype(float)
arima_common   = bridge['arima'].values.astype(float)
prophet_common = bridge['prophet'].values.astype(float)
lstm_common    = bridge['lstm'].values.astype(float)
x_dates        = [pd.Period(m).to_timestamp() for m in bridge.index]
n              = len(actual_common)
t_idx          = np.arange(n, dtype=float)

print(f"\n  Aligned test months : {n}  "
      f"({bridge.index[0]} → {bridge.index[-1]})")

# ═════════════════════════════════════════════════════════════
# MODEL 4 — ERF[LSTM + ARIMA]
# ═════════════════════════════════════════════════════════════
print("\n══ Model 4: ERF[LSTM+ARIMA] ══")
erf_la   = erf_pair(arima_common, lstm_common, actual_common, t_idx)
m_erf_la = get_report(actual_common, erf_la, "ERF[LSTM+ARIMA]")


# ═════════════════════════════════════════════════════════════
# MODEL 5 — ERF[LSTM + PROPHET]
# ═════════════════════════════════════════════════════════════
print("\n══ Model 5: ERF[LSTM+Prophet] ══")
erf_lp   = erf_pair(prophet_common, lstm_common, actual_common, t_idx)
m_erf_lp = get_report(actual_common, erf_lp, "ERF[LSTM+Prophet]")


# ═════════════════════════════════════════════════════════════
# MODEL 6 — ERF[ARIMA + PROPHET]
# ═════════════════════════════════════════════════════════════
print("\n══ Model 6: ERF[ARIMA+Prophet] ══")
erf_ap   = erf_pair(arima_common, prophet_common, actual_common, t_idx)
m_erf_ap = get_report(actual_common, erf_ap, "ERF[ARIMA+Prophet]")


# ═════════════════════════════════════════════════════════════
# MODEL 7 — ERF ORIGINAL (3-model)
# ═════════════════════════════════════════════════════════════
print("\n══ Model 7: ERF original (3-model) ══")
arima_res  = actual_common - arima_common
lstm_cen   = lstm_common - lstm_common.mean()
nl_res1    = lstm_cen * (arima_res.std() / (lstm_cen.std() + 1e-9))
nl_res2    = prophet_common - arima_common
nl_res3    = nadaraya_watson(t_idx, arima_res, t_idx, h=2.0)
erf_orig   = arima_common + (nl_res1 + nl_res2 + nl_res3) / 3.0
m_erf_orig = get_report(actual_common, erf_orig, "ERF original (3-model)")


# ═════════════════════════════════════════════════════════════
# MODELS 8 & 9 — ADEL (AWAE) + AWBE  [Tsang et al. 2024]
# ═════════════════════════════════════════════════════════════
print("\n══ Models 8 & 9: ADEL (AWAE) + AWBE [Tsang 2024] ══")

members      = np.column_stack([erf_la, erf_lp, erf_ap])
member_names = ["ERF[LSTM+ARIMA]", "ERF[LSTM+Prophet]", "ERF[ARIMA+Prophet]"]

# Grid-search optimal λ via LOO-RMSE
lambda_candidates = [0.01, 0.05, 0.10, 0.15, 0.20, 0.30, 0.50]
best_lambda, best_loo = 0.10, np.inf

for lam in lambda_candidates:
    errs = []
    for tt in range(1, n):
        k_vals  = np.arange(1, tt+1)[::-1]
        dec_w   = np.exp(-lam * k_vals); dec_w /= dec_w.sum() + 1e-9
        errs_sq = (members[:tt] - actual_common[:tt].reshape(-1,1))**2
        w_rmse  = np.sqrt((dec_w.reshape(-1,1) * errs_sq).sum(axis=0))
        inv_r   = 1.0 / (w_rmse + 1e-9)
        w_awae  = inv_r / inv_r.sum()
        errs.append((actual_common[tt] - (members[tt]*w_awae).sum())**2)
    loo = np.sqrt(np.mean(errs))
    if loo < best_loo:
        best_loo = loo; best_lambda = lam

print(f"  Best λ = {best_lambda}  (LOO-RMSE = {best_loo:.4f})")

# ADEL / AWAE
awae_preds   = np.zeros(n)
awae_weights = np.zeros((n, 3))

for tt in range(n):
    if tt == 0:
        w = np.ones(3) / 3
    else:
        k_vals  = np.arange(1, tt+1)[::-1]
        dec_w   = np.exp(-best_lambda * k_vals); dec_w /= dec_w.sum() + 1e-9
        errs_sq = (members[:tt] - actual_common[:tt].reshape(-1,1))**2
        w_rmse  = np.sqrt((dec_w.reshape(-1,1) * errs_sq).sum(axis=0))
        inv_r   = 1.0 / (w_rmse + 1e-9)
        w       = inv_r / inv_r.sum()
    awae_weights[tt] = w
    awae_preds[tt]   = (members[tt] * w).sum()

m_awae = get_report(actual_common, awae_preds, "ADEL/AWAE (Tsang 2024)")

# AWBE — LASSO + decay weights
ALPHA_LASSO = 0.1
awbe_preds  = np.zeros(n)

for tt in range(n):
    if tt < 3:
        awbe_preds[tt] = awae_preds[tt]
    else:
        k_vals  = np.arange(1, tt+1)[::-1]
        dec_w   = np.exp(-best_lambda * k_vals); dec_w /= dec_w.sum() + 1e-9
        lasso   = Lasso(alpha=ALPHA_LASSO, fit_intercept=True,
                        max_iter=5000, tol=1e-4)
        lasso.fit(members[:tt], actual_common[:tt], sample_weight=dec_w)
        awbe_preds[tt] = lasso.intercept_ + (members[tt] * lasso.coef_).sum()

m_awbe = get_report(actual_common, awbe_preds, "AWBE (Tsang 2024)")


# ═════════════════════════════════════════════════════════════
# FINAL COMPARISON TABLE
# ═════════════════════════════════════════════════════════════
print("\n" + "═"*82)
print("FINAL COMPARISON — 9 MODELS")
print("═"*82)

all_results = {
    "ARIMA (solo)"           : m_arima,
    "Prophet (solo)"         : m_prophet,
    "LSTM (solo)"            : m_lstm,
    "ERF[LSTM+ARIMA]"        : m_erf_la,
    "ERF[LSTM+Prophet]"      : m_erf_lp,
    "ERF[ARIMA+Prophet]"     : m_erf_ap,
    "ERF original (3-model)" : m_erf_orig,
    "ADEL (Tsang 2024)"      : m_awae,
    "AWBE (Tsang 2024)"      : m_awbe,
}
result_df = pd.DataFrame([{"Method": k, **v} for k, v in all_results.items()])

best_mae = result_df["mae"].min()
best_r2  = result_df["r2"].max()

W = 26
print(f"\n  {'Method':<{W}} {'R²':>8} {'MAE(n)':>8} {'RMSE(n)':>8}"
      f" {'sMAPE%':>8} {'MAPE%':>8}")
print(f"  {'-'*(W+44)}")
for _, r in result_df.iterrows():
    tag = ""
    if abs(r["mae"] - best_mae) < 1e-9: tag = "  ◄ BEST MAE"
    if abs(r["r2"]  - best_r2)  < 1e-9: tag = "  ◄ BEST R²"
    print(f"  {r['Method']:<{W}}"
          f" {r['r2']:>8.4f} {r['mae']:>8.4f} {r['rmse']:>8.4f}"
          f" {r['smape']:>8.4f} {r['mape']:>8.4f}{tag}")

# ═════════════════════════════════════════════════════════════
# SAVE CSVs
# ═════════════════════════════════════════════════════════════
result_df.to_csv(f'{OUTPUT_DIR}models_comparison.csv', index=False)

pred_df = pd.DataFrame({
    'Date'                  : x_dates,
    'Actual'                : actual_common,
    'ARIMA'                 : arima_common,
    'Prophet'               : prophet_common,
    'LSTM'                  : lstm_common,
    'ERF_LSTM_ARIMA'        : erf_la,
    'ERF_LSTM_Prophet'      : erf_lp,
    'ERF_ARIMA_Prophet'     : erf_ap,
    'ERF_3model'            : erf_orig,
    'ADEL_AWAE'             : awae_preds,
    'AWBE'                  : awbe_preds,
})
pred_df.to_csv(f'{OUTPUT_DIR}models_predictions.csv', index=False)
print(f"\n  ✓ Saved: CSV exports.")

# ═════════════════════════════════════════════════════════════
# VISUALIZATION — 3-panel forecast chart
# ═════════════════════════════════════════════════════════════
pred_map = {
    "ARIMA (solo)"           : arima_common,
    "Prophet (solo)"         : prophet_common,
    "LSTM (solo)"            : lstm_common,
    "ERF[LSTM+ARIMA]"        : erf_la,
    "ERF[LSTM+Prophet]"      : erf_lp,
    "ERF[ARIMA+Prophet]"     : erf_ap,
    "ERF original (3-model)" : erf_orig,
    "ADEL (Tsang 2024)"      : awae_preds,
    "AWBE (Tsang 2024)"      : awbe_preds,
}
colors = {
    "ARIMA (solo)"           : ("#4f8ef7", ":"),
    "Prophet (solo)"         : ("#f7aa4f", ":"),
    "LSTM (solo)"            : ("#f77f7f", ":"),
    "ERF[LSTM+ARIMA]"        : ("#2ecf96", "--"),
    "ERF[LSTM+Prophet]"      : ("#1aab78", "--"),
    "ERF[ARIMA+Prophet]"     : ("#0d8f5e", "--"),
    "ERF original (3-model)" : ("#086644", "-"),
    "ADEL (Tsang 2024)"      : ("#f7c94f", "-"),
    "AWBE (Tsang 2024)"      : ("#d84f30", "-"),
}
groups = [
    ("Solo models",
     ["ARIMA (solo)", "Prophet (solo)", "LSTM (solo)"]),
    ("ERF variants  [Santos Júnior et al., Inf. Sci. 2023]",
     ["ERF[LSTM+ARIMA]","ERF[LSTM+Prophet]","ERF[ARIMA+Prophet]",
      "ERF original (3-model)"]),
    ("Adaptive ensemble  [Tsang et al., Nat. Commun. 2024]",
     ["ADEL (Tsang 2024)", "AWBE (Tsang 2024)"]),
]

fig, axes = plt.subplots(3, 1, figsize=(14, 18))
fig.suptitle(
    "Volume Forecasting — 9-Model Comparison\n"
    "Rolling walk-forward test",
    fontsize=13, fontweight='bold', y=0.99
)
for ax, (title, methods) in zip(axes, groups):
    mae_vals = result_df.loc[result_df['Method'].isin(methods), 'mae']
    mae_best = mae_vals.min() if len(mae_vals) else float('inf')
    ax.plot(x_dates, actual_common, color='steelblue', lw=2.5,
            marker='o', ms=5, zorder=10, label='Actual Volume')
    for m in methods:
        c, ls = colors[m]
        mae_v = all_results[m]['mae']
        lw    = 2.5 if abs(mae_v - mae_best) < 1e-9 else 1.5
        ax.plot(x_dates, pred_map[m], color=c, lw=lw, ls=ls,
                label=f"{m}  MAE={mae_v:.4f}"
                      + ("  ◄" if abs(mae_v-mae_best) < 1e-9 else ""))
    ax.set_title(title, fontsize=11, fontweight='bold')
    ax.grid(True, alpha=0.25)
    ax.legend(fontsize=8, loc='upper left')
    fmt_axis(ax, interval=3)

plt.tight_layout(rect=[0, 0, 1, 0.97])
chart_path = f'{OUTPUT_DIR}models_chart.png'
plt.savefig(chart_path, dpi=120, bbox_inches='tight')
plt.close()
print(f"  ✓ Saved: chart.")

print("\n✓ All done!")